# 01 — Reproducible telco synthetic-data generator (v4.1.0)

**Scope of this notebook, deliberately narrow.** It builds a reproducible synthetic GPON/FTTH telemetry generator and publishes **versioned datasets, configurations and manifests** to Drive. It does not build features, detectors or models, and it does not map anything to a sector-neutral canonical schema. That mapping comes later, as a separate adapter layer, once there are several sources to generalise over.

**Three versions are tracked separately**, because they change for different reasons:

| version | changes when | current |
|---|---|---|
| generator | simulator semantics change — the data itself differs | `4.1.0` |
| native format | the on-disk shape changes — a breaking change for adapters | `telemetry-synth native v5` |
| dataset | any run: generator version + config + seed + environment | per publication |

**Reproducibility is verified, not asserted.** Section 13 regenerates a published dataset from its stored config and compares content digests table by table.

## v4.1 — ground truth leaves the panel, and the physics gets a second direction

v4.1 responds to an audit of v4.0.1 on its own reference run (400 ONTs × 180 days, seed 20250717, all 23 v4.0 gates green). Every item below is a **measured** defect. Corrections C1–C11, design goals D1–D18 and corrections E1–E14 are retained in full; the gate suite grows from 23 checks to 31.

### The panel is now observables only

v4.0 carried fourteen `gt_` columns inside `reference_dataset.parquet` — 20.8% of compressed bytes — and enforced the separation with a naming convention and a paragraph. `FEATURE_BLOCKLIST_PREFIX` was defined and referenced nowhere. Triaged:

- **Not ground truth at all.** `gt_margin_db` equalled `rx_power_dbm − rx_sensitivity_dbm` to a measured **0.025 dB** mean absolute error (sd 0.032, r = 0.9998), both columns the panel handed over. `gt_physical_temperature_c` sat 0.43 °C from the observable `temperature_c`. **Removed.**
- **Denormalisation of the sidecars.** The nine label columns rebuild from `fault_entity_intervals.csv` by interval join to **99.1%** — and the residual 0.9% was a panel bug, not a reconstruction failure (see F8). **Moved to `gt_panel.parquet`.**
- **Genuinely latent, kept.** The counterfactual healthy trajectory and true degradation depth cannot be recovered from observables and are needed to score lead time or severity honestly. **Moved to `gt_panel.parquet`**, now with a second direction.

Both panels are written from the same loop in the same row order, so downstream attaches them positionally or joins on `(ont_id, timestamp_utc)`. The generator **raises** if any `gt_`-prefixed column reaches the observable panel. Eleven non-telemetry static attributes — plant records, commercial weights, geography — also left the panel; they were never telemetry and `topology.csv` already held them.

### F1–F9, measured defects in v4.0.1

- **F1 One direction only** — v4.0 modelled downstream `rx` alone, so the asymmetry a field engineer localises with did not exist. `olt_rx_power_dbm` added: plant faults attenuate both carriers, an OLT transmit-path fault moves downstream only, an ONT laser fault moves upstream only. Measured on v4.1: a laser fault shifts downstream **−0.85 dB** against upstream **−4.10 dB**, and within-entity healthy correlation between the two channels is **0.096** — a genuinely independent observation, not a copy.
- **F2 One wavelength** — 0.35 dB/km was charged to the only direction modelled, which is the 1310 nm figure. Downstream is now 0.24 dB/km at 1490 nm, upstream 0.35 at 1310.
- **F3 No alarm channel** — LOS is *observable*; the OLT records it. v4.0 wrote it to the ground-truth gap log alone, where **97%** of rows sat on faulty entities, so a strong observable was withheld and labelled ground truth. `alarms.csv` added with LOS, dying gasp, signal-degrade and signal-fail. Deliberately ambiguous: a customer pulling the plug and a fibre break both raise LOS and only the dying gasp separates them. Measured share of alarms raised while **no** fault was active: dying gasp 0.69, signal-degrade 0.43, LOS 0.35, signal-fail 0.34.
- **F4 Margin piled up on its own clamp** — `turnup_min_margin_db` **bound on 51%** of lines, putting a delta spike at exactly 2.5 dB, half a decibel above `impact_margin_db`. Commissioning now aims at a target drawn per line: median **6.29 dB**, sd 1.89, largest 0.1 dB bucket 3.5%.
- **F5 Undeployable split ratios** — 1:4 × (8, 16, 32) gave up to 1:128, and because larger splitters host more subscribers **56%** of the fleet landed there. Median span loss measured **28.0 dB** against a 28 dB class B+ budget, maximum 29.6. Now 1:4 × (4, 8, 16): median span loss **24.1 dB**, 98.5% inside budget.
- **F6 Weather that ignored the calendar** — the seasonal term was built from `np.arange(n) / n`, so the cycle completed 1.5 times across the window whatever the window was: identical shape and a full 5.8 °C excursion at 30, 60, 180 and 365 days, always peaking 85% of the way through. `cfg.start` never entered the physics. Anchored to day-of-year.
- **F7 Room temperature, not transceiver temperature** — the OMCI attribute is the optics module's own, typically 35–55 °C in service. v4.0 reported enclosure ambient, so the fleet sat near 15 °C and `bias_thermal_coeff * (temp − 25.0)` was evaluated below its reference point everywhere, always. Self-heating added per enclosure.
- **F8 Labels collapsed overlapping episodes** — per-sample state was written last-writer-wins over episodes sorted by start, so an entity under a *second* active fault could be labelled `repaired`: **2,654 rows**, about 15% of that class. State is now decided by precedence and fault identity belongs to the earliest-onset active episode. Contradictory rows: **0**.
- **F9 Counters that disagreed with themselves** — `reboot_count` restarted at zero inside the window while `uptime_s` was seeded from up to 55 hours. And a stuck collector record froze six channels while `fec_count` and `crc_errors` carried on varying, handing a two-channel consistency check a free separation of the benign class — the exact separation E5 exists to deny. Both fixed.

### D1 — the difficulty floor is now gated

The v4.0 header listed a "naive difficulty-floor probe" among the fixture checks and never implemented it. The 0.94 → 0.74 union-recall figure was the entire justification for v4 and **nothing held it in place**. It is now check D1, banded on both sides so the fixture can drift neither easy nor impossible. On the v4.1 reference run the 6×MAD union (rx | bias | tx) reaches **0.45 in-episode detection** over 778 episodes and **0.37 pre-impact recall** over the 188 episodes with a non-degenerate window, at a **0.38** clean-fleet alert rate.

### Gate thresholds are scale-relative

E11 and E14 used absolute counts tuned to 400 × 180; a correct 120-entity run failed E14 with one decommission against an expectation of 4.2. Both are now expressed against the configured rates.

## Open decisions carried, not resolved

**OD1** poll jitter and duplicate polls default off. **OD2** collection gaps reconcile against the grid intersected with each entity's service window. **OD3** storm parameters uncalibrated. **OD4** frailty and hazard coefficients uncalibrated and deliberately weak. **OD5** benign-anomaly rates are benchmark tuning. **OD6 (new)** the D1 band, 0.30–0.85, is set from two runs and is a benchmark-design choice rather than a measurement. **OD7 (new)** `us_ratio` per fault family is an engineering estimate with no field basis; it is what makes upstream/downstream localisation learnable, so it is the parameter most worth checking with a practitioner.


## 1. Dependencies and the persistent workspace

In [ ]:
%pip install -q numpy pandas pyarrow scipy

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE_DIRECTORY = Path('/content/drive/MyDrive/anomaly_product_workspace')
except ModuleNotFoundError:
    WORKSPACE_DIRECTORY = Path.cwd() / 'anomaly_product_workspace'

# Staging is scratch and may be deleted at any time. DATASETS_ROOT is the durable,
# versioned store: published datasets there are immutable and are what downstream work
# reads. Nothing downstream should ever point at a stage directory.
STAGE_ROOT = WORKSPACE_DIRECTORY / 'stage_01_generator'
DATASETS_ROOT = WORKSPACE_DIRECTORY / 'datasets'
STAGE_ROOT.mkdir(parents=True, exist_ok=True)
DATASETS_ROOT.mkdir(parents=True, exist_ok=True)

print('Workspace:', WORKSPACE_DIRECTORY)
print('Dataset store:', DATASETS_ROOT)


## 2. Settings, catalogues and fault taxonomy

Definitions only. Twelve fault mechanisms across five topology scopes and seven trajectory shapes, including `intermittent`, `progressive` and `partial_recovery`. Magnitudes are lognormal, so every optical family has genuine mass below the practical wander anchor; v3.1 used uniforms floored at 1.5 dB and only 1 fault in 293 ever failed to become observable.

In [ ]:
from __future__ import annotations

import hashlib
import json as _json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.signal import lfilter

FEATURE_BLOCKLIST_PREFIX = "gt_"
GENERATOR_VERSION = "4.1.0"
NATIVE_FORMAT_VERSION = "telemetry-synth native v5"



# ======================================================================================
# Configuration
# ======================================================================================


@dataclass
class TelecomSimulationSettings:
    # --- scale -----------------------------------------------------------------------
    n_onts: int = 400
    days: int = 180
    sample_minutes: int = 15
    start: str = "2025-01-01"
    seed: int = 20250717
    pre_window_days: int = 28          # left-censoring horizon

    # --- topology (E1, E2, E3) --------------------------------------------------------
    # v4: four OLTs, not two. With two, "cross-OLT correlation" is a comparison between
    # two groups and the topological gradient collapses to one binary factor.
    n_olts: int = 4
    pon_ports_per_olt: int = 8
    l1_split: int = 4
    # v4 (E1): capacity is a property of the installed splitter, not a global constant,
    # and take-up is partial. v3.1 filled every populated splitter to exactly 8.
    # v4.1 (F5): totals are 1:16 / 1:32 / 1:64 behind the 1:4 primary. v4.0 offered
    # (8, 16, 32), i.e. up to 1:128, and because larger splitters host more subscribers
    # 56% of the fleet landed there. Measured median span loss was 28.0 dB against a
    # 28 dB class B+ budget, with a maximum of 29.6 dB. NOTE the weights are per
    # INSTALLED SPLITTER; the subscriber-weighted mix is roughly the reverse.
    l2_capacity_choices: tuple = (4, 8, 16)
    l2_capacity_weights: tuple = (0.20, 0.50, 0.30)
    l2_takeup_beta_a: float = 4.0      # Beta(a, b) take-up fraction of capacity
    l2_takeup_beta_b: float = 2.2
    l2_min_fill: int = 2
    # v4 (E3): route length composes as feeder (PON) + distribution (L1) + drop (ONT),
    # so entities under one splitter share most of their loss budget.
    feeder_km_shape: float = 2.2
    feeder_km_scale: float = 1.5
    distribution_km_shape: float = 1.8
    distribution_km_scale: float = 0.45
    drop_m_shape: float = 2.0
    drop_m_scale: float = 55.0
    n_geo_clusters: int = 8

    # --- churn (E14) ------------------------------------------------------------------
    p_late_install: float = 0.06       # provisioned after t0 -> genuinely cold entities
    p_decommission: float = 0.035      # churned away before the window ends

    # --- optical budget (ITU-T G.984 class B+) ----------------------------------------
    olt_launch_dbm: float = 3.0
    l1_loss_db: float = 7.2
    # v4.1 (F2): downstream is 1490 nm, upstream 1310 nm. v4.0 charged 0.35 dB/km to the
    # only direction it modelled, which is the 1310 nm figure.
    fibre_loss_db_per_km: float = 0.24
    fibre_loss_us_db_per_km: float = 0.35
    connector_loss_mean_db: float = 1.4
    connector_loss_sd_db: float = 0.45
    splices_per_km: float = 0.5
    splice_loss_mean_db: float = 0.12
    splice_loss_sd_db: float = 0.04
    n_extra_connectors: int = 2
    extra_connector_loss_mean_db: float = 0.55
    extra_connector_loss_sd_db: float = 0.20
    as_built_excess_shape: float = 2.0
    as_built_excess_scale: float = 1.3
    extra_plant_loss_min_db: float = 0.5
    extra_plant_loss_max_db: float = 7.0
    # v4.1 (F4): a commissioning TARGET drawn per line. In v4.0 `turnup_min_margin_db`
    # was a clamp that BOUND on 51% of lines, putting a delta spike at exactly 2.5 dB
    # (measured median 2.50 dB, sd 1.37) half a decibel above `impact_margin_db`.
    target_margin_median_db: float = 5.2
    target_margin_log_sd: float = 0.42
    turnup_min_margin_db: float = 1.5

    # --- fault process ----------------------------------------------------------------
    fault_rate_per_ont_year: float = 1.0
    shared_fault_rate_per_node_year: float = 0.80   # per L2 node-year; scaled per level
    # Coarser plant is more reliable AND affects far more customers, so a flat per-node
    # rate at every level saturates the fleet: on the first v4 run, 5 OLT-scope faults
    # left only 1 of 400 entities fault-free. Scaled per level here.
    shared_scope_rate_scale: tuple = (1.0, 0.45, 0.30, 0.12)   # l2, l1, pon, olt
    # v4 (E9): gamma frailty and covariate hazard. Deliberately weak: susceptibility is
    # something a model should have to work to find, not a giveaway.
    frailty_shape: float = 1.6
    beta_fibre_age: float = 0.045      # per year
    beta_outdoor: float = 0.30         # outdoor enclosure log-hazard uplift
    beta_distance_km: float = 0.035
    beta_excess_loss_db: float = 0.10
    p_recurrence: float = 0.22         # repeat fault on a repaired entity
    recurrence_window_days: float = 45.0

    # --- storms / grouped causes (E11) ------------------------------------------------
    storms_per_year: float = 24.0
    storm_duration_h_mean: float = 20.0
    storm_geo_clusters_mean: float = 1.8
    storm_hazard_multiplier: float = 15.0
    storm_collector_outage_multiplier: float = 4.0

    # --- impact -----------------------------------------------------------------------
    impact_margin_db: float = 2.0
    impact_sustain_samples: int = 4
    impact_attribution_min_db: float = 0.8

    # --- reporting / ticketing --------------------------------------------------------
    report_depth_floor_db: float = 1.0
    report_hazard_per_db_day: float = 0.012
    report_hazard_cap_per_day: float = 0.35
    laser_report_lam_floor: float = 0.30
    laser_report_hazard_per_day: float = 0.06
    p_ticket_given_impact: float = 0.72     # v4 (E7): was 0.85; ticket coverage is worse
    nff_ticket_rate: float = 0.14           # v4 (E7): was 0.08
    p_duplicate_ticket: float = 0.10        # same customer reports twice
    p_misattributed_ticket: float = 0.04    # raised against the wrong line
    p_ticket_unresolved: float = 0.09       # closed without a resolution timestamp
    ticket_resolution_jitter_h: float = 6.0
    report_delay_mean_h: float = 3.5
    report_delay_heavy_tail_p: float = 0.15  # a minority report days later
    report_delay_heavy_mean_h: float = 60.0
    mttr_median_h: float = 16.0
    mttr_sigma: float = 0.62

    # --- proactive maintenance --------------------------------------------------------
    proactive_ont_visit_rate_per_year: float = 0.10
    proactive_splitter_visit_rate_per_year: float = 0.6
    proactive_onsite_hours: float = 2.0

    # --- label semantics --------------------------------------------------------------
    repaired_convalescence_days: int = 7

    # --- missingness (E10) ------------------------------------------------------------
    # v4: per-entity reliability, so "badly polled entity" exists as a nuisance class.
    poll_reliability_beta_a: float = 60.0
    poll_reliability_beta_b: float = 1.1
    p_field_nan: float = 0.004
    collector_outage_per_collector_per_month: float = 2.0
    collector_outage_hours_mean: float = 2.5
    n_collectors: int = 3
    # benign dense-gap processes, so a gap burst is not a fault oracle
    benign_outage_rate_per_ont_year: float = 3.0
    benign_outage_hours_mean: float = 7.0
    holiday_absence_rate_per_ont_year: float = 0.7
    holiday_absence_days_mean: float = 6.0
    # OD1: default off; they break the grid-reconstruction invariant
    poll_jitter_s: float = 0.0
    p_duplicate_poll: float = 0.0

    # --- quantisation at emission -----------------------------------------------------
    quant_rx_db: float = 0.1
    quant_tx_db: float = 0.1
    quant_temp_c: float = 1.0
    quant_bias_ma: float = 0.1
    quant_volt_v: float = 0.01
    quant_throughput_mbps: float = 0.1
    quant_olt_rx_db: float = 0.1

    # --- upstream / OLT-side receive (F1) ---------------------------------------------
    # The OLT measures every ONU's upstream burst. v4.0 modelled downstream only, so the
    # asymmetry a field engineer localises with did not exist: plant faults move both
    # directions, an OLT transmit-path fault moves downstream only, and an ONT laser
    # fault moves upstream only. `tx_power_dbm` cannot stand in -- it is APC-controlled
    # and near flat by construction.
    olt_rx_noise_sd_db: float = 0.28        # burst-mode receivers measure less precisely
    olt_rx_sensitivity_dbm: float = -28.0   # class B+ OLT burst receiver
    olt_rx_cal_sd_db: float = 0.35          # per-PON-port calibration offset
    olt_rx_ar_tau_h: float = 36.0
    olt_rx_ar_sd_db: float = 0.09

    # --- alarms (F3) -------------------------------------------------------------------
    # LOS is OBSERVABLE -- the OLT records it. v4.0 wrote it only to the ground-truth gap
    # log, so the fixture withheld an observable; 97% of those rows sat on faulty
    # entities. Alarms are emitted to `alarms.csv` as an operational channel.
    emit_alarms: bool = True
    # Measured on the v4.1 fixture: at 1e-5 signal-degrade fires on 2.8% of healthy
    # samples and 77.7% of impaired ones; at 3e-4 signal-fail fires on 0.8% and 39.7%.
    # Both keep a real clean-fleet false-alarm rate, which is the point -- an alarm that
    # never fires on a healthy line is an oracle, not an alarm.
    alarm_sd_ber_threshold: float = 1e-5     # signal-degrade, pre-FEC
    alarm_sf_ber_threshold: float = 3e-4     # signal-fail, pre-FEC
    # A soak timer is defined in TIME, not in polls. Expressed in samples it silently
    # changes meaning with cadence: at hourly polling a 3-sample soak is three hours, and
    # the transient healthy excursions that give signal-fail its false-alarm rate cannot
    # survive it -- measured 0.024 raised outside an active fault at 60-minute cadence
    # against 0.336 at 15-minute. Same class of defect as the E10 dense-gap check.
    alarm_sustain_minutes: float = 45.0
    alarm_dying_gasp_p: float = 0.75         # share of premises outages that report one

    # --- benign operational change (E12) ----------------------------------------------
    benign_plant_step_rate: float = 0.6
    plant_step_improve_share: float = 0.75   # rework usually improves the line
    plant_step_mean_db: float = 0.35
    provisioning_change_rate_per_ont_year: float = 0.55
    firmware_rollouts: int = 3
    firmware_rx_step_db: float = 0.0         # firmware does not move optics...
    firmware_bias_step_ma: float = 0.55      # ...but does move the reported bias
    firmware_temp_step_c: float = 0.8
    planned_maintenance_per_pon_per_year: float = 1.2
    planned_maintenance_hours_mean: float = 3.5

    # --- benign anomaly layer (E5) ----------------------------------------------------
    p_sensor_glitch: float = 6.0e-5          # isolated implausible reading
    p_stuck_start: float = 2.0e-5            # stale value run begins
    stuck_run_mean_samples: float = 9.0
    p_transient_burst: float = 1.2e-5        # short multi-sample excursion
    transient_burst_mean_samples: float = 5.0
    transient_burst_rx_db: float = 0.9
    transient_burst_bias_ma: float = 1.4
    reranging_rate_per_ont_year: float = 2.2  # ONT re-ranges: short rx/bias step
    p_cpe_power_cycle_per_day: float = 0.012

    # --- healthy CRC floor ------------------------------------------------------------
    crc_burst_prob_per_sample: float = 2.5e-4
    crc_burst_lognorm_mu: float = 1.2
    crc_burst_lognorm_sigma: float = 1.1
    crc_noisy_plant_share: float = 0.08       # entities with chronically noisy plant
    crc_noisy_plant_multiplier: float = 12.0

    # --- explicit chronic cohort (E5/E12) ---------------------------------------------
    chronic_share: float = 0.035
    chronic_extra_loss_db: float = 1.1
    chronic_noise_multiplier: float = 2.2

    # --- climate (F6) ------------------------------------------------------------------
    # v4.0 built the seasonal term from `np.arange(n) / n`, so the cycle completed 1.5
    # times across the window WHATEVER the window was: identical shape at 30, 60, 180 and
    # 365 days, always peaking 85% of the way through, and `start` never entered the
    # physics at all. Anchored to day-of-year here.
    seasonal_amplitude_c: float = 6.5        # UK ambient half-swing
    seasonal_peak_doy: float = 205.0         # late July
    diurnal_amplitude_c: float = 5.5
    diurnal_peak_h: float = 15.0

    # --- shared hierarchy (E2) --------------------------------------------------------
    fleet_weather_tau_h: float = 72.0
    fleet_weather_sd_c: float = 1.4
    geo_weather_sd_c: float = 1.8
    fleet_demand_tau_h: float = 96.0
    fleet_demand_sd: float = 0.10
    olt_launch_tau_h: float = 168.0
    olt_launch_sd_db: float = 0.22
    pon_feeder_tau_h: float = 120.0
    pon_feeder_sd_db: float = 0.10
    pon_congestion_tau_h: float = 12.0
    pon_congestion_sd: float = 0.18
    # v4 (E2): the two levels that did not exist
    l1_loss_tau_h: float = 96.0
    l1_loss_sd_db: float = 0.075
    l2_loss_tau_h: float = 60.0
    l2_loss_sd_db: float = 0.085
    l2_micro_weather_sd_c: float = 1.0        # cabinet-level thermal environment
    l2_micro_weather_tau_h: float = 30.0

    # --- thermal fault modulation -----------------------------------------------------
    thermal_base_frac: float = 0.60
    thermal_temp_gain: float = 0.35

    # --- microclimate -----------------------------------------------------------------
    microclimate_tau_h: float = 36.0
    solar_exposure_low: float = 0.55
    solar_exposure_high: float = 1.45
    sensor_noise_scale: float = 1.0

    # --- laser / device channels (E4) -------------------------------------------------
    tx_device_sd_db: float = 0.65             # device-to-device transmit spread
    tx_temp_coeff_db_per_c: float = -0.012
    tx_age_db_per_yr: float = -0.045
    tx_ar_tau_h: float = 48.0
    tx_ar_sd_db: float = 0.16
    volt_device_sd_v: float = 0.031
    volt_temp_coeff_v_per_c: float = -0.0016
    volt_load_coeff_v: float = -0.022         # rail sags under traffic load
    volt_ar_tau_h: float = 30.0
    volt_ar_sd_v: float = 0.007
    bias_device_noise_shape: float = 3.0
    bias_device_noise_scale: float = 0.085

    # --- error cascade (E6) -----------------------------------------------------------
    ber_implementation_penalty_sd_db: float = 0.85
    ber_temp_penalty_db_per_c: float = 0.018
    ber_slope_sd: float = 0.10                # per-device waterfall slope spread
    fec_overdispersion_k: float = 6.0         # negative-binomial shape (burstiness)
    fec_report_floor: int = 0
    crc_overdispersion_k: float = 2.0

    # --- demand metric (E4/E12) -------------------------------------------------------
    thr_base_log_mean: float = 4.79
    thr_base_log_sd: float = 0.50
    thr_base_min_mbps: float = 20.0
    thr_base_max_mbps: float = 800.0
    thr_floor: float = 0.35
    thr_morning_amp: float = 0.25
    thr_morning_peak_h: float = 9.0
    thr_morning_sd_h: float = 1.8
    thr_evening_amp: float = 0.90
    thr_evening_peak_h: float = 20.5
    thr_evening_sd_h: float = 2.2
    thr_weekend_factor: float = 1.15
    thr_ar_tau_h: float = 6.0
    thr_ar_sd: float = 0.22
    thr_impair_floor: float = 0.25
    thr_laser_impair: float = 0.70
    # v4: household archetypes, so the diurnal profile is not one fleet-wide shape
    thr_profile_weights: tuple = (0.45, 0.25, 0.20, 0.10)   # evening / day / night / flat

    # --- provenance / regime ----------------------------------------------------------
    config_role: str = "benchmark_enriched"
    # The scenario is part of the dataset's IDENTITY, not a comment. Without it, two
    # scenario-grid runs publish to directories distinguishable only by seed, and a
    # downstream result cannot name the environment it was computed under.
    scenario: str = "default"


    # --- output -----------------------------------------------------------------------
    out_path: str = "reference_dataset.parquet"
    gt_out_path: str = "gt_panel.parquet"
    batch_onts: int = 25

# ======================================================================================
# Catalogues
# ======================================================================================

# v4 (E8): magnitudes are lognormal, not uniform, so every optical family carries real
# mass BELOW the practical wander anchor (~0.14 dB day-over-day on this fixture). A
# fixture in which every fault is eventually visible cannot measure missed detection.
# `mag_mu`/`mag_sigma` are the lognormal parameters in dB; `mag_max` clips the tail.
FAULT_TYPES = {
    "connector_contamination": dict(
        weight=0.14, scope="ont", channel="optical", family="optical_ramp",
        onset_lead_h=(72, 240), shape="exp_ramp", mag_mu=0.20, mag_sigma=1.15, mag_max=9.0,
        recovery="full", scar_db=(0.0, 0.15), p_natural=0.25, natural_dwell_days=20.0,
        storm_sensitive=False),
    "cable_damage": dict(
        weight=0.08, scope="ont", channel="optical", family="optical_step",
        onset_lead_h=(0.5, 4.0), shape="step", mag_mu=1.10, mag_sigma=1.00, mag_max=16.0,
        recovery="full", scar_db=(0.05, 0.30), p_natural=0.0, natural_dwell_days=0.0,
        storm_sensitive=True),
    "fibre_bend": dict(
        weight=0.11, scope="ont", channel="optical", family="optical_ramp",
        onset_lead_h=(24, 96), shape="lin_ramp", mag_mu=-0.10, mag_sigma=1.15, mag_max=6.0,
        recovery="partial", scar_db=(0.1, 0.5), p_natural=0.50, natural_dwell_days=10.0,
        storm_sensitive=False),
    # v4 (E8): a genuinely intermittent mechanism. Loose or moisture-affected connectors
    # come and go; a persistence-based detector must not be able to assume monotonicity.
    "intermittent_connector": dict(
        weight=0.09, scope="ont", channel="optical", family="optical_intermittent",
        onset_lead_h=(12, 96), shape="intermittent", mag_mu=0.35, mag_sigma=1.05,
        mag_max=8.0, recovery="full", scar_db=(0.0, 0.2), p_natural=0.35,
        natural_dwell_days=14.0, storm_sensitive=False),
    "water_ingress": dict(
        weight=0.07, scope="ont", channel="optical", family="optical_ramp",
        onset_lead_h=(36, 240), shape="partial_recovery", mag_mu=0.55, mag_sigma=1.00,
        mag_max=10.0, recovery="partial", scar_db=(0.15, 0.8), p_natural=0.30,
        natural_dwell_days=18.0, storm_sensitive=True),
    # v4 (E4): the laser mechanism is no longer a single 22 mA rail. On v3.1 every
    # hardware failure drove bias by 22*lam^2 mA against a fleet-constant 0.22 mA noise
    # floor, so a 6*MAD rule on bias caught 50 of 50 with zero false positives. Gain is
    # now drawn per fault, with a soft mode in which the APC barely moves and the
    # signature sits on tx instead.
    "ont_hardware_failure": dict(
        weight=0.14, scope="ont", channel="laser", family="laser",
        onset_lead_h=(24, 144), shape="lin_ramp", mag_mu=None, mag_sigma=None, mag_max=0.0,
        recovery="full", scar_db=(0.0, 0.0), p_natural=0.0, natural_dwell_days=0.0,
        storm_sensitive=False,
        laser_gain_log_mu=1.35, laser_gain_log_sigma=1.15, laser_gain_max_ma=30.0,
        p_soft_mode=0.35, soft_tx_drop_db=(0.4, 2.2)),
    "thermal_instability": dict(
        weight=0.19, scope="ont", channel="optical_thermal", family="thermal",
        onset_lead_h=(48, 192), shape="thermal", mag_mu=0.30, mag_sigma=1.05, mag_max=7.0,
        recovery="partial", scar_db=(0.1, 0.4), p_natural=0.0, natural_dwell_days=0.0,
        storm_sensitive=False),
    "aging_fibre": dict(
        weight=0.07, scope="ont", channel="optical", family="optical_ramp",
        onset_lead_h=(480, 1440), shape="progressive", mag_mu=0.45, mag_sigma=0.95,
        mag_max=7.0, recovery="partial", scar_db=(0.3, 1.0), p_natural=0.0,
        natural_dwell_days=0.0, storm_sensitive=False),
    # ---- shared-scope mechanisms, now at FOUR levels (E11) --------------------------
    "splitter_degradation": dict(
        weight=0.05, scope="l2", channel="optical", family="optical_ramp",
        onset_lead_h=(24, 120), shape="exp_ramp", mag_mu=0.15, mag_sigma=1.05, mag_max=6.5,
        recovery="full", scar_db=(0.0, 0.2), p_natural=0.0, natural_dwell_days=0.0,
        storm_sensitive=True),
    "distribution_damage": dict(
        weight=0.02, scope="l1", channel="optical", family="optical_step",
        onset_lead_h=(0.5, 6.0), shape="step", mag_mu=1.30, mag_sigma=0.80, mag_max=14.0,
        recovery="full", scar_db=(0.05, 0.35), p_natural=0.0, natural_dwell_days=0.0,
        storm_sensitive=True),
    "feeder_degradation": dict(
        weight=0.015, scope="pon", channel="optical", family="optical_ramp",
        onset_lead_h=(48, 300), shape="lin_ramp", mag_mu=0.25, mag_sigma=0.80, mag_max=5.0,
        recovery="partial", scar_db=(0.1, 0.6), p_natural=0.0, natural_dwell_days=0.0,
        storm_sensitive=True),
    "olt_card_degradation": dict(
        weight=0.005, scope="olt", channel="optical", family="optical_ramp",
        onset_lead_h=(24, 240), shape="lin_ramp", mag_mu=0.10, mag_sigma=0.70, mag_max=3.5,
        recovery="full", scar_db=(0.0, 0.15), p_natural=0.0, natural_dwell_days=0.0,
        storm_sensitive=False),
}

SHARED_SCOPES = {"l2": "splitter_l2", "l1": "splitter_l1", "pon": "pon_port", "olt": "olt_id"}

# v4.1 (F1): how each mechanism presents in each direction.
#   bidirectional  plant loss -- fibre, splices, connectors, splitters -- attenuates the
#                  1490 nm downstream and 1310 nm upstream carriers alike, save for a
#                  wavelength term on bend-like mechanisms, which cost more at 1490.
#   downstream     an OLT transmit-path fault. The ONT sees it; the OLT receiver cannot.
#   upstream       an ONT laser fault. The OLT burst receiver sees it; the ONT cannot.
# `us_ratio` is the share of the downstream loss that also appears upstream.
for _ft, _spec in FAULT_TYPES.items():
    _spec.setdefault("direction",
                     "upstream" if _spec["channel"] == "laser" else "bidirectional")
    _spec.setdefault("us_ratio", 1.0)
FAULT_TYPES["olt_card_degradation"]["direction"] = "downstream"
FAULT_TYPES["fibre_bend"]["us_ratio"] = 0.62      # macrobend loss falls with wavelength
FAULT_TYPES["water_ingress"]["us_ratio"] = 0.78
FAULT_TYPES["aging_fibre"]["us_ratio"] = 0.85

# v4 (E4): tx and voltage are now device properties, not fleet constants. `tx_nominal`
# and `volt_nominal` vary by model; the per-device draw around them is what v3.1 lacked.
DEVICE_MODELS = {
    "HG8245H":  dict(vendor="Huawei", weight=0.26, rx_sensitivity_dbm=-27.0,
                     bias_nominal_ma=34.0, bias_thermal_coeff=0.16, bias_age_ma_per_yr=0.30,
                     tx_nominal_dbm=2.60, volt_nominal_v=3.300, fec_scale=1.00,
                     impl_penalty_db=0.00),
    "HG8546M":  dict(vendor="Huawei", weight=0.18, rx_sensitivity_dbm=-28.0,
                     bias_nominal_ma=36.5, bias_thermal_coeff=0.14, bias_age_ma_per_yr=0.26,
                     tx_nominal_dbm=2.85, volt_nominal_v=3.295, fec_scale=1.00,
                     impl_penalty_db=0.15),
    "EG8145V5": dict(vendor="Huawei", weight=0.16, rx_sensitivity_dbm=-27.5,
                     bias_nominal_ma=32.0, bias_thermal_coeff=0.18, bias_age_ma_per_yr=0.34,
                     tx_nominal_dbm=3.10, volt_nominal_v=3.310, fec_scale=1.00,
                     impl_penalty_db=-0.10),
    "G-240W-B": dict(vendor="Nokia", weight=0.15, rx_sensitivity_dbm=-28.5,
                     bias_nominal_ma=30.5, bias_thermal_coeff=0.12, bias_age_ma_per_yr=0.22,
                     tx_nominal_dbm=2.20, volt_nominal_v=3.280, fec_scale=0.55,
                     impl_penalty_db=0.35),
    "G-010S-A": dict(vendor="Nokia", weight=0.10, rx_sensitivity_dbm=-27.8,
                     bias_nominal_ma=29.0, bias_thermal_coeff=0.13, bias_age_ma_per_yr=0.25,
                     tx_nominal_dbm=2.45, volt_nominal_v=3.285, fec_scale=0.55,
                     impl_penalty_db=0.25),
    "F660":     dict(vendor="ZTE", weight=0.09, rx_sensitivity_dbm=-26.5,
                     bias_nominal_ma=35.5, bias_thermal_coeff=0.19, bias_age_ma_per_yr=0.38,
                     tx_nominal_dbm=3.40, volt_nominal_v=3.320, fec_scale=2.10,
                     impl_penalty_db=0.45),
    "F670L":    dict(vendor="ZTE", weight=0.06, rx_sensitivity_dbm=-27.2,
                     bias_nominal_ma=33.0, bias_thermal_coeff=0.17, bias_age_ma_per_yr=0.32,
                     tx_nominal_dbm=3.15, volt_nominal_v=3.315, fec_scale=2.10,
                     impl_penalty_db=0.30),
}

ENCLOSURES = {
    "indoor_wall": dict(weight=0.55, offset_c=14.0, diurnal_factor=0.45,
                        seasonal_factor=0.70, sensor_noise_sd_c=0.35, micro_sd_c=0.8,
                        phase_sd_h=1.1),
    "indoor_cabinet": dict(weight=0.25, offset_c=17.0, diurnal_factor=0.28,
                           seasonal_factor=0.60, sensor_noise_sd_c=0.30, micro_sd_c=0.6,
                           phase_sd_h=1.1),
    "outdoor_cabinet": dict(weight=0.20, offset_c=8.0, diurnal_factor=1.60,
                            seasonal_factor=1.00, sensor_noise_sd_c=0.80, micro_sd_c=2.4,
                            phase_sd_h=1.8),
}

# v4.1 (F7): the OMCI temperature attribute is the transceiver's own temperature --
# ambient plus self-heating, typically 35-55 C in service. v4.0 reported enclosure
# ambient, so the fleet sat near 15 C and `bias_thermal_coeff * (temp - 25.0)` was
# evaluated below its reference point on every entity, all of the time.
for _en, _spec in ENCLOSURES.items():
    _spec.setdefault("self_heat_c", 0.0)
ENCLOSURES["indoor_wall"]["self_heat_c"] = 24.0
ENCLOSURES["indoor_cabinet"]["self_heat_c"] = 29.0
ENCLOSURES["outdoor_cabinet"]["self_heat_c"] = 20.0

# v4 (E12): firmware is a real cohort with a rollout schedule, not a copy of the model.
FIRMWARE_BY_VENDOR = {
    "Huawei": ["V5R019C10", "V5R020C00", "V5R020C10"],
    "Nokia":  ["3FE-4.6.02", "3FE-4.7.01", "3FE-4.7.04"],
    "ZTE":    ["V2.1.3P4", "V2.2.0P1", "V2.2.1P2"],
}

THROUGHPUT_PROFILES = {
    "evening_peak": dict(morning=0.25, evening=0.90, night=0.05, floor=0.35),
    "day_worker":   dict(morning=0.70, evening=0.45, night=0.04, floor=0.40),
    "night_owl":    dict(morning=0.10, evening=0.55, night=0.65, floor=0.30),
    "flat_light":   dict(morning=0.15, evening=0.20, night=0.10, floor=0.55),
}


## 3. Topology

Variable fan-out (E1), route length composed as feeder + distribution + drop so peers under one splitter share their loss budget (E3), positional geography that groups PON ports across OLTs (E11), permuted identifiers (E13) and service windows (E14).

In [ ]:
# ======================================================================================
# Topology (E1, E2, E3, E13, E14)
# ======================================================================================


def _weighted_choice(rng, mapping, size=None):
    keys = list(mapping)
    p = np.array([mapping[k]["weight"] for k in keys], dtype=float)
    return rng.choice(keys, p=p / p.sum(), size=size)


def build_network_topology(cfg: TelecomSimulationSettings, rng: np.random.Generator) -> pd.DataFrame:
    """One row per ONT: static device, topology, geography and commercial attributes.

    v4 changes against v3.1, all measured defects:

      E1  Fan-out. v3.1 assigned subscribers round-robin, so every populated L2 splitter
          held exactly `l2_split` ONTs (measured sd 0.00 across 50 splitters). Here each
          installed splitter has its own capacity and a Beta take-up fraction.
      E3  Route length. v3.1 drew `distance_m` i.i.d. per ONT, giving a median within-
          splitter spread of 7,080 m for entities that physically share a feeder
          (within-L2 ICC 0.144). Length now composes as feeder + distribution + drop.
      E13 Identifiers. v3.1 numbered ONTs in nested loop order, so the identifier was a
          monotone function of topological position (r = 0.866 against OLT index) and an
          entity-disjoint split taken in identifier order was a topology split. The
          numbering is now a permutation.
      E14 Churn. A minority of entities are provisioned after t0 or churn away before
          the window ends, so the cold-entity protocol has genuinely unseen entities.
    """
    # ---- the physical tree ----------------------------------------------------------
    slots = []
    for o in range(1, cfg.n_olts + 1):
        for p in range(1, cfg.pon_ports_per_olt + 1):
            for a in range(1, cfg.l1_split + 1):
                slots.append((f"OLT-{o:02d}", f"PON-{o:02d}-{p:02d}",
                              f"SPL1-{o:02d}-{p:02d}", f"SPL2-{o:02d}-{p:02d}-{a:02d}"))

    # ---- geography: cluster centres, then nodes placed near their centre -------------
    # Geography is CORRELATED with, but not determined by, the OLT: an operator's
    # serving areas overlap at their edges. v3.1 assigned one letter per PON port, so
    # `geo_cluster` was a deterministic coarsening of `pon_port` and carried no
    # information a topology-aware model did not already have.
    n_geo = cfg.n_geo_clusters
    geo_names = [f"GEO-{i:02d}" for i in range(1, n_geo + 1)]
    geo_centre = {g: (float(rng.uniform(51.0, 55.5)), float(rng.uniform(-4.5, 0.5)))
                  for g in geo_names}
    olt_geo_prior = {}
    for o in sorted({s[0] for s in slots}):
        alpha = np.full(n_geo, 0.25)
        alpha[rng.integers(0, n_geo)] += 3.0          # a dominant serving area
        alpha[rng.integers(0, n_geo)] += 1.5          # and a secondary one
        olt_geo_prior[o] = rng.dirichlet(alpha)

    l1_pos = {}
    for olt, port, l1, _l2 in slots:
        if l1 in l1_pos:
            continue
        g = geo_names[int(rng.choice(n_geo, p=olt_geo_prior[olt]))]
        lat, lon = geo_centre[g]
        l1_pos[l1] = (lat + float(rng.normal(0, 0.13)), lon + float(rng.normal(0, 0.18)))

    # The L2 cabinet sits a short distance from its primary splitter, and its geographic
    # cluster follows POSITION, not the tree. v3.1 assigned one letter per PON port, so
    # `geo_cluster` was a deterministic function of `pon_port` -- it could not disagree
    # with the topology, and a storm footprint was therefore indistinguishable from a
    # PON-port fault. Here a cabinet near a serving-area boundary lands in the
    # neighbouring cluster, so geography and topology cross.
    centres = np.array([geo_centre[g] for g in geo_names])
    l2_geo, l2_pos = {}, {}
    for _o, _p, l1, l2 in slots:
        lat, lon = l1_pos[l1]
        pos = (lat + float(rng.normal(0, 0.05)), lon + float(rng.normal(0, 0.07)))
        d = np.hypot(centres[:, 0] - pos[0], centres[:, 1] - pos[1])
        l2_geo[l2] = geo_names[int(np.argmin(d))]
        l2_pos[l2] = pos

    # ---- route-length components, shared down the tree (E3) -------------------------
    feeder_km = {port: float(np.clip(rng.gamma(cfg.feeder_km_shape, cfg.feeder_km_scale),
                                     0.2, 14.0))
                 for _o, port, _l1, _l2 in slots}
    distribution_km = {l1: float(np.clip(rng.gamma(cfg.distribution_km_shape,
                                                   cfg.distribution_km_scale), 0.02, 3.5))
                       for _o, _p, l1, _l2 in slots}

    # ---- which splitters are installed, and how full (E1) ---------------------------
    # Deployment is contiguous: an operator brings a PON port into service and populates
    # its cabinets, rather than scattering single splitters across the whole estate.
    # Ports are taken round-robin across OLTs so no OLT is left with a handful of lines.
    by_olt = {}
    for _o, port, l1, l2 in slots:
        by_olt.setdefault(_o, {}).setdefault(port, []).append(l2)
    port_queue = []
    olt_ports = {o: list(rng.permutation(list(d))) for o, d in by_olt.items()}
    while any(olt_ports.values()):
        for o in sorted(olt_ports):
            if olt_ports[o]:
                port_queue.append(olt_ports[o].pop())
    slot_by_l2 = {s[3]: s for s in slots}

    caps = np.array(cfg.l2_capacity_choices)
    cap_w = np.array(cfg.l2_capacity_weights, dtype=float)
    cap_w = cap_w / cap_w.sum()
    chosen, remaining = [], cfg.n_onts
    for port in port_queue:
        if remaining <= 0:
            break
        olt = port.split("-")[1]
        l2s = by_olt[f"OLT-{olt}"][port]
        n_install = int(rng.integers(max(1, len(l2s) - 2), len(l2s) + 1))
        for l2 in rng.permutation(l2s)[:n_install]:
            if remaining <= 0:
                break
            cap = int(rng.choice(caps, p=cap_w))
            take = int(np.clip(round(cap * rng.beta(cfg.l2_takeup_beta_a, cfg.l2_takeup_beta_b)),
                               cfg.l2_min_fill, cap))
            take = min(take, remaining)
            chosen.append((slot_by_l2[l2], cap, take))
            remaining -= take
    if remaining > 0:
        raise ValueError(f"topology cannot host {cfg.n_onts} ONTs; {remaining} unplaced")

    # ---- per-ONT attributes ----------------------------------------------------------
    rows = []
    for (olt, port, l1, l2), cap, take in chosen:
        for _ in range(take):
            model = str(_weighted_choice(rng, DEVICE_MODELS))
            spec_m = DEVICE_MODELS[model]
            enclosure = str(_weighted_choice(rng, ENCLOSURES))
            spec_e = ENCLOSURES[enclosure]
            drop_m = float(np.clip(rng.gamma(cfg.drop_m_shape, cfg.drop_m_scale), 8.0, 400.0))
            dist_m = (feeder_km[port] + distribution_km[l1]) * 1000.0 + drop_m
            sens = spec_m["rx_sensitivity_dbm"]
            l2_loss = {4: 7.2, 8: 10.5, 16: 13.8, 32: 17.0}[cap]

            n_splices = int(rng.poisson(cfg.splices_per_km * dist_m / 1000.0)) + 1
            splice_loss = float(np.clip(rng.normal(cfg.splice_loss_mean_db,
                                                   cfg.splice_loss_sd_db, n_splices),
                                        0.02, 0.30).sum())
            extra_conn_loss = float(np.clip(
                rng.normal(cfg.extra_connector_loss_mean_db, cfg.extra_connector_loss_sd_db,
                           cfg.n_extra_connectors), 0.10, 1.20).sum())
            as_built_excess = float(rng.gamma(cfg.as_built_excess_shape, cfg.as_built_excess_scale))
            extra_plant = float(np.clip(splice_loss + extra_conn_loss + as_built_excess,
                                        cfg.extra_plant_loss_min_db, cfg.extra_plant_loss_max_db))
            conn_loss_i = float(np.clip(rng.normal(cfg.connector_loss_mean_db,
                                                   cfg.connector_loss_sd_db), 0.3, 3.5))
            margin_wo_extra = (cfg.olt_launch_dbm
                               - (cfg.l1_loss_db + l2_loss
                                  + dist_m / 1000.0 * cfg.fibre_loss_db_per_km + conn_loss_i)
                               - sens)
            # v4.1 (F4): commissioning aims at a TARGET drawn per line, so the realised
            # margin has a distribution rather than a spike on the clamp. The hard floor
            # remains, but as a last resort rather than the modal outcome.
            target_margin = float(rng.lognormal(np.log(cfg.target_margin_median_db),
                                                cfg.target_margin_log_sd))
            allowed = margin_wo_extra - target_margin
            extra_plant = float(min(extra_plant,
                                    max(allowed, cfg.extra_plant_loss_min_db)))
            extra_plant = float(min(extra_plant,
                                    max(margin_wo_extra - cfg.turnup_min_margin_db,
                                        cfg.extra_plant_loss_min_db)))

            fibre_age = float(np.round(rng.uniform(1, 15), 1))
            ont_age = float(np.round(np.clip(rng.gamma(2.2, 1.7), 0.1, min(fibre_age, 12.0)), 1))
            vendor = spec_m["vendor"]
            lat, lon = l2_pos[l2]
            profile = str(rng.choice(list(THROUGHPUT_PROFILES), p=np.array(cfg.thr_profile_weights)))

            rows.append(dict(
                olt_id=olt, pon_port=port, splitter_l1=l1, splitter_l2=l2,
                l2_splitter_capacity=cap, geo_cluster=l2_geo[l2],
                lat=round(lat + float(rng.normal(0, 0.012)), 5),
                lon=round(lon + float(rng.normal(0, 0.016)), 5),
                device_model=model, vendor=vendor, enclosure=enclosure,
                feeder_km=round(feeder_km[port], 3),
                distribution_km=round(distribution_km[l1], 3),
                drop_m=round(drop_m, 1),
                distance_m=round(dist_m),
                fibre_age_yr=fibre_age, ont_age_yr=ont_age,
                rx_sensitivity_dbm=sens,
                l2_loss_db=l2_loss,
                bias_nominal_ma=float(spec_m["bias_nominal_ma"] + rng.normal(0, 0.8)),
                bias_thermal_coeff=spec_m["bias_thermal_coeff"],
                bias_age_ma_per_yr=spec_m["bias_age_ma_per_yr"],
                # v4 (E4): the device channels that v3.1 emitted as fleet constants
                tx_nominal_dbm=float(spec_m["tx_nominal_dbm"] + rng.normal(0, cfg.tx_device_sd_db)),
                volt_nominal_v=float(spec_m["volt_nominal_v"] + rng.normal(0, cfg.volt_device_sd_v)),
                # v4 (E6): per-device receiver implementation penalty and waterfall slope
                impl_penalty_db=float(spec_m["impl_penalty_db"]
                                      + rng.normal(0, cfg.ber_implementation_penalty_sd_db)),
                ber_slope=float(np.clip(rng.normal(1.0, cfg.ber_slope_sd), 0.7, 1.35)),
                fec_scale=float(spec_m["fec_scale"] * np.exp(rng.normal(0, 0.25))),
                temp_sensor_noise_sd_c=spec_e["sensor_noise_sd_c"],
                connector_loss_db=conn_loss_i,
                extra_plant_loss_db=round(extra_plant, 3),
                noise_sd_db=float(np.clip(rng.gamma(4.0, 0.028), 0.03, 0.35) * cfg.sensor_noise_scale),
                bias_noise_sd_ma=float(np.clip(rng.gamma(cfg.bias_device_noise_shape,
                                                         cfg.bias_device_noise_scale), 0.08, 0.7)),
                tx_noise_scale=float(np.clip(rng.gamma(2.2, 0.45), 0.25, 3.0)),
                volt_noise_scale=float(np.clip(rng.gamma(2.2, 0.45), 0.25, 3.0)),
                solar_factor=float(np.round(rng.uniform(cfg.solar_exposure_low,
                                                        cfg.solar_exposure_high), 3)),
                service_impact_weight=float(np.round(rng.choice([0.5, 0.8, 1.0, 2.0, 3.0],
                                                                p=[.25, .30, .25, .15, .05]), 2)),
                customer_priority_weight=float(np.round(rng.choice([1.0, 1.5, 2.0, 4.0],
                                                                   p=[.60, .22, .13, .05]), 2)),
                temp_phase_shift_h=float(rng.normal(0, spec_e["phase_sd_h"])),
                enclosure_offset_c=spec_e["offset_c"],
                enclosure_self_heat_c=spec_e["self_heat_c"],
                enclosure_diurnal_factor=spec_e["diurnal_factor"],
                enclosure_seasonal_factor=spec_e["seasonal_factor"],
                microclimate_sd_c=spec_e["micro_sd_c"],
                thr_base_mbps=float(np.round(np.clip(rng.lognormal(cfg.thr_base_log_mean,
                                                                   cfg.thr_base_log_sd),
                                                     cfg.thr_base_min_mbps,
                                                     cfg.thr_base_max_mbps), 1)),
                thr_profile=profile,
                firmware_version=str(rng.choice(FIRMWARE_BY_VENDOR[vendor][:2], p=[0.62, 0.38])),
                # v4 (E10): "badly polled entity" is a real nuisance class
                poll_reliability=float(rng.beta(cfg.poll_reliability_beta_a,
                                                cfg.poll_reliability_beta_b)),
                # v4.1 (F9): ONT counters are LIFETIME counters. v4.0 started
                # `reboot_count` at zero inside the window while seeding `uptime_s` from
                # up to 55 hours, so the two disagreed about device history at t0.
                reboot_base=int(rng.poisson(2.5 * float(np.clip(rng.gamma(2.2, 1.7),
                                                                0.1, 12.0)))),
                # v4 (E9): unobservable frailty driving the fault hazard
                gt_frailty=float(rng.gamma(cfg.frailty_shape, 1.0 / cfg.frailty_shape)),
            ))

    t = pd.DataFrame(rows)

    # ---- E13: identifiers carry no topological information ---------------------------
    perm = rng.permutation(len(t))
    t["ont_id"] = [f"ONT-{i + 1:05d}" for i in perm]

    # ---- explicit chronic cohort (E5) -------------------------------------------------
    chronic = rng.random(len(t)) < cfg.chronic_share
    t["gt_chronic"] = chronic
    t.loc[chronic, "extra_plant_loss_db"] = (t.loc[chronic, "extra_plant_loss_db"]
                                             + cfg.chronic_extra_loss_db)
    t.loc[chronic, "noise_sd_db"] = t.loc[chronic, "noise_sd_db"] * cfg.chronic_noise_multiplier
    t["gt_noisy_plant"] = rng.random(len(t)) < cfg.crc_noisy_plant_share

    # ---- churn (E14) -------------------------------------------------------------------
    n = int(cfg.days * 24 * 60 / cfg.sample_minutes)
    t["install_i"] = 0
    late = rng.random(len(t)) < cfg.p_late_install
    t.loc[late, "install_i"] = rng.integers(int(0.08 * n), int(0.75 * n), int(late.sum()))
    t["decommission_i"] = n
    gone = (rng.random(len(t)) < cfg.p_decommission) & (~late)
    t.loc[gone, "decommission_i"] = rng.integers(int(0.30 * n), n, int(gone.sum()))

    # v4.1 (F1): the upstream span differs from the downstream span only in the fibre
    # term, since splitter, connector and splice losses are wavelength-flat to first
    # order over 1310/1490 nm.
    t["span_loss_ds_db"] = (cfg.l1_loss_db + t.l2_loss_db
                            + t.distance_m / 1000.0 * cfg.fibre_loss_db_per_km
                            + t.connector_loss_db + t.extra_plant_loss_db)
    t["span_loss_us_db"] = (cfg.l1_loss_db + t.l2_loss_db
                            + t.distance_m / 1000.0 * cfg.fibre_loss_us_db_per_km
                            + t.connector_loss_db + t.extra_plant_loss_db)

    t["distance_bucket"] = pd.cut(t.distance_m, [0, 2500, 6000, 1e9],
                                  labels=["near", "mid", "far"]).astype(str)
    t["splitter_ratio"] = "1:" + t.l2_splitter_capacity.astype(str)

    # Vendor-published expected RX from plant records: right on average, wrong on any
    # individual line, because as-built connector, splice and excess loss is not recorded.
    mean_extra = ((cfg.splices_per_km * t.distance_m / 1000.0 + 1.0) * cfg.splice_loss_mean_db
                  + cfg.n_extra_connectors * cfg.extra_connector_loss_mean_db
                  + cfg.as_built_excess_shape * cfg.as_built_excess_scale)
    span_loss = (cfg.l1_loss_db + t.l2_loss_db + t.distance_m / 1000.0 * cfg.fibre_loss_db_per_km
                 + cfg.connector_loss_mean_db + mean_extra)
    t["expected_rx_power_dbm"] = np.round(cfg.olt_launch_dbm - span_loss
                                          + rng.normal(0, 0.9, len(t)), 2)
    return t.reset_index(drop=True)


## 4. Healthy telemetry and the shared hierarchy

Six levels of shared structure — fleet, region, OLT, PON port, L1 and L2 splitter — where v3.1 had four (E2). `tx_power_dbm` and `voltage_v` become device properties rather than fleet constants (E4), and a labelled benign-anomaly layer stops the healthy class being noise-free (E5).

In [ ]:
# ======================================================================================
# Shared hierarchy (E2) and healthy channels (E4, E5, E12)
# ======================================================================================


def _ar1(n, tau_h, sd, sample_minutes, rng):
    """AR(1) started from its stationary distribution (v3.1 C7, retained)."""
    phi = float(np.exp(-(sample_minutes / 60.0) / max(tau_h, 1e-6)))
    eps = rng.normal(0, sd * np.sqrt(max(1 - phi ** 2, 1e-12)), n)
    zi = np.array([phi * rng.normal(0, sd)])
    out, _ = lfilter([1.0], [1.0, -phi], eps, zi=zi)
    return out


def _weather_components(cfg, ts_h, doy):
    """Ambient diurnal and seasonal components, anchored to the CALENDAR (F6).

    v4.0 took `day_frac = np.arange(n) / n` and wrote `4.0 * day_frac + 2.2 * sin(2 pi *
    day_frac * 1.5)`, so the seasonal cycle completed 1.5 times across the window
    whatever the window was. Measured: identical shape and a full 5.8 C excursion at 30,
    60, 180 and 365 days, always peaking 85% of the way through, and `cfg.start` had no
    effect on the physics at all. A 30-day January run and a 30-day July run were the
    same weather.
    """
    diurnal = cfg.diurnal_amplitude_c * np.cos(2 * np.pi * (ts_h - cfg.diurnal_peak_h) / 24.0)
    seasonal = cfg.seasonal_amplitude_c * np.cos(
        2 * np.pi * (doy - cfg.seasonal_peak_doy) / 365.25)
    return seasonal, diurnal


def build_shared_hierarchy(cfg, topo, ts, rng):
    """Effects shared by every entity beneath a node.

    v4 (E2) adds the two levels v3.1 omitted. Measured on the v3.1 reference run, median
    healthy hourly rx correlation was 0.575 within an L2 splitter against 0.564 within a
    PON port but a different L2 -- the splitter level contributed 0.011, so the finest
    and most specific cohort in the leave-one-out ladder had no shared component to
    remove. The C5 gate compared same-PON against cross-OLT only, and so could not see
    this. `splitter_l1` gets a distribution-segment loss factor and `splitter_l2` gets
    both a splitter-loss factor and a cabinet micro-weather factor.
    """
    n = len(ts)
    fleet = dict(
        weather_anomaly_c=_ar1(n, cfg.fleet_weather_tau_h, cfg.fleet_weather_sd_c,
                               cfg.sample_minutes, rng),
        demand_shock=_ar1(n, cfg.fleet_demand_tau_h, cfg.fleet_demand_sd,
                          cfg.sample_minutes, rng),
    )
    geo = {g: dict(weather_anomaly_c=_ar1(n, cfg.fleet_weather_tau_h, cfg.geo_weather_sd_c,
                                          cfg.sample_minutes, rng))
           for g in sorted(topo.geo_cluster.unique())}
    olt = {o: dict(launch_drift_db=_ar1(n, cfg.olt_launch_tau_h, cfg.olt_launch_sd_db,
                                        cfg.sample_minutes, rng))
           for o in sorted(topo.olt_id.unique())}
    # v4.1 (F1): the OLT's burst receiver has its own per-port calibration offset and
    # its own slow drift, INDEPENDENT of the downstream launch drift. Without that, the
    # upstream channel would be a deterministic reflection of the downstream one.
    pon = {p: dict(
        feeder_loss_db=_ar1(n, cfg.pon_feeder_tau_h, cfg.pon_feeder_sd_db, cfg.sample_minutes, rng),
        congestion=_ar1(n, cfg.pon_congestion_tau_h, cfg.pon_congestion_sd, cfg.sample_minutes, rng),
        olt_rx_drift_db=_ar1(n, cfg.olt_rx_ar_tau_h, cfg.olt_rx_ar_sd_db, cfg.sample_minutes, rng),
        olt_rx_cal_db=float(rng.normal(0, cfg.olt_rx_cal_sd_db)))
        for p in sorted(topo.pon_port.unique())}
    l1 = {s: dict(dist_loss_db=_ar1(n, cfg.l1_loss_tau_h, cfg.l1_loss_sd_db,
                                    cfg.sample_minutes, rng))
          for s in sorted(topo.splitter_l1.unique())}
    l2 = {s: dict(spl_loss_db=_ar1(n, cfg.l2_loss_tau_h, cfg.l2_loss_sd_db,
                                   cfg.sample_minutes, rng),
                  micro_weather_c=_ar1(n, cfg.l2_micro_weather_tau_h, cfg.l2_micro_weather_sd_c,
                                       cfg.sample_minutes, rng))
          for s in sorted(topo.splitter_l2.unique())}
    return dict(fleet=fleet, geo=geo, olt=olt, pon=pon, l1=l1, l2=l2)


def simulate_healthy_ont_signals(cfg, row, ts, ts_h, doy, shared, rng):
    """Per-ONT healthy trajectories: physical temperature, rx, bias base, tx base,
    voltage base, plus the log of benign operational changes and benign anomalies.

    v4 additions, all measured defects in v3.1:

      E4  `tx_power_dbm` was 3.05 + 0.004 dB/degC + N(0, 0.045) for every device in the
          fleet (measured between-entity sd of healthy medians: 0.0497 dB over six
          distinct quantised values), and `voltage_v` was 3.30 + N(0, 0.006) with a
          measured between-entity sd of 0.0000 V. Both are now device properties with
          real temperature, load and ageing behaviour.
      E5  Healthy data was Gaussian noise on a smooth mean: across 105 clean ONTs and
          180 days, no entity ever exceeded 6 robust sigma on bias, and 96.2% never did
          on rx. Sensor glitches, stuck values, transient bursts and re-ranging steps
          are added here and logged to `gt_benign_anomalies`.
      E12 Firmware rollouts, provisioning changes and predominantly-improving plant
          rework replace v3.1's zero-mean plant step as the benign-change layer.
    """
    n = len(ts)
    seasonal, diurnal = _weather_components(cfg, ts_h + row.temp_phase_shift_h, doy)
    events, benign = [], []

    # --- temperature: fleet -> region -> cabinet -> enclosure -> ONT microclimate ------
    micro = _ar1(n, cfg.microclimate_tau_h, row.microclimate_sd_c, cfg.sample_minutes, rng)
    temp_phys = (row.enclosure_offset_c
                 + row.enclosure_self_heat_c
                 + row.enclosure_seasonal_factor * seasonal
                 + row.enclosure_diurnal_factor * diurnal * row.solar_factor
                 + row.enclosure_seasonal_factor * (shared["fleet"]["weather_anomaly_c"]
                                                    + shared["geo"][row.geo_cluster]["weather_anomaly_c"])
                 + shared["l2"][row.splitter_l2]["micro_weather_c"]
                 + micro)

    # --- rx ----------------------------------------------------------------------------
    span_loss = (cfg.l1_loss_db + row.l2_loss_db
                 + row.distance_m / 1000.0 * cfg.fibre_loss_db_per_km
                 + row.connector_loss_db + row.extra_plant_loss_db)
    rx0 = cfg.olt_launch_dbm - span_loss
    ageing = -(0.02 / 30.0 / 24.0) * (1 + row.fibre_age_yr / 12.0) * np.arange(n) * (cfg.sample_minutes / 60.0)
    thermal = -0.018 * (temp_phys - temp_phys.mean())
    shared_optics = (shared["olt"][row.olt_id]["launch_drift_db"]
                     + shared["pon"][row.pon_port]["feeder_loss_db"]
                     + shared["l1"][row.splitter_l1]["dist_loss_db"]
                     + shared["l2"][row.splitter_l2]["spl_loss_db"])
    pink = np.cumsum(rng.normal(0, 0.004, n))
    pink -= pink.mean()

    # --- benign operational change (E12) ------------------------------------------------
    plant = np.zeros(n)
    for _ in range(rng.poisson(cfg.benign_plant_step_rate)):
        k = int(rng.integers(int(0.05 * n), n))
        improving = rng.random() < cfg.plant_step_improve_share
        delta = abs(rng.normal(0, cfg.plant_step_mean_db)) * (1.0 if improving else -1.0)
        plant[k:] += delta
        events.append(dict(entity_id=row.ont_id, ts_i=k, event_type="plant_rework",
                           level_change_db=round(float(delta), 3), detail=""))

    prov = np.zeros(n)
    for _ in range(rng.poisson(cfg.provisioning_change_rate_per_ont_year * cfg.days / 365.0)):
        k = int(rng.integers(int(0.05 * n), n))
        delta = float(rng.normal(0, 0.22))
        prov[k:] += delta
        events.append(dict(entity_id=row.ont_id, ts_i=k, event_type="provisioning_change",
                           level_change_db=round(delta, 3), detail="profile_change"))

    rx_healthy = rx0 + ageing + thermal + shared_optics + pink + plant + prov

    # --- bias: ONT age drives laser ageing ---------------------------------------------
    # v4: v3.1 wrote `bias_age_ma_per_yr * ont_age_yr / 12.0`, dividing an annual rate by
    # twelve, so a five-year-old ONT accumulated one twelfth of its ageing and `ont_age_yr`
    # had essentially no observable footprint.
    in_window_yr = np.arange(n) * (cfg.sample_minutes / 60.0) / (24.0 * 365.0)
    bias_base = (row.bias_nominal_ma
                 + row.bias_thermal_coeff * (temp_phys - 25.0)
                 + row.bias_age_ma_per_yr * row.ont_age_yr
                 + row.bias_age_ma_per_yr * in_window_yr)

    # --- tx: device offset, temperature coefficient, ageing, slow APC wander (E4) -------
    tx_base = (row.tx_nominal_dbm
               + cfg.tx_temp_coeff_db_per_c * (temp_phys - 25.0)
               + cfg.tx_age_db_per_yr * (row.ont_age_yr + in_window_yr)
               + _ar1(n, cfg.tx_ar_tau_h, cfg.tx_ar_sd_db * row.tx_noise_scale,
                      cfg.sample_minutes, rng))

    # --- voltage: device offset, temperature, load, slow wander (E4) --------------------
    volt_base = (row.volt_nominal_v
                 + cfg.volt_temp_coeff_v_per_c * (temp_phys - 25.0)
                 + _ar1(n, cfg.volt_ar_tau_h, cfg.volt_ar_sd_v * row.volt_noise_scale,
                        cfg.sample_minutes, rng))

    # --- firmware rollout: a step in reported bias and temperature, not in optics -------
    fw_step_i = None
    if rng.random() < 0.55:
        fw_step_i = int(rng.integers(int(0.15 * n), int(0.9 * n)))
        bias_base[fw_step_i:] += float(rng.normal(cfg.firmware_bias_step_ma, 0.15))
        temp_phys[fw_step_i:] += float(rng.normal(cfg.firmware_temp_step_c, 0.25))
        events.append(dict(entity_id=row.ont_id, ts_i=fw_step_i, event_type="firmware_upgrade",
                           level_change_db=0.0, detail="reported_counters_rebased"))

    # --- benign anomalies: the reason a 6-sigma rule must have a false-alarm rate (E5) --
    # Re-ranging: the ONT re-acquires the PON and its reported rx/bias step for a while.
    for _ in range(rng.poisson(cfg.reranging_rate_per_ont_year * cfg.days / 365.0)):
        k = int(rng.integers(0, n - 1))
        ln = int(np.clip(rng.exponential(20.0), 2, 300))
        rx_healthy[k:k + ln] += float(rng.normal(0, 0.55))
        bias_base[k:k + ln] += float(rng.normal(0, 1.1))
        tx_base[k:k + ln] += float(rng.normal(0, 0.45))
        benign.append(dict(entity_id=row.ont_id, ts_i=k, n_samples=int(ln),
                           gt_benign_type="re_ranging"))
    # Transient bursts: reflection or interference events on one or both channels.
    for k in np.flatnonzero(rng.random(n) < cfg.p_transient_burst):
        ln = int(np.clip(rng.exponential(cfg.transient_burst_mean_samples), 2, 60))
        sgn = 1.0 if rng.random() < 0.5 else -1.0
        rx_healthy[k:k + ln] += sgn * abs(rng.normal(0, cfg.transient_burst_rx_db))
        bias_base[k:k + ln] += sgn * abs(rng.normal(0, cfg.transient_burst_bias_ma))
        benign.append(dict(entity_id=row.ont_id, ts_i=int(k), n_samples=int(ln),
                           gt_benign_type="transient_burst"))

    return (temp_phys.astype(np.float32), rx_healthy.astype(np.float32),
            bias_base.astype(np.float32), tx_base.astype(np.float32),
            volt_base.astype(np.float32), events, benign)


## 5. Storms, fault arrivals, trajectories and the error cascade

Gamma frailty and weak covariate hazard give per-entity counts real overdispersion, and repaired lines are likelier to fail again (E9). Regional storms populate `group_id` across a geographic footprint that cuts across the topology tree (E11). Per-receiver implementation penalties, a temperature term, vendor counter scaling and burst overdispersion stop FEC being a noiseless readout of optical margin (E6).

In [ ]:
# ======================================================================================
# Storms, fault arrivals, trajectories (E8, E9, E11)
# ======================================================================================


def sample_storms(cfg, topo, rng) -> pd.DataFrame:
    """Regional weather events producing GROUPED faults (E11).

    v3.1 had no grouped-cause mechanism at all: `group_id` was null on all 293 faults and
    the only shared-cause pattern in existence was a single L2 splitter fault affecting
    exactly eight entities. Incident compression, root-cause top-k and shared-fault
    recall per domain are degenerate on that fixture. A storm raises the hazard of
    storm-sensitive mechanisms across one or two GEOGRAPHIC clusters -- which cut across
    the topology tree -- and also raises the collector-outage rate, so a correlation
    engine must separate "one region in trouble" from "one PON port in trouble".

    Rates and footprint are uncalibrated (OD3) and recorded as such in the provenance.
    """
    total_days = cfg.days + cfg.pre_window_days
    k = rng.poisson(cfg.storms_per_year / 365.0 * total_days)
    geos = sorted(topo.geo_cluster.unique())
    rows = []
    for i in range(int(k)):
        n_geo = int(np.clip(rng.poisson(cfg.storm_geo_clusters_mean), 1, max(1, len(geos))))
        rows.append(dict(
            group_id=f"STORM-{i + 1:03d}",
            start_day=float(rng.uniform(-cfg.pre_window_days, cfg.days)),
            duration_h=float(np.clip(rng.exponential(cfg.storm_duration_h_mean), 2.0, 96.0)),
            geo_clusters=",".join(sorted(rng.choice(geos, size=n_geo, replace=False))),
        ))
    return pd.DataFrame(rows, columns=["group_id", "start_day", "duration_h", "geo_clusters"])


def _entity_hazard(cfg, topo):
    """Relative per-entity fault hazard (E9).

    v3.1 drew fault targets uniformly at random, so per-entity counts were exactly
    Poisson (measured variance/mean 0.925) and no static attribute predicted them
    (|r| < 0.10 for distance, fibre age, ONT age, connector loss and excess plant loss).
    The plan's static-attribute susceptibility baseline therefore scores at chance by
    construction, and the post-MVP hazard model has nothing to learn. The signal here is
    deliberately weak: unobservable gamma frailty dominates the observable covariates, so
    susceptibility is discoverable but far from determined.
    """
    lin = (cfg.beta_fibre_age * topo.fibre_age_yr.to_numpy()
           + cfg.beta_outdoor * (topo.enclosure.to_numpy() == "outdoor_cabinet")
           + cfg.beta_distance_km * (topo.distance_m.to_numpy() / 1000.0)
           + cfg.beta_excess_loss_db * topo.extra_plant_loss_db.to_numpy())
    h = topo.gt_frailty.to_numpy() * np.exp(lin - lin.mean())
    return h / h.mean()


def sample_fault_events(cfg, topo, storms, rng) -> pd.DataFrame:
    """Inhomogeneous arrivals with frailty, covariates, storms and recurrence."""
    names = list(FAULT_TYPES)
    w = np.array([FAULT_TYPES[k]["weight"] for k in names], dtype=float)
    ont_mask = np.array([FAULT_TYPES[k]["scope"] == "ont" for k in names], dtype=float)
    ont_w = w * ont_mask
    ont_w = ont_w / ont_w.sum()

    total_days = cfg.days + cfg.pre_window_days
    haz = _entity_hazard(cfg, topo)
    ont_ids = topo.ont_id.to_numpy()
    ent_geo = dict(zip(topo.ont_id, topo.geo_cluster))
    rows, fid = [], 0

    def _storm_for(day, geo):
        for s in storms.itertuples():
            if s.start_day <= day <= s.start_day + s.duration_h / 24.0 and geo in s.geo_clusters.split(","):
                return s.group_id
        return None

    def _magnitude(spec, rng):
        if spec["mag_mu"] is None:
            return 0.0
        return float(np.clip(rng.lognormal(spec["mag_mu"], spec["mag_sigma"]), 0.02, spec["mag_max"]))

    def _emit(ft, scope, target, day, group_id=None, parent=None):
        nonlocal fid
        spec = FAULT_TYPES[ft]
        fid += 1
        gain_ma, tx_drop_db = 0.0, 0.0
        if spec["channel"] == "laser":
            soft = rng.random() < spec["p_soft_mode"]
            gain_ma = float(np.clip(rng.lognormal(spec["laser_gain_log_mu"],
                                                  spec["laser_gain_log_sigma"]),
                                    0.4, spec["laser_gain_max_ma"]))
            if soft:
                gain_ma = min(gain_ma, 3.5)
                tx_drop_db = float(rng.uniform(*spec["soft_tx_drop_db"]))
        return dict(
            gt_fault_id=f"F-{fid:05d}", gt_fault_type=ft, scope=scope, target=target,
            onset_day=day, onset_lead_h=rng.uniform(*spec["onset_lead_h"]),
            magnitude_db=_magnitude(spec, rng), scar_db=rng.uniform(*spec["scar_db"]),
            shape=spec["shape"], channel=spec["channel"], recovery=spec["recovery"],
            direction=spec["direction"], us_ratio=float(spec["us_ratio"]),
            group_id=group_id, gt_recurrence_of=parent,
            laser_gain_ma=gain_ma, laser_tx_drop_db=tx_drop_db,
        )

    # ---- entity-scope arrivals ---------------------------------------------------------
    n_ont_faults = rng.poisson(cfg.fault_rate_per_ont_year / 365.0 * total_days * cfg.n_onts)
    p_ent = haz / haz.sum()
    for _ in range(int(n_ont_faults)):
        j = int(rng.choice(len(ont_ids), p=p_ent))
        ft = str(rng.choice(names, p=ont_w))
        day = float(rng.uniform(-cfg.pre_window_days, cfg.days))
        gid = _storm_for(day, ent_geo[ont_ids[j]]) if FAULT_TYPES[ft]["storm_sensitive"] else None
        rows.append(_emit(ft, "ont", ont_ids[j], day, gid))

    # ---- storm-driven extra entity faults ----------------------------------------------
    storm_types = [k for k in names if FAULT_TYPES[k]["storm_sensitive"] and FAULT_TYPES[k]["scope"] == "ont"]
    sw = np.array([FAULT_TYPES[k]["weight"] for k in storm_types], dtype=float)
    sw = sw / sw.sum()
    for s in storms.itertuples():
        gl = s.geo_clusters.split(",")
        idx = np.flatnonzero(topo.geo_cluster.isin(gl).to_numpy())
        if not len(idx):
            continue
        base = cfg.fault_rate_per_ont_year / 365.0 / 24.0 * s.duration_h * len(idx)
        for _ in range(int(rng.poisson(base * (cfg.storm_hazard_multiplier - 1.0)))):
            j = int(rng.choice(idx, p=haz[idx] / haz[idx].sum()))
            ft = str(rng.choice(storm_types, p=sw))
            day = float(s.start_day + rng.uniform(0, s.duration_h / 24.0))
            rows.append(_emit(ft, "ont", ont_ids[j], day, s.group_id))

    # ---- shared-scope arrivals at four levels (E11) --------------------------------------
    scope_scale = dict(zip(["l2", "l1", "pon", "olt"], cfg.shared_scope_rate_scale))
    for scope, col in SHARED_SCOPES.items():
        rate = cfg.shared_fault_rate_per_node_year * scope_scale.get(scope, 1.0)
        types = [k for k in names if FAULT_TYPES[k]["scope"] == scope]
        if not types:
            continue
        tw = np.array([FAULT_TYPES[k]["weight"] for k in types], dtype=float)
        tw = tw / tw.sum()
        nodes = topo[col].unique()
        node_geo = topo.groupby(col).geo_cluster.first().to_dict()
        k_nodes = rng.poisson(rate / 365.0 * total_days * len(nodes))
        for _ in range(int(k_nodes)):
            node = str(rng.choice(nodes))
            ft = str(rng.choice(types, p=tw))
            day = float(rng.uniform(-cfg.pre_window_days, cfg.days))
            gid = _storm_for(day, node_geo[node]) if FAULT_TYPES[ft]["storm_sensitive"] else None
            rows.append(_emit(ft, scope, node, day, gid))
        # storm uplift on shared plant
        for s in storms.itertuples():
            gl = s.geo_clusters.split(",")
            cand = [nd for nd in nodes if node_geo[nd] in gl]
            if not cand:
                continue
            base = rate / 365.0 / 24.0 * s.duration_h * len(cand)
            for _ in range(int(rng.poisson(base * (cfg.storm_hazard_multiplier - 1.0)))):
                ft = str(rng.choice(types, p=tw))
                rows.append(_emit(ft, scope, str(rng.choice(cand)),
                                  float(s.start_day + rng.uniform(0, s.duration_h / 24.0)),
                                  s.group_id))

    # ---- recurrence: a repaired line is more likely to fail again (E9) -------------------
    base_rows = list(rows)
    for r in base_rows:
        if r["scope"] != "ont" or rng.random() >= cfg.p_recurrence:
            continue
        day = r["onset_day"] + float(rng.uniform(2.0, cfg.recurrence_window_days))
        if day >= cfg.days:
            continue
        rows.append(_emit(r["gt_fault_type"], "ont", r["target"], day,
                          None, r["gt_fault_id"]))

    f = pd.DataFrame(rows)
    if len(f):
        f["onset_ts"] = pd.Timestamp(cfg.start, tz="UTC") + pd.to_timedelta(f.onset_day * 24, unit="h")
    return f


def build_fault_degradation_curve(shape, n_from_onset, lead_samples, magnitude, temp_z=None,
                                  thermal_base_frac=0.60, thermal_temp_gain=0.35,
                                  rng=None, samples_per_h=4.0):
    """Unrepaired degradation trajectory in dB (negative = optical loss).

    v4 (E8) adds three trajectories v3.1 lacked. In v3.1 every mechanism rose
    monotonically to a fixed magnitude and held there until repair, so persistence was a
    free win for any detector and a "did it get better on its own" hypothesis never
    needed testing.

      intermittent      on/off duty cycle -- a loose or damp connector
      progressive       ramp, plateau, then a second acceleration
      partial_recovery  worsens, then partially self-heals and re-worsens
    """
    rng = rng if rng is not None else np.random.default_rng(0)
    x = np.arange(n_from_onset, dtype=np.float64)
    u = np.clip(x / max(lead_samples, 1.0), 0.0, 1.0)
    if shape == "step":
        d = np.where(u >= 1.0, 1.0, u ** 3)
    elif shape == "lin_ramp":
        d = u
    elif shape == "exp_ramp":
        d = (np.exp(2.6 * u) - 1.0) / (np.exp(2.6) - 1.0)
    elif shape == "thermal":
        tz = np.asarray(temp_z) if temp_z is not None else np.zeros(n_from_onset)
        if len(tz) < n_from_onset:
            tz = np.pad(tz, (0, n_from_onset - len(tz)), mode="edge") if len(tz) else np.zeros(n_from_onset)
        tz = tz[:n_from_onset]
        d = np.clip(u * (thermal_base_frac + thermal_temp_gain * np.clip(tz, -2.0, 3.0)), 0.0, 1.8)
    elif shape == "progressive":
        # ramp to a plateau at ~45% depth, hold, then accelerate
        d = np.where(u < 0.45, u / 0.45 * 0.45,
                     np.where(u < 0.75, 0.45, 0.45 + (u - 0.75) / 0.25 * 0.55))
        d = np.clip(d, 0, 1)
    elif shape == "partial_recovery":
        d = np.clip(u * 1.35, 0, 1.35)
        heal_start = int(0.55 * max(lead_samples, 1.0))
        heal_len = int(max(1, 0.5 * max(lead_samples, 1.0)))
        if heal_start < n_from_onset:
            seg = slice(heal_start, min(n_from_onset, heal_start + heal_len))
            d[seg] = d[seg] * np.linspace(1.0, 0.35, max(0, seg.stop - seg.start))
            if seg.stop < n_from_onset:
                tail = np.clip(np.arange(n_from_onset - seg.stop) / max(lead_samples, 1.0), 0, 1)
                d[seg.stop:] = d[seg.stop - 1] + tail * (1.0 - d[seg.stop - 1])
        d = np.clip(d, 0, 1.35)
    elif shape == "intermittent":
        d = np.zeros(n_from_onset)
        env = np.clip(x / max(lead_samples, 1.0), 0.15, 1.0)     # episodes deepen over time
        on_mean = max(2.0, 0.25 * samples_per_h * 6.0)
        off_mean = max(4.0, 0.25 * samples_per_h * 30.0)
        i = 0
        while i < n_from_onset:
            off = int(np.clip(rng.exponential(off_mean), 1, n_from_onset))
            i += off
            if i >= n_from_onset:
                break
            on = int(np.clip(rng.exponential(on_mean), 1, n_from_onset - i))
            d[i:i + on] = env[i:i + on] * float(np.clip(rng.normal(0.85, 0.2), 0.2, 1.2))
            i += on
    else:
        raise ValueError(shape)
    return -magnitude * d


def apply_repair_to_fault_curve(deg, onset_i, repair_i, recovery, scar_db, samples_per_h):
    """Repair recovers the loss over a truck-roll window, leaving a residual scar.
    v3.1 (C3): the repair may never deepen the loss. Retained unchanged."""
    if repair_i is None or repair_i >= len(deg):
        return deg
    level = deg[repair_i]
    end_level = -scar_db if recovery == "full" else min(-scar_db, level * 0.35)
    end_level = max(end_level, level)
    ramp = int(max(1, 0.75 * samples_per_h))
    stop = min(len(deg), repair_i + ramp)
    deg[repair_i:stop] = np.linspace(level, end_level, stop - repair_i)
    deg[stop:] = end_level
    return deg


def simulate_error_cascade(margin_db, rng, cfg, impl_penalty_db, ber_slope, fec_scale,
                           temp_c, is_noisy_plant):
    """margin -> pre-FEC BER -> FEC corrections -> post-FEC BER -> CRC errors.

    v4 (E6). In v3.1 a single global curve mapped margin to BER with no device,
    temperature or vendor variation and Poisson counting noise only, so `log10(fec_count)`
    recovered `gt_margin_db` with r = -0.988 and residual scatter of 0.30 decades --
    margin to +/- 0.33 dB, comparable to or better than the 0.1 dB-quantised rx
    observable. The cascade was a cleaner second copy of the fault-carrying variable
    rather than an independent channel. Here each receiver carries its own implementation
    penalty and waterfall slope, temperature adds a penalty, the vendor scales the
    reported counter, and errors arrive in bursts (negative binomial), not as a Poisson
    readout of the mean.
    """
    n = len(margin_db)
    secs = cfg.sample_minutes * 60
    eff_margin = (margin_db - impl_penalty_db
                  - cfg.ber_temp_penalty_db_per_c * np.maximum(temp_c - 35.0, 0.0))
    log_ber = np.clip(-3.0 - ber_slope * eff_margin, -12.0, -1.0)
    ber_true = 10.0 ** log_ber

    n_codewords = 2.488e9 * secs / 1904.0
    fec_mu = np.clip(n_codewords * 1904.0 * ber_true * fec_scale, 0, 5e6)
    k = cfg.fec_overdispersion_k
    p = k / (k + np.maximum(fec_mu, 1e-9))
    fec = rng.negative_binomial(k, np.clip(p, 1e-9, 1 - 1e-12)).astype(np.int64)
    fec = np.minimum(fec, np.int64(5e6))

    log_post = np.clip(9.0 * log_ber + 34.4, -16.0, -1.0)
    frames = 78e6 * secs / (1500 * 8)
    crc_mu = np.clip(frames * 12000.0 * (10.0 ** log_post), 0, 5e6)
    if is_noisy_plant:
        crc_mu = crc_mu * cfg.crc_noisy_plant_multiplier
    kc = cfg.crc_overdispersion_k
    pc = kc / (kc + np.maximum(crc_mu, 1e-9))
    crc = rng.negative_binomial(kc, np.clip(pc, 1e-9, 1 - 1e-12)).astype(np.int64)
    crc = np.minimum(crc, np.int64(5e6))

    # The ONT estimates pre-FEC BER from observed corrections; below a reporting floor it
    # returns zero, so `ber` really is censored (in v3.1 it was zero on 0.48% of rows and
    # the `continuous_censored` signal class had nothing to exercise).
    ber_obs = np.where(fec > 0, ber_true * np.exp(rng.normal(0, 0.30, n)), 0.0)
    ber_obs = np.where(ber_obs < 1e-9, 0.0, ber_obs)
    return ber_obs, fec, crc


## 6. Generation

Emits the generator's **native form only** — a wide panel plus sidecars. No canonical mapping happens here.

In [ ]:
# ======================================================================================
# Main generation
# ======================================================================================


def generate_telecom_reference_data(cfg: TelecomSimulationSettings | None = None,
                                    out_dir: str | Path = "."):
    cfg = cfg or TelecomSimulationSettings()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    master = np.random.default_rng(cfg.seed)

    topo = build_network_topology(cfg, master)
    storms = sample_storms(cfg, topo, master)
    faults = sample_fault_events(cfg, topo, storms, master)

    n = int(cfg.days * 24 * 60 / cfg.sample_minutes)
    t0 = pd.Timestamp(cfg.start, tz="UTC")
    ts = t0 + pd.to_timedelta(np.arange(n) * cfg.sample_minutes, unit="m")
    ts_h = (ts.hour + ts.minute / 60.0).to_numpy()
    # v4.1 (F6): real day-of-year, so `cfg.start` and `cfg.days` mean what they say.
    doy = ts.dayofyear.to_numpy().astype(float) + ts_h / 24.0
    sph = 60 / cfg.sample_minutes
    spd = int(24 * sph)
    weekend = np.asarray(ts.dayofweek >= 5)

    shared = build_shared_hierarchy(cfg, topo, ts, np.random.default_rng([cfg.seed, 999_999]))

    # ---- Phase A: healthy channels ----------------------------------------------------
    chan = {}
    eng_log, benign_log = [], []
    for i, row in enumerate(topo.itertuples()):
        rng = np.random.default_rng([cfg.seed, i])
        temp_p, rxh, biasb, txb, voltb, events, benign = simulate_healthy_ont_signals(
            cfg, row, ts, ts_h, doy, shared, rng)
        chan[row.ont_id] = (temp_p, rxh, biasb, txb, voltb)
        eng_log.extend(events)
        benign_log.extend(benign)

    # ---- Phase A2: robust per-entity baseline wander (practical anchor) ---------------
    wander_rx, wander_bias = {}, {}
    for o, (temp_c, rxh_c, bias_c, _tx, _v) in chan.items():
        dm = rxh_c[: cfg.days * spd].astype(np.float64).reshape(cfg.days, spd).mean(1)
        wander_rx[o] = float(1.4826 * np.median(np.abs(np.diff(dm))))
        db = bias_c[: cfg.days * spd].astype(np.float64).reshape(cfg.days, spd).mean(1)
        wander_bias[o] = float(1.4826 * np.median(np.abs(np.diff(db))))

    # ---- service windows (E14) --------------------------------------------------------
    install_i = dict(zip(topo.ont_id, topo.install_i))
    decom_i = dict(zip(topo.ont_id, topo.decommission_i))

    # ---- Phase B: unrepaired degradation per entity -----------------------------------
    episodes = []
    curve_rng = np.random.default_rng(cfg.seed + 31)
    for f in faults.itertuples():
        if f.scope == "ont":
            targets = [f.target]
        else:
            targets = topo.loc[topo[SHARED_SCOPES[f.scope]] == f.target, "ont_id"].tolist()
        onset_i = int(round(f.onset_day * 24 * sph))
        if onset_i >= n:
            continue
        for o in targets:
            start = max(onset_i, install_i[o], 0)
            if start >= min(n, decom_i[o]):
                continue
            m = n - start
            temp = chan[o][0]
            tz_full = (temp - temp.mean()) / max(temp.std(), 1e-6)
            offset = max(0, start - onset_i)
            tz_curve = np.empty(m + offset, dtype=np.float64)
            tz_curve[offset:] = tz_full[start:start + m]
            if offset:
                tz_curve[:offset] = tz_full[start]
            curve = build_fault_degradation_curve(
                f.shape, m + offset, f.onset_lead_h * sph, f.magnitude_db, tz_curve,
                cfg.thermal_base_frac, cfg.thermal_temp_gain, curve_rng, sph)[offset:]
            episodes.append(dict(gt_fault_id=f.gt_fault_id, ont_id=o,
                                 gt_fault_type=f.gt_fault_type, scope=f.scope,
                                 target=f.target, onset_i=onset_i, start=start,
                                 curve=curve, channel=f.channel, recovery=f.recovery,
                                 scar_db=f.scar_db, onset_lead_h=f.onset_lead_h,
                                 left_censored=onset_i < 0, group_id=f.group_id,
                                 direction=f.direction, us_ratio=f.us_ratio,
                                 laser_gain_ma=f.laser_gain_ma,
                                 laser_tx_drop_db=f.laser_tx_drop_db))

    ep_by_fault = {}
    for ep in episodes:
        ep_by_fault.setdefault(ep["gt_fault_id"], []).append(ep)

    sens = topo.set_index("ont_id").rx_sensitivity_dbm.to_dict()
    noise_sd = topo.set_index("ont_id").noise_sd_db.to_dict()
    rng2 = np.random.default_rng(cfg.seed + 7)

    ont_visits, node_visits = {}, {}
    for o in topo.ont_id:
        k = rng2.poisson(cfg.proactive_ont_visit_rate_per_year * cfg.days / 365.0)
        ont_visits[o] = np.sort(rng2.integers(0, n, k)) if k else np.array([], dtype=int)
    for col in SHARED_SCOPES.values():
        for nd in topo[col].unique():
            k = rng2.poisson(cfg.proactive_splitter_visit_rate_per_year * cfg.days / 365.0)
            node_visits[nd] = np.sort(rng2.integers(0, n, k)) if k else np.array([], dtype=int)
    onsite_samples = int(round(cfg.proactive_onsite_hours * sph))

    # ---- Phase C: impact, reporting, repair economy -----------------------------------
    fault_rows, ticket_records = [], []
    for f in faults.itertuples():
        eps = ep_by_fault.get(f.gt_fault_id, [])
        if not eps:
            continue
        spec = FAULT_TYPES[f.gt_fault_type]

        impacts_cf = {}
        for ep in eps:
            o = ep["ont_id"]
            if ep["channel"] == "laser":
                k = int(ep["onset_i"] + f.onset_lead_h * sph * 0.8)
                k = max(k, ep["start"])
                impacts_cf[o] = k if 0 <= k < n else None
                continue
            healthy_from_start = chan[o][1].astype(np.float64)[ep["start"]:]
            episode_margin = healthy_from_start + ep["curve"] - sens[o]
            max_search = int(np.ceil(f.onset_lead_h * sph * 1.5)) + cfg.impact_sustain_samples
            episode_margin = episode_margin[:max_search]
            below = episode_margin < cfg.impact_margin_db
            if below.sum() < cfg.impact_sustain_samples:
                impacts_cf[o] = None
                continue
            c = np.convolve(below.astype(int), np.ones(cfg.impact_sustain_samples), "valid")
            hit = np.flatnonzero(c >= cfg.impact_sustain_samples)
            attribution_min = max(cfg.impact_attribution_min_db, 2.2 * noise_sd[o])
            k_attr = None
            for h in hit:
                if abs(ep["curve"][int(h)]) >= attribution_min:
                    k_attr = int(h)
                    break
            impacts_cf[o] = ep["start"] + k_attr if k_attr is not None else None

        cf_valid = [v for v in impacts_cf.values() if v is not None]
        cf_first_impact = min(cf_valid) if cf_valid else None

        obs_sensor, obs_practical, lam_u_by_ont = [], [], {}
        for ep in eps:
            o, s = ep["ont_id"], ep["start"]
            if ep["channel"] == "laser":
                lead_s = max(1.0, ep["onset_lead_h"] * sph)
                lam_u = np.clip((np.arange(len(ep["curve"])) + (s - ep["onset_i"])) / lead_s, 0, 1)
                lam_u_by_ont[o] = lam_u
                k_s = max(s, int(np.ceil(ep["onset_i"] + 0.1414 * ep["onset_lead_h"] * sph)))
                thr_b = max(2 * noise_sd[o], 4 * wander_bias[o])
                gain = max(ep.get("laser_gain_ma", 22.0), 1e-6)
                lam_thr = float(np.sqrt(min(thr_b / gain, 1.0)))
                k_p = max(s, int(np.ceil(ep["onset_i"] + lam_thr * ep["onset_lead_h"] * sph)))
                if k_s < n:
                    obs_sensor.append(k_s)
                if k_p < n:
                    obs_practical.append(k_p)
            else:
                thr_s = 2.0 * noise_sd[o]
                thr_p = max(thr_s, 4.0 * wander_rx[o])
                hits_s = np.flatnonzero(np.abs(ep["curve"]) > thr_s)
                hits_p = np.flatnonzero(np.abs(ep["curve"]) > thr_p)
                if len(hits_s) and s + int(hits_s[0]) < n:
                    obs_sensor.append(s + int(hits_s[0]))
                if len(hits_p) and s + int(hits_p[0]) < n:
                    obs_practical.append(s + int(hits_p[0]))
        first_obs_sensor = min(obs_sensor) if obs_sensor else None
        first_obs_practical = min(obs_practical) if obs_practical else None

        sev_reports = []
        for ep in eps:
            o, s = ep["ont_id"], ep["start"]
            if ep["channel"] == "laser":
                lam_u = lam_u_by_ont[o]
                frac = np.clip((lam_u - cfg.laser_report_lam_floor)
                               / max(1.0 - cfg.laser_report_lam_floor, 1e-6), 0, 1)
                p_day = cfg.laser_report_hazard_per_day * frac
            else:
                depth = np.abs(ep["curve"])
                p_day = np.clip(cfg.report_hazard_per_db_day * (depth - cfg.report_depth_floor_db),
                                0, cfg.report_hazard_cap_per_day)
            hit = np.flatnonzero(rng2.random(len(p_day)) < p_day / spd)
            if len(hit):
                sev_reports.append((s + int(hit[0]), o))
        first_sev = min(t for t, _ in sev_reports) if sev_reports else None

        visits = ont_visits[f.target] if f.scope == "ont" else node_visits.get(f.target, np.array([], dtype=int))
        vs = visits[visits >= max(int(round(f.onset_day * 24 * sph)), 0)]
        proactive_t = int(vs[0]) + onsite_samples if len(vs) else None
        if proactive_t is not None and proactive_t >= n:
            proactive_t = None
        natural_t = None
        if spec["p_natural"] > 0 and rng2.random() < spec["p_natural"]:
            dwell = rng2.exponential(spec["natural_dwell_days"]) * spd
            natural_t = int(max(int(round(f.onset_day * 24 * sph)), -10 ** 9)
                            + f.onset_lead_h * sph + dwell)
            natural_t = natural_t if 0 <= natural_t < n else None
        mttr_samples = int(rng2.lognormal(np.log(cfg.mttr_median_h), cfg.mttr_sigma) * sph)

        cand = [t for t in [(first_sev + mttr_samples) if first_sev is not None else None,
                            proactive_t, natural_t] if t is not None and t < n]
        cand_repair = min(cand) if cand else None
        impact_realised = cf_first_impact is not None and (cand_repair is None or cf_first_impact < cand_repair)

        imp_reports = []
        if impact_realised:
            for ep in eps:
                k_o = impacts_cf[ep["ont_id"]]
                if k_o is None or (cand_repair is not None and k_o >= cand_repair):
                    continue
                if rng2.random() < cfg.p_ticket_given_impact:
                    if rng2.random() < cfg.report_delay_heavy_tail_p:
                        delay = rng2.exponential(cfg.report_delay_heavy_mean_h)
                    else:
                        delay = rng2.exponential(cfg.report_delay_mean_h)
                    imp_reports.append((k_o + int(delay * sph), ep["ont_id"]))

        all_reports = sev_reports + imp_reports
        first_report = min(t for t, _ in all_reports) if all_reports else None
        repair_options = []
        if first_report is not None and first_report + mttr_samples < n:
            repair_options.append((first_report + mttr_samples, "ticket"))
        if proactive_t is not None:
            repair_options.append((proactive_t, "proactive"))
        if natural_t is not None:
            repair_options.append((natural_t, "natural"))
        repair_i, repair_source = min(repair_options, key=lambda x: x[0]) if repair_options else (None, None)

        impacts_real = {o: (k if (k is not None and (repair_i is None or k < repair_i)) else None)
                        for o, k in impacts_cf.items()}
        real_valid = [v for v in impacts_real.values() if v is not None]
        first_impact = min(real_valid) if (impact_realised and real_valid) else None
        averted = bool(cf_first_impact is not None and first_impact is None)

        valid_reports = sorted([(t, o) for t, o in all_reports
                                if t < n and (repair_i is None or t < repair_i)])
        # v4 (E7): duplicates and mis-attribution
        extra = []
        for t_rep, o_rep in valid_reports:
            if rng2.random() < cfg.p_duplicate_ticket:
                extra.append((min(n - 1, t_rep + int(rng2.exponential(8.0) * sph)), o_rep))
        valid_reports = sorted(valid_reports + extra)
        n_tickets = len(valid_reports)

        for t_rep, o_rep in valid_reports:
            k_o = impacts_real.get(o_rep)
            if k_o is not None and k_o <= t_rep:
                symptom = "no_service"
            elif f.gt_fault_type == "ont_hardware_failure" and o_rep in lam_u_by_ont and \
                    lam_u_by_ont[o_rep][min(max(t_rep - max(int(round(f.onset_day * 24 * sph)), 0), 0),
                                            len(lam_u_by_ont[o_rep]) - 1)] > 0.6:
                symptom = "intermittent"
            else:
                symptom = "slow_service"
            reported_on = o_rep
            misattributed = False
            if rng2.random() < cfg.p_misattributed_ticket:
                peers = topo.loc[topo.splitter_l2 == topo.set_index("ont_id").loc[o_rep, "splitter_l2"],
                                 "ont_id"].tolist()
                if len(peers) > 1:
                    reported_on = str(rng2.choice([p for p in peers if p != o_rep]))
                    misattributed = True
            ticket_records.append(dict(
                ont_id=reported_on, reported_i=t_rep, repair_i=repair_i,
                reported_symptom=symptom, gt_fault_id=f.gt_fault_id,
                gt_fault_type=f.gt_fault_type, gt_is_nff=False,
                gt_misattributed=misattributed))

        prodromal = bool(first_obs_sensor is not None and first_impact is not None
                         and (first_impact - first_obs_sensor) >= 12 * sph)
        prodromal_practical = bool(first_obs_practical is not None and first_impact is not None
                                   and (first_impact - first_obs_practical) >= 12 * sph)

        fault_rows.append(dict(
            gt_fault_id=f.gt_fault_id, gt_fault_type=f.gt_fault_type, family=spec["family"],
            scope=f.scope, target=f.target, group_id=f.group_id,
            gt_recurrence_of=f.gt_recurrence_of,
            onset_i=int(round(f.onset_day * 24 * sph)),
            impact_i=first_impact, counterfactual_impact_i=cf_first_impact, averted=averted,
            repair_i=repair_i, repair_source=repair_source, n_tickets=n_tickets,
            first_observable_i=first_obs_sensor, first_observable_practical_i=first_obs_practical,
            prodromal=prodromal, prodromal_practical=prodromal_practical,
            magnitude_db=f.magnitude_db, onset_lead_h=f.onset_lead_h,
            gt_laser_gain_ma=round(float(f.laser_gain_ma), 3),
            gt_laser_tx_drop_db=round(float(f.laser_tx_drop_db), 3),
            n_onts_affected=len(eps), left_censored=bool(f.onset_day < 0)))

        for ep in eps:
            ep["impact_i"] = impacts_real[ep["ont_id"]]
            ep["n_tickets"] = n_tickets
            ep["repair_i"] = repair_i

    faults_out = pd.DataFrame(fault_rows)
    return _emit_panel(cfg, topo, faults, faults_out, episodes, storms, chan, shared,
                       eng_log, benign_log, ticket_records, ts, ts_h, doy, weekend,
                       n, sph, spd, t0, master, out_dir, install_i, decom_i)


def _q(x, q):
    return np.round(np.round(x / q) * q, 6)


def _alarm_runs(mask, min_len: int):
    """Contiguous True runs of at least `min_len` samples, as (start, end_exclusive)."""
    m = np.asarray(mask, dtype=np.int8)
    if not m.any():
        return []
    d = np.diff(np.concatenate(([0], m, [0])))
    starts, ends = np.flatnonzero(d == 1), np.flatnonzero(d == -1)
    return [(int(a), int(b)) for a, b in zip(starts, ends) if b - a >= min_len]


def _emit_panel(cfg, topo, faults, faults_out, episodes, storms, chan, shared,
                eng_log, benign_log, ticket_records, ts, ts_h, doy, weekend,
                n, sph, spd, t0, master, out_dir, install_i, decom_i):
    """Phase D: apply repair, emit observables, apply missingness, write everything.

    v4.1 changes, all responding to measurements on the v4.0.1 reference run:

      F1  Upstream. The OLT measures every ONU's burst. v4.0 emitted downstream rx only,
          so the asymmetry a field engineer localises with did not exist. Plant faults
          now attenuate both carriers, an OLT transmit-path fault moves downstream only,
          and an ONT laser fault moves upstream only.
      F3  Alarms. LOS is observable -- the OLT records it. v4.0 wrote it to the
          ground-truth gap log alone, where 97% of the rows sat on faulty entities, so a
          strong observable was being withheld and labelled ground truth.
      F8  Labels. v4.0 wrote per-sample state with last-writer-wins over episodes sorted
          by start, so an entity under a second active fault could be labelled
          `repaired`: measured 2,654 rows, about 15% of that class. State is now decided
          by PRECEDENCE (impaired > degrading > repaired > healthy) and the fault
          identity at each sample belongs to the earliest-onset active episode, which is
          order-independent.
      F9  Counters. `reboot_count` is a lifetime counter seeded per device, and a stuck
          collector record now freezes the error counters along with everything else.
          v4.0 froze six channels while `fec_count` and `crc_errors` carried on varying,
          which handed a two-channel consistency check a free separation of the benign
          class -- the exact separation E5 exists to deny.

    GROUND TRUTH LEAVES THE PANEL. The observable panel and `gt_panel.parquet` are
    written from the same loop in the same row order, so downstream can attach them
    positionally for whole-panel work or join on (ont_id, timestamp_utc) for a subset.
    """
    # ---- degradation with repair, resolved per direction (F1) -------------------------
    deg_ds = {o: np.zeros(n, dtype=np.float64) for o in topo.ont_id}
    deg_us = {o: np.zeros(n, dtype=np.float64) for o in topo.ont_id}
    laser_lead = {o: np.zeros(n, dtype=np.float64) for o in topo.ont_id}
    laser_bias = {o: np.zeros(n, dtype=np.float64) for o in topo.ont_id}
    laser_txdrop = {o: np.zeros(n, dtype=np.float64) for o in topo.ont_id}
    for ep in episodes:
        o, s = ep["ont_id"], ep["start"]
        curve = ep["curve"].copy()
        rep = ep.get("repair_i")
        rel_rep = None if rep is None else rep - s
        if rel_rep is not None and 0 <= rel_rep < len(curve):
            curve = apply_repair_to_fault_curve(curve, 0, rel_rep, ep["recovery"],
                                                ep["scar_db"], sph)
        if ep["channel"] == "laser":
            lead = max(1.0, ep["onset_lead_h"] * sph)
            lam = np.clip((np.arange(len(curve)) + (s - ep["onset_i"])) / lead, 0, 1)
            if rel_rep is not None and 0 <= rel_rep < len(curve):
                lam[rel_rep:] = 0.0
            laser_lead[o][s:] = np.maximum(laser_lead[o][s:], lam)
            laser_bias[o][s:] += ep["laser_gain_ma"] * lam ** 2
            laser_txdrop[o][s:] += ep["laser_tx_drop_db"] * lam ** 2
        else:
            if ep["direction"] in ("bidirectional", "downstream"):
                deg_ds[o][s:] += curve
            if ep["direction"] in ("bidirectional", "upstream"):
                deg_us[o][s:] += curve * ep["us_ratio"]

    ep_by_ont = {}
    for ep in episodes:
        ep_by_ont.setdefault(ep["ont_id"], []).append(ep)

    # ---- collector outages, at a COLLECTOR tier and storm-correlated (E10/E11) --------
    coll_of_olt = {o: f"COLL-{i % max(cfg.n_collectors, 1) + 1:02d}"
                   for i, o in enumerate(sorted(topo.olt_id.unique()))}
    storm_windows = [(int(round(s.start_day * 24 * sph)),
                      int(round((s.start_day + s.duration_h / 24.0) * 24 * sph)))
                     for s in storms.itertuples()]
    outage = {}
    for coll in sorted(set(coll_of_olt.values())):
        mask = np.zeros(n, dtype=bool)
        k = master.poisson(cfg.collector_outage_per_collector_per_month * cfg.days / 30.0)
        for _ in range(int(k)):
            st = int(master.integers(0, n))
            ln = int(master.exponential(cfg.collector_outage_hours_mean * sph)) + 1
            mask[st:st + ln] = True
        for a, b in storm_windows:
            extra = master.poisson(max(0.0, (cfg.storm_collector_outage_multiplier - 1.0)
                                       * cfg.collector_outage_per_collector_per_month / 30.0
                                       * max(b - a, 1) / spd))
            for _ in range(int(extra)):
                st = int(master.integers(max(a, 0), max(min(b, n), max(a, 0) + 1)))
                ln = int(master.exponential(cfg.collector_outage_hours_mean * sph)) + 1
                mask[st:st + ln] = True
        outage[coll] = mask

    # ---- planned maintenance windows per PON port (E12) -------------------------------
    maint = {}
    for port in topo.pon_port.unique():
        mask = np.zeros(n, dtype=bool)
        k = master.poisson(cfg.planned_maintenance_per_pon_per_year * cfg.days / 365.0)
        for _ in range(int(k)):
            st = int(master.integers(0, n))
            ln = int(master.exponential(cfg.planned_maintenance_hours_mean * sph)) + 1
            mask[st:st + ln] = True
            eng_log.append(dict(entity_id=port, ts_i=st, event_type="planned_maintenance",
                                level_change_db=0.0, detail=f"{ln} samples"))
        maint[port] = mask

    conv_samples = int(cfg.repaired_convalescence_days * spd)
    gap_reason_log, alarm_log = [], []
    writer = gt_writer = None
    batch, gt_batch, schema, gt_schema = [], [], None, None
    topo_idx = topo.set_index("ont_id")
    hh_e = ((ts_h - cfg.thr_evening_peak_h + 12) % 24) - 12
    hh_m = ((ts_h - cfg.thr_morning_peak_h + 12) % 24) - 12
    hh_n = ((ts_h - 2.5 + 12) % 24) - 12
    ts_ns = ts.tz_convert("UTC").tz_localize(None).to_numpy()

    # Columns the panel carries. Everything else an operator holds about a line -- plant
    # records, commercial weights, geography -- lives in `topology.csv`, because a
    # 15-minute telemetry record does not carry a customer priority weight and an adapter
    # bound to a panel that does is bound to a shape no OSS emits.
    PANEL_STATIC = ["olt_id", "pon_port", "splitter_l1", "splitter_l2", "device_model",
                    "vendor", "firmware_version"]

    for i, row in enumerate(topo.itertuples()):
        o = row.ont_id
        rng = np.random.default_rng([cfg.seed, 10_000 + i])
        temp, rxh, bias_base, tx_base, volt_base = [c.astype(np.float64) for c in chan[o]]
        d_ds, d_us = deg_ds[o], deg_us[o]
        lam = laser_lead[o]
        lbias = laser_bias[o]
        ltx = laser_txdrop[o]

        rx = rxh + d_ds + rng.normal(0, row.noise_sd_db, n)
        margin = rx - row.rx_sensitivity_dbm

        bias = bias_base + lbias + rng.normal(0, row.bias_noise_sd_ma, n)
        # The power the ONT actually launches, before its own reporting noise.
        tx_true = tx_base - ltx - 4.5 * np.clip(lam - 0.85, 0, 1) / 0.15
        tx = tx_true + rng.normal(0, 0.075 * row.tx_noise_scale, n)

        # ---- F1: what the OLT's burst receiver measures for this ONU ------------------
        olt_rx = (tx_true - row.span_loss_us_db + d_us
                  + shared["pon"][row.pon_port]["olt_rx_drift_db"]
                  + shared["pon"][row.pon_port]["olt_rx_cal_db"]
                  + rng.normal(0, cfg.olt_rx_noise_sd_db, n))
        olt_margin = olt_rx - cfg.olt_rx_sensitivity_dbm

        prof = THROUGHPUT_PROFILES[row.thr_profile]
        shape_d = (prof["floor"]
                   + prof["morning"] * np.exp(-hh_m ** 2 / (2 * cfg.thr_morning_sd_h ** 2))
                   + prof["evening"] * np.exp(-hh_e ** 2 / (2 * cfg.thr_evening_sd_h ** 2))
                   + prof["night"] * np.exp(-hh_n ** 2 / (2 * 2.0 ** 2)))
        wk = np.where(weekend, cfg.thr_weekend_factor, 1.0)
        phi_t = float(np.exp(-(cfg.sample_minutes / 60.0) / cfg.thr_ar_tau_h))
        ar = lfilter([1.0], [1.0, -phi_t], rng.normal(0, cfg.thr_ar_sd * np.sqrt(1 - phi_t ** 2), n))
        avail = np.clip(cfg.thr_impair_floor + (1 - cfg.thr_impair_floor)
                        * np.clip(margin / cfg.impact_margin_db, 0, 1), cfg.thr_impair_floor, 1.0)
        avail = avail * (1 - cfg.thr_laser_impair * np.clip(lam - 0.6, 0, 0.4) / 0.4)
        contention = np.clip(shared["pon"][row.pon_port]["congestion"], 0, None)
        shared_demand = np.exp(shared["fleet"]["demand_shock"]) / (1.0 + contention)
        throughput = row.thr_base_mbps * shape_d * wk * np.exp(ar) * shared_demand * avail
        load = np.clip(throughput / max(row.thr_base_mbps, 1e-6), 0, 3.0)

        volt = (volt_base + cfg.volt_load_coeff_v * load - 0.05 * lam
                + rng.normal(0, 0.006 * row.volt_noise_scale, n))
        temp_obs = (temp + 0.06 * (bias - bias_base)
                    + rng.normal(0, row.temp_sensor_noise_sd_c * cfg.sensor_noise_scale, n))

        ber, fec, crc = simulate_error_cascade(margin, rng, cfg, row.impl_penalty_db,
                                               row.ber_slope, row.fec_scale, temp,
                                               bool(row.gt_noisy_plant))
        burst_mask = rng.random(n) < cfg.crc_burst_prob_per_sample
        if burst_mask.any():
            sizes = np.maximum(1, np.round(rng.lognormal(cfg.crc_burst_lognorm_mu,
                                                         cfg.crc_burst_lognorm_sigma,
                                                         int(burst_mask.sum())))).astype(np.int64)
            crc = crc.copy()
            crc[burst_mask] += sizes

        reboot_p = 0.0006 + 0.05 * lam ** 2 + 0.02 * (margin < 0.5)
        reboot = rng.random(n) < reboot_p
        cpe_cycle = rng.random(n) < (cfg.p_cpe_power_cycle_per_day / spd)
        reboot = reboot | cpe_cycle
        uptime = np.zeros(n)
        acc = int(rng.integers(0, 200_000))
        for k in range(n):
            acc = 0 if reboot[k] else acc + cfg.sample_minutes * 60
            uptime[k] = acc
        # v4.1 (F9): lifetime counter, seeded per device from its age.
        reboot_count = int(row.reboot_base) + np.cumsum(reboot)

        # ---- benign anomaly layer applied to the OBSERVABLES (E5) ---------------------
        ent_benign = []
        # A sensor glitch is a READOUT defect: it moves the reported value and nothing
        # physical, so it deliberately does NOT propagate to the error counters.
        gl = np.flatnonzero(rng.random(n) < cfg.p_sensor_glitch)
        for k in gl:
            which = rng.integers(0, 4)
            if which == 0:
                rx[k] += float(rng.normal(0, 3.0))
            elif which == 1:
                bias[k] += float(rng.normal(0, 6.0))
            elif which == 2:
                temp_obs[k] += float(rng.normal(0, 9.0))
            else:
                olt_rx[k] += float(rng.normal(0, 3.0))
            ent_benign.append((int(k), 1, "sensor_glitch"))
        # A stuck run is a CACHED RECORD: v4.1 freezes the counters with it.
        for k in np.flatnonzero(rng.random(n) < cfg.p_stuck_start):
            ln = int(np.clip(rng.exponential(cfg.stuck_run_mean_samples), 2, 200))
            sl = slice(int(k), min(n, int(k) + ln))
            for arr in (rx, bias, tx, olt_rx, volt, temp_obs, throughput, ber, uptime):
                arr[sl] = arr[int(k)]
            for arr in (fec, crc, reboot_count):
                arr[sl] = arr[int(k)]
            ent_benign.append((int(k), int(sl.stop - sl.start), "stuck_value"))
        for k, ln, kind in ent_benign:
            benign_log.append(dict(entity_id=o, ts_i=k, n_samples=ln, gt_benign_type=kind))

        # ---- observable panel ---------------------------------------------------------
        df = pd.DataFrame({
            "timestamp_utc": ts,
            "ont_id": o,
            "rx_power_dbm": _q(rx, cfg.quant_rx_db),
            "tx_power_dbm": _q(tx, cfg.quant_tx_db),
            "olt_rx_power_dbm": _q(olt_rx, cfg.quant_olt_rx_db),
            "temperature_c": _q(temp_obs, cfg.quant_temp_c),
            "bias_current_ma": _q(bias, cfg.quant_bias_ma),
            "voltage_v": _q(volt, cfg.quant_volt_v),
            "ber": ber,
            "fec_count": fec,
            "crc_errors": crc,
            "uptime_s": uptime,
            "reboot_count": reboot_count,
            "throughput_mbps": _q(throughput, cfg.quant_throughput_mbps),
        })
        for c in PANEL_STATIC:
            df[c] = topo_idx.loc[o, c]

        # ---- ground-truth panel, same rows, same order --------------------------------
        gt = pd.DataFrame({
            "timestamp_utc": ts,
            "ont_id": o,
            "gt_rx_healthy_dbm": np.round(rxh, 3),
            "gt_optical_degradation_db": np.round(d_ds, 4),
            "gt_optical_degradation_us_db": np.round(d_us, 4),
            "gt_laser_degradation": np.round(lam, 4),
        })

        # ---- state by PRECEDENCE, identity by earliest onset (F8) ---------------------
        n_active = np.zeros(n, dtype=np.int16)
        n_impaired = np.zeros(n, dtype=np.int16)
        conv = np.zeros(n, dtype=bool)
        best_onset = np.full(n, np.iinfo(np.int64).max, dtype=np.int64)
        fid_a = np.array([pd.NA] * n, dtype=object)
        ftype_a = np.array([pd.NA] * n, dtype=object)
        onset_a = np.full(n, np.datetime64("NaT"), dtype="datetime64[ns]")
        impact_a = np.full(n, np.datetime64("NaT"), dtype="datetime64[ns]")
        repair_a = np.full(n, np.datetime64("NaT"), dtype="datetime64[ns]")
        lc_a = np.zeros(n, dtype=bool)
        sh_a = np.zeros(n, dtype=bool)
        for ep in ep_by_ont.get(o, []):
            s = ep["start"]
            imp, rep = ep.get("impact_i"), ep.get("repair_i")
            end = rep if rep is not None else n
            if end <= s:
                continue
            n_active[s:end] += 1
            if imp is not None and imp < end:
                n_impaired[max(imp, s):end] += 1
            if rep is not None:
                conv[rep:min(rep + conv_samples, n)] = True
            w = np.flatnonzero(best_onset[s:end] > ep["onset_i"]) + s
            if len(w):
                best_onset[w] = ep["onset_i"]
                fid_a[w] = ep["gt_fault_id"]
                ftype_a[w] = ep["gt_fault_type"]
                onset_a[w] = ts_ns[max(ep["onset_i"], 0)]
                impact_a[w] = ts_ns[imp] if imp is not None else np.datetime64("NaT")
                repair_a[w] = ts_ns[rep] if rep is not None else np.datetime64("NaT")
                lc_a[w] = ep["left_censored"]
                sh_a[w] = ep["scope"] != "ont"
        state = np.where(n_impaired > 0, "impaired",
                         np.where(n_active > 0, "degrading",
                                  np.where(conv, "repaired", "healthy"))).astype(object)
        gt["gt_state"] = state
        gt["gt_fault_id"] = fid_a
        gt["gt_fault_type"] = ftype_a
        gt["gt_onset_ts"] = pd.to_datetime(onset_a, utc=True)
        gt["gt_impact_ts"] = pd.to_datetime(impact_a, utc=True)
        gt["gt_repair_ts"] = pd.to_datetime(repair_a, utc=True)
        gt["gt_left_censored"] = lc_a
        gt["gt_shared_fault"] = sh_a
        gt["gt_active_fault_count"] = n_active

        # ---- missingness (E10) --------------------------------------------------------
        p_drop = 1.0 - row.poll_reliability
        m_dropout = rng.random(n) < p_drop
        # v4.1 (F1): loss of signal is a two-sided condition. A failing ONT laser takes
        # the upstream burst below the OLT receiver's sensitivity while the downstream
        # reading stays healthy -- in v4.0 a laser fault never produced LOS at all.
        los_condition = (margin < 0.0) | (olt_margin < 0.0)
        m_los = rng.random(n) < np.clip(0.35 * los_condition, 0, 1)
        m_outage = outage[coll_of_olt[row.olt_id]]
        m_maint = maint[row.pon_port]
        m_reboot = reboot
        m_benign = np.zeros(n, dtype=bool)
        benign_starts = []
        for _ in range(int(rng.poisson(cfg.benign_outage_rate_per_ont_year * cfg.days / 365.0))):
            st = int(rng.integers(0, n))
            ln = int(rng.exponential(cfg.benign_outage_hours_mean * sph)) + 1
            m_benign[st:st + ln] = True
            benign_starts.append(st)
        for _ in range(int(rng.poisson(cfg.holiday_absence_rate_per_ont_year * cfg.days / 365.0))):
            st = int(rng.integers(0, n))
            ln = int(rng.exponential(cfg.holiday_absence_days_mean * spd)) + 1
            m_benign[st:st + ln] = True
        m_churn = np.ones(n, dtype=bool)
        m_churn[install_i[o]:decom_i[o]] = False

        # ---- F3: the alarm channel, observable ----------------------------------------
        if cfg.emit_alarms:
            in_service = ~m_churn
            # An ONU that stops answering IS loss of signal from the OLT's point of view,
            # whatever the cause -- a fibre break and a customer pulling the plug look
            # identical on the port. What separates them is whether a dying gasp arrived
            # first. Raising LOS on the optical condition ALONE would have made the alarm
            # a perfect fault oracle: measured 0.0% of `los` events on clean entities
            # before this change.
            soak = max(1, int(round(cfg.alarm_sustain_minutes / cfg.sample_minutes)))
            for a, b in _alarm_runs((m_los | m_benign) & in_service, soak):
                alarm_log.append(dict(entity_id=o, alarm_type="los",
                                      raised_ts=ts[a], cleared_ts=ts[min(b, n - 1)],
                                      duration_samples=int(b - a)))
            for a, b in _alarm_runs((ber > cfg.alarm_sf_ber_threshold) & in_service, soak):
                alarm_log.append(dict(entity_id=o, alarm_type="signal_fail",
                                      raised_ts=ts[a], cleared_ts=ts[min(b, n - 1)],
                                      duration_samples=int(b - a)))
            for a, b in _alarm_runs((ber > cfg.alarm_sd_ber_threshold)
                                    & (ber <= cfg.alarm_sf_ber_threshold) & in_service, soak):
                alarm_log.append(dict(entity_id=o, alarm_type="signal_degrade",
                                      raised_ts=ts[a], cleared_ts=ts[min(b, n - 1)],
                                      duration_samples=int(b - a)))
            # A premises power cut reports a dying gasp; a fibre break does not.
            for st in benign_starts:
                if not m_churn[st] and rng.random() < cfg.alarm_dying_gasp_p:
                    alarm_log.append(dict(entity_id=o, alarm_type="dying_gasp",
                                          raised_ts=ts[st], cleared_ts=pd.NaT,
                                          duration_samples=1))

        drop = m_dropout | m_los | m_outage | m_reboot | m_benign | m_maint | m_churn
        if (drop & ~m_churn).any():
            reason = np.empty(n, dtype=object)
            reason[m_dropout] = "random_dropout"
            reason[m_reboot] = "cpe_restart"
            reason[m_benign] = "premises_outage"
            reason[m_los] = "loss_of_signal"
            reason[m_maint] = "planned_maintenance"
            reason[m_outage] = "collector_outage"
            gi = np.flatnonzero(drop & ~m_churn)
            gap_reason_log.append(pd.DataFrame({"entity_id": o, "ts": ts[gi],
                                                "gt_gap_reason": reason[gi]}))
        keep = ~drop
        df = df.loc[keep].copy()
        gt = gt.loc[keep].copy()

        for c in ["rx_power_dbm", "tx_power_dbm", "olt_rx_power_dbm", "temperature_c",
                  "bias_current_ma", "voltage_v", "ber", "fec_count", "crc_errors",
                  "throughput_mbps"]:
            m = rng.random(len(df)) < cfg.p_field_nan
            df.loc[m, c] = np.nan

        batch.append(df)
        gt_batch.append(gt)
        if len(batch) >= cfg.batch_onts or i == len(topo) - 1:
            out = pd.concat(batch, ignore_index=True)
            gt_out = pd.concat(gt_batch, ignore_index=True)
            for c in ["ont_id"] + PANEL_STATIC:
                out[c] = out[c].astype("string")
            for c in ["ont_id", "gt_state", "gt_fault_id", "gt_fault_type"]:
                gt_out[c] = gt_out[c].astype("string")
            leaked = [c for c in out.columns if c.startswith(FEATURE_BLOCKLIST_PREFIX)]
            if leaked:
                raise AssertionError(f"ground truth reached the observable panel: {leaked}")
            tbl = pa.Table.from_pandas(out, preserve_index=False)
            gt_tbl = pa.Table.from_pandas(gt_out, preserve_index=False)
            if writer is None:
                schema, gt_schema = tbl.schema, gt_tbl.schema
                writer = pq.ParquetWriter(out_dir / cfg.out_path, schema, compression="zstd")
                gt_writer = pq.ParquetWriter(out_dir / cfg.gt_out_path, gt_schema,
                                             compression="zstd")
            writer.write_table(tbl.cast(schema))
            gt_writer.write_table(gt_tbl.cast(gt_schema))
            batch, gt_batch = [], []
    if writer is not None:
        writer.close()
        gt_writer.close()

    return _write_sidecars(cfg, topo, faults_out, episodes, storms, eng_log, benign_log,
                           ticket_records, gap_reason_log, alarm_log, ts, n, sph, t0,
                           master, out_dir, install_i, decom_i)


def _write_sidecars(cfg, topo, faults_out, episodes, storms, eng_log, benign_log,
                    ticket_records, gap_reason_log, alarm_log, ts, n, sph, t0, master,
                    out_dir, install_i, decom_i):
    """Registry, tickets, intervals, events and provenance.

    v4 (E7): ticket identifiers are OPAQUE. v3.1 issued `TKT-{fault_id[2:]}-{k:02d}` for
    customer tickets and `TKT-NFF{j:04d}` for no-fault-found, so the canonical
    `service_tickets` table -- the table the Week-2 ticket-proxy evaluation and the
    Week-5 localisation check are scored against -- handed over exact fault grouping,
    exact cross-entity shared-fault grouping and exact NFF identification from a string
    prefix. Resolution timestamps are now per ticket, some tickets never resolve, and
    duplicates and mis-attributions are present.
    """
    # ---- opaque, order-shuffled ticket identifiers ------------------------------------
    n_nff = int(round(len(ticket_records) * cfg.nff_ticket_rate
                      / max(1 - cfg.nff_ticket_rate, 1e-6)))
    nff_rows = []
    marginal = topo.sort_values("extra_plant_loss_db", ascending=False).ont_id.tolist()
    for _ in range(n_nff):
        # NFF is not uniform over the fleet: it concentrates on marginal and chronic lines
        # and on customers who have recently been in trouble.
        if master.random() < 0.45 and len(marginal):
            ent = str(master.choice(marginal[: max(10, len(marginal) // 4)]))
        else:
            ent = str(master.choice(topo.ont_id.values))
        nff_rows.append(dict(ont_id=ent, reported_i=int(master.integers(0, n)),
                             repair_i=None, reported_symptom=str(master.choice(
                                 ["intermittent", "slow_service", "no_service"],
                                 p=[0.5, 0.35, 0.15])),
                             gt_fault_id=pd.NA, gt_fault_type="nff", gt_is_nff=True,
                             gt_misattributed=False))
    all_tk = ticket_records + nff_rows
    # Opaque alphanumeric references, shuffled so issue order carries nothing either.
    alphabet = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"
    order = master.permutation(len(all_tk))
    ids, used = {}, set()
    for idx in order:
        while True:
            tok = "".join(alphabet[k] for k in master.integers(0, len(alphabet), 8))
            if tok not in used:
                used.add(tok)
                break
        ids[int(idx)] = f"TKT-{tok}"

    tick = []
    for j, r in enumerate(all_tk):
        rep_i = r["repair_i"]
        if rep_i is None or master.random() < cfg.p_ticket_unresolved:
            resolved = pd.NaT
        else:
            jitter = int(master.normal(0, cfg.ticket_resolution_jitter_h * sph))
            k = int(np.clip(rep_i + jitter, r["reported_i"] + 1, n - 1))
            resolved = ts[k]
        tick.append(dict(
            ticket_id=ids[j], ont_id=r["ont_id"], reported_ts=ts[int(r["reported_i"])],
            resolved_ts=resolved, reported_symptom=r["reported_symptom"],
            gt_fault_id=r["gt_fault_id"], gt_fault_type=r["gt_fault_type"],
            gt_is_nff=r["gt_is_nff"], gt_misattributed=r["gt_misattributed"]))
    tickets = pd.DataFrame(tick).sort_values("reported_ts").reset_index(drop=True)
    tickets["resolved_ts"] = pd.to_datetime(tickets.resolved_ts, utc=True)

    # ---- fault registry -----------------------------------------------------------------
    faults_out["onset_ts"] = t0 + pd.to_timedelta(faults_out.onset_i * cfg.sample_minutes, unit="m")
    for col_i, col_ts in [("impact_i", "impact_ts"),
                          ("counterfactual_impact_i", "counterfactual_impact_ts"),
                          ("repair_i", "repair_ts"),
                          ("first_observable_i", "first_observable_ts"),
                          ("first_observable_practical_i", "first_observable_practical_ts")]:
        faults_out[col_ts] = [ts[int(v)] if pd.notna(v) else pd.NaT for v in faults_out[col_i]]
    tk_by_fault = (tickets.loc[~tickets.gt_is_nff].groupby("gt_fault_id").ticket_id
                   .apply(lambda s: sorted(s)[0]))
    faults_out["ticket_id"] = faults_out.gt_fault_id.map(tk_by_fault)

    # ---- per-entity intervals -----------------------------------------------------------
    interval_rows = []
    for ep in episodes:
        s = ep["start"]
        rep_i = ep.get("repair_i")
        end_i = min(rep_i if rep_i is not None else n - 1, n - 1, decom_i[ep["ont_id"]] - 1)
        end_i = max(end_i, s)
        imp_i = ep.get("impact_i")
        interval_rows.append(dict(
            fault_id=ep["gt_fault_id"], entity_id=ep["ont_id"],
            fault_family=FAULT_TYPES[ep["gt_fault_type"]]["family"], channel=ep["channel"],
            active_start_ts=ts[s], active_end_ts=ts[end_i],
            impact_ts=ts[imp_i] if imp_i is not None else pd.NaT,
            contribution_db=round(float(np.max(np.abs(ep["curve"]))) if len(ep["curve"]) else 0.0, 3)))
    fault_entity_intervals = pd.DataFrame(interval_rows)

    engineering_events = pd.DataFrame(eng_log) if eng_log else pd.DataFrame(
        columns=["entity_id", "ts_i", "event_type", "level_change_db", "detail"])
    if len(engineering_events):
        engineering_events["ts"] = [ts[int(k)] for k in engineering_events.ts_i]
        engineering_events = engineering_events.drop(columns=["ts_i"])
    benign_anomalies = pd.DataFrame(benign_log) if benign_log else pd.DataFrame(
        columns=["entity_id", "ts_i", "n_samples", "gt_benign_type"])
    if len(benign_anomalies):
        benign_anomalies["ts"] = [ts[int(min(k, n - 1))] for k in benign_anomalies.ts_i]
        benign_anomalies = benign_anomalies.drop(columns=["ts_i"])

    gaps_gt = (pd.concat(gap_reason_log, ignore_index=True) if gap_reason_log
               else pd.DataFrame({"entity_id": pd.Series(dtype="object"),
                                  "ts": pd.Series(dtype="datetime64[ns, UTC]"),
                                  "gt_gap_reason": pd.Series(dtype="object")}))

    # v4.1 (F3): alarms are OPERATIONAL, not ground truth. The OLT genuinely raises
    # these, so a detector may use them; they are incomplete and ambiguous in the way a
    # real alarm feed is, and carry no fault identifier.
    alarms = (pd.DataFrame(alarm_log).sort_values("raised_ts").reset_index(drop=True)
              if alarm_log else
              pd.DataFrame(columns=["entity_id", "alarm_type", "raised_ts", "cleared_ts",
                                    "duration_samples"]))

    registry_service = pd.DataFrame({
        "entity_id": topo.ont_id,
        "install_ts": [ts[int(k)] for k in topo.install_i],
        "decommission_ts": [ts[int(k)] if k < n else pd.NaT for k in topo.decommission_i]})

    storms_out = storms.copy()
    if len(storms_out):
        storms_out["start_ts"] = t0 + pd.to_timedelta(storms_out.start_day * 24, unit="h")
        storms_out["end_ts"] = storms_out.start_ts + pd.to_timedelta(storms_out.duration_h, unit="h")

    out_dir = Path(out_dir)
    tickets.to_csv(out_dir / "tickets.csv", index=False)
    fault_entity_intervals.to_csv(out_dir / "fault_entity_intervals.csv", index=False)
    engineering_events.to_csv(out_dir / "engineering_events.csv", index=False)
    benign_anomalies.to_csv(out_dir / "gt_benign_anomalies.csv", index=False)
    storms_out.to_csv(out_dir / "gt_fault_groups.csv", index=False)
    registry_service.to_csv(out_dir / "entity_service_windows.csv", index=False)
    gaps_gt.to_parquet(out_dir / "gt_collection_gaps.parquet", index=False)
    alarms.to_csv(out_dir / "alarms.csv", index=False)
    parameter_provenance(cfg).to_csv(out_dir / "parameter_provenance.csv", index=False)
    faults_out.drop(columns=["onset_i", "impact_i", "counterfactual_impact_i", "repair_i",
                             "first_observable_i", "first_observable_practical_i"]).to_csv(
        out_dir / "gt_fault_registry.csv", index=False)
    topo.to_csv(out_dir / "topology.csv", index=False)
    pd.Series(asdict(cfg)).to_json(out_dir / "generator_config.json", indent=2)
    return dict(topology=topo, faults=faults_out, tickets=tickets, storms=storms_out,
                fault_entity_intervals=fault_entity_intervals,
                engineering_events=engineering_events, benign_anomalies=benign_anomalies,
                gap_reasons=gaps_gt, alarms=alarms,
                panel_path=str(out_dir / cfg.out_path),
                gt_panel_path=str(out_dir / cfg.gt_out_path))


## 7. Provenance and named configurations

46 parameters with status and evidence. The `benchmark_tuning` rows are the caveat list for any claim made on this fixture.

In [ ]:
# ======================================================================================
# Provenance and named configurations
# ======================================================================================


def parameter_provenance(cfg=None) -> pd.DataFrame:
    """Status and evidence for every material parameter.

    Statuses: vendor_specification | engineering_estimate | uncalibrated_assumption |
    benchmark_tuning. `benchmark_tuning` means the value was set to hit a gate target on
    THIS fixture. Those values are properties of the benchmark and must never be quoted
    as operator facts.
    """
    cfg = cfg or TelecomSimulationSettings()
    enriched = cfg.config_role == "benchmark_enriched"
    rows = [
        ("olt_launch_dbm", cfg.olt_launch_dbm, "vendor_specification", "ITU-T G.984 class B+ OLT transmit power"),
        ("l1_loss_db", cfg.l1_loss_db, "vendor_specification", "Typical 1:4 splitter insertion loss"),
        ("l2 insertion loss (8/16/32)", "10.5 / 13.8 / 17.0", "vendor_specification",
         "Typical insertion loss by secondary split ratio"),
        ("fibre_loss_db_per_km / fibre_loss_us_db_per_km",
         f"{cfg.fibre_loss_db_per_km} / {cfg.fibre_loss_us_db_per_km}", "vendor_specification",
         "v4.1 (F2): G.652 attenuation at 1490 nm downstream and 1310 nm upstream; v4.0 charged the 1310 figure to its only direction"),
        ("olt_rx_sensitivity_dbm", cfg.olt_rx_sensitivity_dbm, "vendor_specification",
         "v4.1 (F1): class B+ OLT burst-mode receiver sensitivity"),
        ("olt_rx_noise_sd_db / olt_rx_cal_sd_db",
         f"{cfg.olt_rx_noise_sd_db} / {cfg.olt_rx_cal_sd_db}", "engineering_estimate",
         "v4.1 (F1): burst-mode measurement is less precise than a continuous receiver, and each port carries a calibration offset"),
        ("FAULT_TYPES us_ratio", "0.62 to 1.00", "engineering_estimate",
         "v4.1 (F1): share of downstream loss appearing upstream; bend-like mechanisms cost more at 1490 nm than 1310 nm"),
        ("target_margin_median_db / target_margin_log_sd",
         f"{cfg.target_margin_median_db} / {cfg.target_margin_log_sd}", "engineering_estimate",
         "v4.1 (F4): commissioning target. v4.0's 2.5 dB clamp BOUND on 51% of lines, putting a delta spike 0.5 dB above the impact threshold"),
        ("seasonal_amplitude_c / seasonal_peak_doy",
         f"{cfg.seasonal_amplitude_c} / {cfg.seasonal_peak_doy}", "engineering_estimate",
         "v4.1 (F6): UK ambient half-swing, anchored to day-of-year; v4.0 used window fraction so `start` and `days` had no effect on the weather"),
        ("ENCLOSURES self_heat_c", "20 to 29", "engineering_estimate",
         "v4.1 (F7): the OMCI temperature attribute is the transceiver's, not the room's; v4.0 reported ambient and the fleet sat near 15 C"),
        ("alarm_sd_ber_threshold / alarm_sf_ber_threshold",
         f"{cfg.alarm_sd_ber_threshold} / {cfg.alarm_sf_ber_threshold}", "benchmark_tuning",
         "v4.1 (F3): set so each alarm keeps a clean-fleet false-alarm rate; an alarm that never fires on a healthy line is an oracle"),
        ("alarm_sustain_minutes", cfg.alarm_sustain_minutes, "engineering_estimate",
         "v4.1 (F3): alarm soak timer, in TIME not polls -- expressed in samples it changes meaning with cadence"),
        ("alarm_dying_gasp_p", cfg.alarm_dying_gasp_p, "uncalibrated_assumption",
         "v4.1 (F3): share of premises outages reporting a dying gasp -- the only thing separating a power cut from a fibre break"),
        ("rx_sensitivity_dbm (per model)", "-26.5 to -28.5", "vendor_specification", "Class B+ ONT receiver sensitivity"),
        ("tx_nominal_dbm (per model)", "2.2 to 3.4", "vendor_specification", "Class B+ ONT launch power range"),
        ("quant_rx_db / quant_temp_c", f"{cfg.quant_rx_db} / {cfg.quant_temp_c}", "engineering_estimate",
         "OMCI/OLT reporting granularity"),
        ("impact_margin_db", cfg.impact_margin_db, "engineering_estimate", "Headroom at which service degrades"),
        ("cascade waterfall exponent (9.0)", 9.0, "engineering_estimate", "RS(255,239) t=8: t+1 decades per input decade"),
        ("l2_capacity_choices / weights", f"{cfg.l2_capacity_choices} / {cfg.l2_capacity_weights}",
         "engineering_estimate",
         "v4.1 (F5): mixed 1:4/1:8/1:16 secondary splitters, i.e. 1:16/1:32/1:64 total behind the 1:4 primary. "
         "Weights are PER INSTALLED SPLITTER; the subscriber-weighted mix is roughly the reverse. "
         "v4.0 offered up to 1:128, which class B+ optics cannot serve"),
        ("l2_takeup_beta_a/b", f"{cfg.l2_takeup_beta_a}/{cfg.l2_takeup_beta_b}", "uncalibrated_assumption",
         "v4 (E1): partial take-up per installed splitter; v3.1 filled every splitter to exactly 8"),
        ("feeder/distribution/drop length", "gamma composition", "engineering_estimate",
         "v4 (E3): route length shared down the tree; v3.1 drew it i.i.d. per ONT"),
        ("tx_device_sd_db / volt_device_sd_v", f"{cfg.tx_device_sd_db} / {cfg.volt_device_sd_v}",
         "engineering_estimate", "v4 (E4): device-to-device spread; v3.1 emitted both channels as fleet constants"),
        ("tx_temp_coeff_db_per_c", cfg.tx_temp_coeff_db_per_c, "engineering_estimate", "APC-controlled laser thermal drift"),
        ("volt_temp_coeff_v_per_c / volt_load_coeff_v",
         f"{cfg.volt_temp_coeff_v_per_c} / {cfg.volt_load_coeff_v}", "uncalibrated_assumption",
         "v4 (E4): rail behaviour under temperature and traffic load"),
        ("ber_implementation_penalty_sd_db", cfg.ber_implementation_penalty_sd_db, "engineering_estimate",
         "v4 (E6): receiver-to-receiver implementation penalty; v3.1 used one global curve"),
        ("fec_overdispersion_k / crc_overdispersion_k", f"{cfg.fec_overdispersion_k} / {cfg.crc_overdispersion_k}",
         "benchmark_tuning", "v4 (E6): set so margin is not recoverable from fec to better than ~1 dB"),
        ("fec_scale (per vendor)", "0.55 / 1.00 / 2.10", "uncalibrated_assumption",
         "v4 (E6): vendors do not scale FEC counters alike; magnitudes are a guess"),
        ("l1_loss_sd_db / l2_loss_sd_db", f"{cfg.l1_loss_sd_db} / {cfg.l2_loss_sd_db}", "uncalibrated_assumption",
         "v4 (E2): the two shared levels v3.1 omitted; amplitudes uncalibrated"),
        ("l2_micro_weather_sd_c", cfg.l2_micro_weather_sd_c, "uncalibrated_assumption", "v4 (E2): cabinet thermal environment"),
        ("olt_launch_sd_db / pon_feeder_sd_db", f"{cfg.olt_launch_sd_db} / {cfg.pon_feeder_sd_db}",
         "uncalibrated_assumption", "Shared drift amplitudes"),
        ("frailty_shape", cfg.frailty_shape, "benchmark_tuning",
         "v4 (E9): set to give per-entity fault counts variance/mean ~1.5; v3.1 measured 0.93 (exactly Poisson)"),
        ("beta_fibre_age / beta_outdoor / beta_distance_km / beta_excess_loss_db",
         f"{cfg.beta_fibre_age}/{cfg.beta_outdoor}/{cfg.beta_distance_km}/{cfg.beta_excess_loss_db}",
         "uncalibrated_assumption", "v4 (E9): deliberately weak susceptibility signal (OD4)"),
        ("p_recurrence / recurrence_window_days", f"{cfg.p_recurrence}/{cfg.recurrence_window_days}",
         "uncalibrated_assumption", "v4 (E9): repeat-offender behaviour; no operator data behind it"),
        ("storms_per_year / duration / footprint",
         f"{cfg.storms_per_year}/{cfg.storm_duration_h_mean}h/{cfg.storm_geo_clusters_mean}",
         "benchmark_tuning", "v4 (E11): set to populate group_id on >=10% of faults with >=3 multi-node groups (OD3)"),
        ("storm_hazard_multiplier", cfg.storm_hazard_multiplier, "benchmark_tuning", "As above (OD3)"),
        ("FAULT_TYPES magnitude lognormals", "mag_mu/mag_sigma per type", "benchmark_tuning",
         "v4 (E8): set so ~15% of optical faults sit below 0.5 dB; v3.1 used uniforms floored at 1.5 dB"),
        ("intermittent duty cycle", "on ~1.5 h / off ~7.5 h", "uncalibrated_assumption", "v4 (E8): no field basis"),
        ("poll_reliability_beta_a/b", f"{cfg.poll_reliability_beta_a}/{cfg.poll_reliability_beta_b}",
         "benchmark_tuning", "v4 (E10): set so per-entity missingness spans an order of magnitude; v3.1 spanned 1.26x"),
        ("benign_outage / holiday_absence rates",
         f"{cfg.benign_outage_rate_per_ont_year}/{cfg.holiday_absence_rate_per_ont_year}",
         "benchmark_tuning", "v4 (E10): set so dense gap bursts are not exclusive to faulty entities (OD5)"),
        ("p_sensor_glitch / p_stuck_start / p_transient_burst / reranging_rate",
         f"{cfg.p_sensor_glitch}/{cfg.p_stuck_start}/{cfg.p_transient_burst}/{cfg.reranging_rate_per_ont_year}",
         "benchmark_tuning",
         "v4 (E5): set so a 6*MAD rule has a non-zero clean-fleet false-alarm rate; on v3.1 it was exactly zero on bias (OD5)"),
        ("chronic_share / chronic_extra_loss_db", f"{cfg.chronic_share}/{cfg.chronic_extra_loss_db}",
         "uncalibrated_assumption", "v4: chronic cohort is now declared, not emergent"),
        ("crc_noisy_plant_share / multiplier", f"{cfg.crc_noisy_plant_share}/{cfg.crc_noisy_plant_multiplier}",
         "uncalibrated_assumption", "Entities with chronically noisy plant"),
        ("p_late_install / p_decommission", f"{cfg.p_late_install}/{cfg.p_decommission}",
         "uncalibrated_assumption", "v4 (E14): churn, so the cold-entity protocol has unseen entities"),
        ("nff_ticket_rate", cfg.nff_ticket_rate, "uncalibrated_assumption", "v4 (E7): raised from 0.08; still a guess"),
        ("p_ticket_given_impact", cfg.p_ticket_given_impact, "uncalibrated_assumption", "v4 (E7): lowered from 0.85"),
        ("p_duplicate_ticket / p_misattributed_ticket / p_ticket_unresolved",
         f"{cfg.p_duplicate_ticket}/{cfg.p_misattributed_ticket}/{cfg.p_ticket_unresolved}",
         "uncalibrated_assumption", "v4 (E7): ticket-channel noise; no operator data behind it"),
        ("mttr_median_h / mttr_sigma", f"{cfg.mttr_median_h}/{cfg.mttr_sigma}", "uncalibrated_assumption",
         "Engineering assumption; no operator data behind it"),
        ("report_delay heavy tail", f"{cfg.report_delay_heavy_tail_p} at {cfg.report_delay_heavy_mean_h}h",
         "uncalibrated_assumption", "v4 (E7): a minority of customers report days late"),
        ("repaired_convalescence_days", cfg.repaired_convalescence_days, "uncalibrated_assumption",
         "OPEN DECISION pending sign-off"),
        ("benign_plant_step_rate / improve_share",
         f"{cfg.benign_plant_step_rate}/{cfg.plant_step_improve_share}", "uncalibrated_assumption",
         "v4 (E12): rework now predominantly improves the line; v3.1 steps were zero-mean"),
        ("firmware_bias_step_ma / firmware_temp_step_c",
         f"{cfg.firmware_bias_step_ma}/{cfg.firmware_temp_step_c}", "uncalibrated_assumption",
         "v4 (E12): firmware rebases reported counters without moving the optics"),
        ("planned_maintenance_per_pon_per_year", cfg.planned_maintenance_per_pon_per_year,
         "uncalibrated_assumption", "v4 (E12): maintenance now suppresses collection"),
        ("poll_jitter_s / p_duplicate_poll", f"{cfg.poll_jitter_s}/{cfg.p_duplicate_poll}",
         "uncalibrated_assumption", "v4 (E14): OFF by default -- enabling requires a contract amendment (OD1)"),
        ("fault_rate_per_ont_year", cfg.fault_rate_per_ont_year,
         "benchmark_tuning" if enriched else "uncalibrated_assumption",
         "Enriched above field prevalence for per-family statistics; field regime is a guess"),
        ("shared_fault_rate_per_node_year", cfg.shared_fault_rate_per_node_year,
         "benchmark_tuning" if enriched else "uncalibrated_assumption", "As above, for shared-mode faults"),
    ]
    prov = pd.DataFrame(rows, columns=["parameter", "value", "status", "evidence"])
    prov.insert(0, "config_role", cfg.config_role)
    return prov


def build_reference_configs(base_seed: int = 20250717) -> dict:
    return {
        "benchmark_enriched": TelecomSimulationSettings(seed=base_seed, config_role="benchmark_enriched"),
        "field_prevalence": TelecomSimulationSettings(
            scenario="field_prevalence", seed=base_seed, config_role="field_prevalence",
            fault_rate_per_ont_year=0.35, shared_fault_rate_per_node_year=0.30,
            storms_per_year=10.0),
    }


def build_scenario_grid(base_seed: int = 20250717) -> dict:
    return {
        "default_reference": TelecomSimulationSettings(seed=base_seed, scenario="default_reference"),
        "field_prevalence": TelecomSimulationSettings(
            scenario="field_prevalence", seed=base_seed + 1, config_role="field_prevalence",
            fault_rate_per_ont_year=0.35, shared_fault_rate_per_node_year=0.30,
            storms_per_year=10.0),
        "tight_margins": TelecomSimulationSettings(
            scenario="tight_margins", seed=base_seed + 2, extra_connector_loss_mean_db=0.85, splices_per_km=0.8),
        "noisy_sensors": TelecomSimulationSettings(scenario="noisy_sensors", seed=base_seed + 3, sensor_noise_scale=2.0),
        "hourly_polling": TelecomSimulationSettings(scenario="hourly_polling", seed=base_seed + 4, sample_minutes=60),
        # v4: poor data quality is a first-class environment, not a nuisance
        "poor_collection": TelecomSimulationSettings(
            scenario="poor_collection", seed=base_seed + 5, poll_reliability_beta_a=25.0, poll_reliability_beta_b=2.2,
            collector_outage_per_collector_per_month=5.0, benign_outage_rate_per_ont_year=6.0),
        "storm_season": TelecomSimulationSettings(scenario="storm_season", seed=base_seed + 6, storms_per_year=45.0),
    }


## 8. Fixture validation gate

31 checks (23 from v4.0 plus F0–F9 and D1). Two v3.1 checks that passed **vacuously** on its own output — "no L2 splitter exceeds capacity" and "every peer group has at least four ONTs", both trivially true when every splitter held exactly eight — are replaced by checks with content. Each E-series check carries the v3.1 measurement it was written to move.

E9 and E11 measure statistical richness the `field_prevalence` regime is *designed* not to have, so they are enforced on the enriched regime and reported on field, marked `[reported, not gated on field regime]`.

**D1** runs three trivial per-entity threshold rules against the fixture and requires the union to neither solve it nor fail on it. It is the only check that reads the panel as a modeller would; it builds nothing reusable and exists so that a later parameter change cannot quietly restore the v3.1 difficulty floor.

In [ ]:
# Fixture-specific validation.
#
# Deliberately separate from `telemetry_contract.validate`, which holds only checks that
# are true of ANY telemetry source (plan section 6.4). Everything here needs knowledge the
# contract must never have: what the generator was configured with, what v3.1 measured,
# and what "too easy" means for this fixture.
#
#   contract (universal)              synth (fixture-specific, here)
#   schema, dtypes, nullability       long/wide row conservation against the panel
#   referential integrity             gaps reconstruct the CONFIGURED grid
#   temporal ordering                 registry size against configured n_onts
#   gaps vs telemetry itself          the C1-C11, E1-E14 and F1-F9 correction gates
#   catalogue covers metrics          naive difficulty-floor probe (D1, implemented)
#
# v4.1: the difficulty-floor probe was listed in this table from v4.0 and never written.
# The 0.94 -> 0.74 union-recall figure was the whole justification for v4 and nothing in
# the notebook held it in place, so a later parameter change could have made the fixture
# trivial again with all gates green. It is check D1 below, banded on BOTH sides.


from pathlib import Path

import numpy as np
import pandas as pd



def _as_utc_ns(x) -> int:
    """Integer nanoseconds UTC, whether or not the incoming Timestamp is tz-aware."""
    t = pd.Timestamp(x)
    t = t.tz_localize("UTC") if t.tzinfo is None else t.tz_convert("UTC")
    return int(t.value)


def validate_reference_data(panel_path, registry_path, topology_path, tickets_path, stage_dir=None):
    """Pass/fail table for the reference-data contract.

    Retains the v3.1 invariants and adds one check per v4 correction (E1-E14). Several
    v3.1 checks were VACUOUS on v3.1's own output -- "no L2 splitter exceeds capacity"
    and "every peer group has at least four ONTs" both passed on a fixture where every
    splitter held exactly eight -- so they are replaced by checks with content.
    """
    import pyarrow.parquet as parquet
    import json as _json_local

    stage_dir = Path(stage_dir) if stage_dir is not None else Path(panel_path).parent
    registry = pd.read_csv(registry_path, parse_dates=[
        "onset_ts", "impact_ts", "repair_ts", "counterfactual_impact_ts",
        "first_observable_ts", "first_observable_practical_ts"])
    topology = pd.read_csv(topology_path)
    tickets = pd.read_csv(tickets_path, parse_dates=["reported_ts", "resolved_ts"])
    cols = parquet.ParquetFile(panel_path).schema_arrow.names
    cfg_d = _json_local.loads((stage_dir / "generator_config.json").read_text())
    n = int(cfg_d["days"] * 24 * 60 / cfg_d["sample_minutes"])

    role = cfg_d.get("config_role", "unspecified")
    enriched = role == "benchmark_enriched"
    checks = []
    def rec(name, ok, details, enriched_only=False, underpowered=False):
        # v4.1: some checks measure a DISTRIBUTIONAL property that needs a minimum number
        # of events before it can be estimated at all. Below that the check is not
        # failing, it is unevaluable -- gating on it would fail a correct fixture for
        # being small. Reported with its measurement so a human can look, exactly as the
        # field-regime carve-out below already does.
        if underpowered:
            checks.append(dict(check_name=name + " [reported, insufficient events to gate]",
                               passed=True, details=str(details)))
            return
        # Two checks measure statistical richness (overdispersion, grouped causes) that
        # the field-prevalence regime is DESIGNED not to have: its per-family and
        # per-group counts are thin by construction. Enforcing them there would either
        # fail a correct fixture or push the field rates up to satisfy a gate written for
        # the development regime. They are enforced on enriched and REPORTED on field.
        if enriched_only and not enriched:
            checks.append(dict(check_name=name + " [reported, not gated on field regime]",
                               passed=True, details=str(details)))
            return
        checks.append(dict(check_name=name, passed=bool(ok), details=str(details)))

    # ---- retained structural invariants ------------------------------------------------
    gt_panel_path = stage_dir / "gt_panel.parquet"
    gt_cols = [c for c in cols if c.startswith("gt_")]
    rec("F0 The observable panel carries NO ground truth", not gt_cols,
        f"gt_ columns in panel={gt_cols} (v4.0 carried 14, 20.8% of compressed bytes)")
    gtc = list(parquet.ParquetFile(gt_panel_path).schema_arrow.names) if gt_panel_path.exists() else []
    n_panel = parquet.ParquetFile(panel_path).metadata.num_rows
    n_gt = parquet.ParquetFile(gt_panel_path).metadata.num_rows if gt_panel_path.exists() else -1
    rec("F0 Ground truth is a separate table, row-aligned with the panel",
        bool(gt_panel_path.exists() and n_gt == n_panel and len(gtc) >= 10),
        f"gt_panel rows={n_gt} vs panel rows={n_panel}, gt columns={len(gtc)}")
    need = {"timestamp_utc", "ont_id", "rx_power_dbm", "tx_power_dbm", "olt_rx_power_dbm",
            "temperature_c", "bias_current_ma", "voltage_v", "fec_count", "crc_errors",
            "reboot_count", "splitter_l2", "throughput_mbps", "firmware_version"}
    rec("Required observable telemetry is present", not (need - set(cols)),
        f"missing={sorted(need - set(cols))}")

    dated = registry.dropna(subset=["impact_ts"]).copy()
    dated["lead_h"] = (dated.impact_ts - dated.onset_ts).dt.total_seconds() / 3600
    rec("Every realised impact occurs after its true onset", dated.lead_h.ge(0).all(),
        f"violations={int((~dated.lead_h.ge(0)).sum())}")

    shared_reg = registry.loc[registry.scope.ne("ont")]
    rec("Shared faults reference a resolvable topology node",
        bool(shared_reg.empty or shared_reg.apply(
            lambda r: r.target in set(topology[SHARED_SCOPES[r.scope]]), axis=1).all()),
        f"n_shared={len(shared_reg)}, scopes={shared_reg.scope.value_counts().to_dict()}")

    both = registry.dropna(subset=["first_observable_ts", "first_observable_practical_ts"])
    rec("Practical observability anchor never precedes the sensor anchor",
        (both.first_observable_practical_ts >= both.first_observable_ts).all(),
        f"n_with_both={len(both)}")
    av = registry.loc[registry.averted.astype(bool)]
    rec("Averted faults have no realised impact and repair precedes the counterfactual crossing",
        bool(av.empty or (av.impact_ts.isna() & av.counterfactual_impact_ts.notna()
                          & av.repair_ts.notna() & (av.repair_ts <= av.counterfactual_impact_ts)).all()),
        f"n_averted={len(av)}")
    repaired = registry.dropna(subset=["repair_ts"])
    rec("Every repaired fault records a repair_source",
        repaired.repair_source.isin(["ticket", "proactive", "natural"]).all(),
        f"n_repaired={len(repaired)}, sources={repaired.repair_source.value_counts().to_dict()}")
    rx = pd.read_parquet(panel_path, columns=["rx_power_dbm"]).rx_power_dbm.dropna()
    rec("rx_power_dbm lies on the 0.1 dB reporting grid",
        (rx / 0.1 - (rx / 0.1).round()).abs().max() < 1e-6, "grid check")
    del rx
    pairs = topology.groupby("device_model").vendor.nunique()
    rec("C1 Each device model maps to exactly one vendor", bool((pairs == 1).all()),
        f"models={len(pairs)}")
    rec("C6 Fibre age and ONT age are separate attributes",
        bool((topology.ont_age_yr <= topology.fibre_age_yr + 1e-9).all()
             and topology.fibre_age_yr.corr(topology.ont_age_yr) < 0.99),
        f"corr={topology.fibre_age_yr.corr(topology.ont_age_yr):.3f}")

    # ---- E1 fan-out has real variance ---------------------------------------------------
    fan = topology.groupby("splitter_l2").ont_id.nunique()
    caps = topology.groupby("splitter_l2").l2_splitter_capacity.first()
    rec("E1 Splitter fan-out varies and take-up is partial",
        bool(fan.std() > 1.5 and fan.nunique() >= 4 and (fan <= caps).all()),
        f"n_splitters={len(fan)}, fill min/med/max={fan.min()}/{int(fan.median())}/{fan.max()}, "
        f"sd={fan.std():.2f}, distinct sizes={fan.nunique()} (v3.1 measured sd=0.00, all 8)")

    # ---- E3 route length is coherent within a splitter ----------------------------------
    within = topology.groupby("splitter_l2").distance_m.std().median()
    icc = 1 - (within ** 2) / max(topology.distance_m.var(), 1e-9)
    rec("E3 Route length is shared down the tree",
        bool(icc > 0.75),
        f"within-L2 ICC={icc:.3f}, median within-splitter sd={within:.0f} m "
        f"(v3.1 measured ICC 0.144, spread 7,080 m)")

    # ---- E13 identifiers carry no topology ----------------------------------------------
    idx = topology.ont_id.str[-5:].astype(int)
    olt_i = topology.olt_id.str[-2:].astype(int)
    blocks = topology.groupby("splitter_l2").apply(
        lambda g: (g.ont_id.str[-5:].astype(int).max() - g.ont_id.str[-5:].astype(int).min()) == len(g) - 1)
    tol = max(0.15, 3.0 / np.sqrt(max(len(topology), 4)))
    rec("E13 Entity identifiers are not ordered by topology",
        bool(abs(np.corrcoef(idx, olt_i)[0, 1]) < tol and blocks.mean() < 0.2),
        f"corr(id, OLT)={np.corrcoef(idx, olt_i)[0, 1]:+.3f}, contiguous splitter blocks="
        f"{blocks.mean():.2f} (v3.1 measured +0.866 and 1.00)")

    # ---- E4 no observable channel is a fleet constant -------------------------------
    # v4.1: `gt_state` lives in the ground-truth panel now. The two files are written
    # from the same loop in the same row order, so a positional attach is exact.
    healthy_mask = (pd.read_parquet(gt_panel_path, columns=["gt_state"])
                    .gt_state.astype(str).eq("healthy").to_numpy())
    het = {}
    for m in ["rx_power_dbm", "tx_power_dbm", "olt_rx_power_dbm", "bias_current_ma",
              "voltage_v", "temperature_c"]:
        s = pd.read_parquet(panel_path, columns=["ont_id", m])
        s = s.loc[healthy_mask].dropna()
        het[m] = float(s.groupby("ont_id")[m].median().std())
        del s
    rec("E4 Every observable channel has device-to-device heterogeneity",
        bool(het["tx_power_dbm"] > 0.25 and het["voltage_v"] > 0.010),
        f"between-entity sd of healthy medians: {({k: round(v, 4) for k, v in het.items()})} "
        f"(v3.1 measured tx 0.0497 dB and voltage 0.0000 V)")

    # ---- E8 weak faults exist -----------------------------------------------------------
    optical = registry.loc[registry.family.ne("laser")]
    weak = float((optical.magnitude_db < 0.5).mean()) if len(optical) else 0.0
    invisible = float(registry.first_observable_ts.isna().mean())
    rec("E8 A material share of faults is weak, and some never become observable",
        bool(weak > 0.06 and invisible > 0.02),
        f"optical faults below 0.5 dB={weak:.1%}, faults with no sensor anchor={invisible:.1%} "
        f"(v3.1 measured 0.0% and 0.3%)")

    # ---- E9 overdispersion and susceptibility -------------------------------------------
    fei_p = stage_dir / "fault_entity_intervals.csv"
    if fei_p.exists():
        fei = pd.read_csv(fei_p)
        cnt = topology.ont_id.map(fei.groupby("entity_id").fault_id.nunique()).fillna(0)
        # A point threshold on variance/mean is the wrong instrument: at 400 entities
        # with roughly one fault each, its sampling spread is wide enough that a correct
        # fixture fails by chance -- `tight_margins` measured 1.104 against a 1.15
        # threshold while running the identical arrival process. Test it instead.
        # Under Poisson, T = sum((x - xbar)^2) / xbar ~ chi2(n - 1).
        from scipy import stats as _st
        vm = float(cnt.var() / max(cnt.mean(), 1e-9))
        T = float(((cnt - cnt.mean()) ** 2).sum() / max(cnt.mean(), 1e-9))
        dof = int(len(cnt) - 1)
        p_over = float(_st.chi2.sf(T, dof))
        # Deliberately lenient at alpha=0.10: at this fleet size the test guards against
        # Poisson-EXACTNESS (v3.1 measured 0.925, p=0.86) rather than measuring the
        # frailty precisely. Tightening it needs a larger fleet, not a smaller alpha.
        rec("E9 Per-entity fault counts are overdispersed relative to Poisson",
            bool(p_over < 0.10),
            f"variance/mean={vm:.3f}, dispersion test chi2({dof})={T:.0f}, p={p_over:.3f} "
            f"(v3.1 measured 0.925, p=0.86 -- indistinguishable from Poisson)",
            enriched_only=True, underpowered=bool(cnt.sum() < 120))
    else:
        rec("E9 Per-entity fault counts are overdispersed relative to Poisson", False, "intervals missing")

    # ---- E11 shared faults at several levels, and grouped causes exist -------------------
    lvl = registry.loc[registry.scope.ne("ont"), "scope"].nunique()
    grouped = float(registry.group_id.notna().mean())
    multi = registry.dropna(subset=["group_id"]).groupby("group_id").target.nunique()
    # v4.1: the number of DISTINCT shared scopes realised is a combinatorial function of
    # fleet size -- a 120-entity run installs ~23 L2 splitters and cannot be expected to
    # draw an OLT-scope arrival. Required scopes scale with the node count.
    n_nodes = int(sum(topology[c].nunique() for c in SHARED_SCOPES.values()))
    need_lvl = 3 if n_nodes >= 120 else 1
    rec("E11 Shared faults attach at several topology levels and groups are populated",
        bool(lvl >= need_lvl and grouped >= 0.08 and int((multi >= 3).sum()) >= 3),
        f"shared scopes={lvl} (required {need_lvl} at {n_nodes} shared nodes), "
        f"group_id populated={grouped:.1%}, groups spanning >=3 nodes="
        f"{int((multi >= 3).sum())} (v3.1: one level, 0% grouped)", enriched_only=True,
        underpowered=bool(len(registry) < 120))

    # ---- E7 ticket identifiers carry no fault information --------------------------------
    real = tickets.loc[~tickets.gt_is_nff.astype(bool)]
    leak = 0
    if len(real):
        leak = int(sum(str(r.gt_fault_id).replace("F-", "") in str(r.ticket_id)
                       for r in real.itertuples()))
    nff_pref = tickets.loc[tickets.gt_is_nff.astype(bool), "ticket_id"].astype(str)
    nff_sep = bool(len(nff_pref) and not nff_pref.str.contains("NFF").any())
    res_var = real.groupby("gt_fault_id").resolved_ts.nunique()
    n_multi = int((real.groupby("gt_fault_id").size() > 1).sum())
    rec("E7 Ticket identifiers carry no fault information",
        bool(leak == 0 and nff_sep and (n_multi == 0 or (res_var > 1).any())),
        f"ids embedding a fault id={leak}, NFF distinguishable by id={not nff_sep}, "
        f"faults whose tickets resolve at different times={int((res_var > 1).sum())} "
        f"(v3.1: all ids embedded the fault id; NFF carried an NFF prefix; 260/260 shared one resolution)")

    # ---- E10 missingness is heterogeneous and not fault-exclusive -------------------------
    gp = stage_dir / "gt_collection_gaps.parquet"
    if gp.exists():
        gg = pd.read_parquet(gp)
        sw = pd.read_csv(stage_dir / "entity_service_windows.csv", parse_dates=["install_ts", "decommission_ts"])
        per = (gg.groupby("entity_id").size().reindex(topology.ont_id).fillna(0) / n)
        spread = float(per.quantile(0.95) / max(per.quantile(0.05), 1e-9))
        faulty = set(pd.read_csv(fei_p).entity_id) if fei_p.exists() else set()
        clean_ents = [e for e in topology.ont_id if e not in faulty]
        # A dense gap must be defined relative to the CADENCE, not to a clock hour.
        # The first form of this check counted ">=3 of 4 polls lost in an hour", which is
        # unsatisfiable at hourly cadence and failed `hourly_polling` for a property the
        # fixture does not have -- a gate bug, not a fixture defect.
        # Compare Timedeltas, not raw int64. A tz-aware timestamp[us] column casts to
        # MICROSECONDS, so an assumed-nanosecond step silently matches nothing and every
        # entity reports a maximum gap run of 1.
        step = pd.Timedelta(minutes=int(cfg_d["sample_minutes"]))
        gg2 = gg.copy()
        gg2["ts"] = pd.to_datetime(gg2.ts, utc=True)
        burst_ents = set()
        for ent, grp in gg2.groupby("entity_id"):
            a = grp.ts.sort_values()
            if len(a) < 3:
                continue
            consecutive = a.diff() == step
            run = 0
            for flag in consecutive.to_numpy():
                run = run + 1 if flag else 0
                if run >= 2:            # two consecutive steps == three polls in a row
                    burst_ents.add(ent)
                    break
        clean_bursts = len(burst_ents & set(clean_ents))
        rec("E10 Missingness varies by entity and dense gaps are not fault-exclusive",
            bool(spread > 2.0 and clean_bursts >= max(1, int(0.05 * len(clean_ents)))),
            f"per-entity p95/p05 rate ratio={spread:.2f}, clean entities losing >=3 consecutive polls="
            f"{clean_bursts}/{len(clean_ents)} "
            f"(v3.1 measured 1.26 and LOS on faulty entities only)")
    else:
        rec("E10 Missingness varies by entity and dense gaps are not fault-exclusive", False, "gaps missing")

    # ---- E5 the healthy class is not perfectly clean ---------------------------------------
    ben_p = stage_dir / "gt_benign_anomalies.csv"
    ben = pd.read_csv(ben_p) if ben_p.exists() else pd.DataFrame()
    rec("E5 A benign anomaly layer exists and is labelled",
        bool(len(ben) > 0 and ben.gt_benign_type.nunique() >= 3),
        f"benign anomalies={len(ben)}, types={ben.gt_benign_type.value_counts().to_dict() if len(ben) else {}}")

    # ---- E6 the cascade is not a noiseless copy of margin ------------------------------------
    # v4.1: `gt_margin_db` is gone. It was never ground truth -- it equalled
    # `rx_power_dbm - rx_sensitivity_dbm` to a measured 0.025 dB mean absolute error,
    # both of them columns the panel handed over. Margin is reconstructed here from the
    # genuinely latent pair plus the receiver sensitivity from topology.
    gtm = pd.read_parquet(gt_panel_path,
                          columns=["ont_id", "gt_rx_healthy_dbm", "gt_optical_degradation_db"])
    sens_map = topology.set_index("ont_id").rx_sensitivity_dbm
    true_margin = (gtm.gt_rx_healthy_dbm + gtm.gt_optical_degradation_db
                   - gtm.ont_id.astype(str).map(sens_map)).to_numpy()
    del gtm
    fc = pd.read_parquet(panel_path, columns=["fec_count"])
    fc["margin"] = true_margin
    fc = fc.dropna()
    fc = fc.sample(min(300_000, len(fc)), random_state=0)
    lf = np.log10(fc.fec_count + 1)
    fit = np.polyfit(fc.margin, lf, 1)
    resid_db = float((lf - np.polyval(fit, fc.margin)).std() / max(abs(fit[0]), 1e-9))
    rec("E6 Optical margin is not recoverable from FEC counts to better than ~1 dB",
        bool(resid_db > 0.8),
        f"margin recoverable from log10(fec) to +/-{resid_db:.2f} dB (1 sd); v3.1 measured +/-0.33 dB")
    del fc, true_margin

    # ---- E12/E14 benign change and churn ------------------------------------------------------
    ep = stage_dir / "engineering_events.csv"
    eng = pd.read_csv(ep) if ep.exists() else pd.DataFrame()
    rec("E12 Benign operational change is logged and varied",
        bool(len(eng) and eng.event_type.nunique() >= 3),
        f"events={len(eng)}, types={eng.event_type.value_counts().to_dict() if len(eng) else {}}")
    sw_p = stage_dir / "entity_service_windows.csv"
    if sw_p.exists():
        sw = pd.read_csv(sw_p, parse_dates=["install_ts", "decommission_ts"])
        # v4.1: expressed as RATES against the configured probabilities. v4.0 asked for
        # three of each, which is a threshold tuned to 400 entities: a correct 120-entity
        # run measured one decommission against an expectation of 4.2 and failed a gate
        # for a property the fixture has.
        late = int((sw.install_ts > sw.install_ts.min()).sum())
        gone = int(sw.decommission_ts.notna().sum())
        exp_late = float(cfg_d["p_late_install"]) * len(sw)
        exp_gone = float(cfg_d["p_decommission"]) * len(sw)
        from scipy import stats as _st2
        p_late = float(_st2.poisson.cdf(late, max(exp_late, 1e-9)))
        p_gone = float(_st2.poisson.cdf(gone, max(exp_gone, 1e-9)))
        rec("E14 Entities are provisioned and decommissioned inside the window",
            bool(late >= 1 and gone >= 1 and p_late > 0.005 and p_gone > 0.005),
            f"late installs={late} (expected {exp_late:.1f}, lower-tail p={p_late:.3f}), "
            f"decommissions={gone} (expected {exp_gone:.1f}, p={p_gone:.3f})")
    else:
        rec("E14 Entities are provisioned and decommissioned inside the window", False, "service windows missing")

    # ---- F4 the turn-up margin has a distribution, not a clamp spike --------------------
    span_ds = topology.span_loss_ds_db if "span_loss_ds_db" in topology.columns else None
    if span_ds is not None:
        tm = float(cfg_d["olt_launch_dbm"]) - span_ds - topology.rx_sensitivity_dbm
        modal = float(tm.round(1).value_counts().max() / len(tm))
        rec("F4 Commissioning margin is a distribution, not a pile-up on the floor",
            bool(tm.std() > 1.0 and modal < 0.15
                 and tm.median() > float(cfg_d["impact_margin_db"]) + 1.5),
            f"median={tm.median():.2f} dB, sd={tm.std():.2f}, largest 0.1 dB bucket={modal:.1%} "
            f"(v4.0 measured median 2.50, sd 1.37, with the clamp binding on 51%)")
        rec("F5 Span loss sits inside the class B+ budget for most of the fleet",
            bool((span_ds <= 28.0).mean() > 0.90),
            f"median span loss={span_ds.median():.1f} dB, share within 28 dB="
            f"{(span_ds <= 28.0).mean():.1%} (v4.0: median 28.0 dB, 56% of subscribers on 1:128)")

    # ---- F1 the upstream channel exists and is directionally informative ---------------
    gtm2 = pd.read_parquet(gt_panel_path, columns=["gt_state", "gt_fault_type"])
    ur = pd.read_parquet(panel_path, columns=["ont_id", "rx_power_dbm", "olt_rx_power_dbm"])
    ur["state"] = gtm2.gt_state.astype(str).to_numpy()
    ur["ftype"] = gtm2.gt_fault_type.astype(str).to_numpy()
    del gtm2
    uh = ur.loc[ur.state.eq("healthy")].dropna(subset=["rx_power_dbm", "olt_rx_power_dbm"])
    within = (uh.groupby("ont_id")
                .apply(lambda s: s.rx_power_dbm.corr(s.olt_rx_power_dbm) if len(s) > 500 else np.nan)
                .dropna())
    rec("F1 Upstream is an independent observation, not a copy of downstream",
        bool(len(within) and abs(float(within.median())) < 0.45),
        f"within-entity healthy corr(rx, olt_rx): median={float(within.median()):.3f} "
        f"over {len(within)} entities")
    laser = ur.loc[ur.ftype.eq("ont_hardware_failure") & ur.state.ne("healthy")].dropna(
        subset=["rx_power_dbm", "olt_rx_power_dbm"])
    d_rx = (laser.rx_power_dbm.mean() - uh.rx_power_dbm.mean()) if len(laser) > 100 else 0.0
    d_us = (laser.olt_rx_power_dbm.mean() - uh.olt_rx_power_dbm.mean()) if len(laser) > 100 else 0.0
    rec("F1 An ONT laser fault presents upstream, not downstream",
        bool(len(laser) > 100 and d_us < d_rx - 1.0),
        f"laser-fault shift vs healthy fleet: downstream {d_rx:+.2f} dB, upstream {d_us:+.2f} dB "
        f"(v4.0 had no upstream channel)")
    del ur, uh

    # ---- F3 alarms are observable, and are not a fault oracle --------------------------
    al_p = stage_dir / "alarms.csv"
    if al_p.exists() and fei_p.exists():
        al = pd.read_csv(al_p, parse_dates=["raised_ts"])
        fei_a = pd.read_csv(fei_p, parse_dates=["active_start_ts", "active_end_ts"])
        # Measured TIME-RESOLVED, not per entity. Under an enriched fault rate most of
        # the fleet is faulty at some point -- 328 of 400 on the reference run -- so
        # "share of alarms on never-faulty entities" is bounded by the size of the clean
        # cohort rather than by the alarm's informativeness. What matters is whether an
        # alarm firing NOW means a fault is active NOW.
        def _ns_col(s):
            return np.asarray(pd.to_datetime(s, utc=True).dt.tz_localize(None),
                              dtype="datetime64[ns]").astype("int64")
        iv = {}
        if len(fei_a):
            fei_a["ns_start"] = _ns_col(fei_a.active_start_ts)
            fei_a["ns_end"] = _ns_col(fei_a.active_end_ts)
            iv = {str(e): (g.ns_start.to_numpy(), g.ns_end.to_numpy())
                  for e, g in fei_a.groupby("entity_id")}
        if len(al):
            al["ns_raised"] = _ns_col(al.raised_ts)
            def _during(ent, t):
                v = iv.get(str(ent))
                return False if v is None else bool(((v[0] <= t) & (t <= v[1])).any())
            al["during"] = [_during(e, t) for e, t in zip(al.entity_id, al.ns_raised)]
            outside = (1.0 - al.groupby("alarm_type").during.mean())
            worst = float(outside.min())
            overall = float(1.0 - al.during.mean())
        else:
            outside, worst, overall = pd.Series(dtype=float), 0.0, 0.0
        # Two thresholds doing different jobs. `overall` asks whether the channel as a
        # whole hands over the labels; `worst` asks whether any single type is a perfect
        # oracle. Signal-fail is LEGITIMATELY the most precise alarm here and gets more
        # precise as the soak lengthens relative to cadence -- 0.336 outside an active
        # fault at 15-minute polling against 0.066 at 60-minute, on the same physics --
        # so a common floor across types would fail `hourly_polling` for a property the
        # fixture has rather than one it lacks.
        rec("F3 An observable alarm channel exists and no alarm type is fault-exclusive",
            bool(len(al) > 0 and al.alarm_type.nunique() >= 3
                 and overall > 0.20 and worst > 0.03),
            f"alarms={len(al)}, types={al.alarm_type.value_counts().to_dict() if len(al) else {}}, "
            f"share raised while NO fault was active: overall={overall:.3f}, "
            f"by type={outside.round(3).to_dict()} "
            f"(v4.0 had no alarm channel at all; LOS was ground truth and 97% fault-exclusive)",
            underpowered=bool(len(al) < 2000))
    else:
        rec("F3 An observable alarm channel exists and no alarm type is fault-exclusive",
            False, "alarms.csv or fault_entity_intervals.csv missing")

    # ---- F8 no sample is labelled repaired or healthy while a fault is active ----------
    st = pd.read_parquet(gt_panel_path, columns=["gt_state", "gt_active_fault_count"])
    contra = int(((st.gt_active_fault_count > 0)
                  & st.gt_state.astype(str).isin(["repaired", "healthy"])).sum())
    rec("F8 Per-sample state is consistent with the active-fault count",
        contra == 0,
        f"contradictory rows={contra} (v4.0 measured 2,654, about 15% of the repaired class)")
    del st

    # ---- D1 the difficulty floor, measured rather than asserted -------------------------
    # Three trivial per-entity rules, exactly the union that reached 0.94 pre-impact
    # recall with zero false positives on v3.1 and 0.74 on v4.0. Banded on both sides:
    # too high means the fixture has become solvable by a threshold again, too low means
    # it has drifted into a regime where nothing simple works and the benchmark stops
    # discriminating between methods.
    probe_cols = ["ont_id", "timestamp_utc", "rx_power_dbm", "bias_current_ma", "tx_power_dbm"]
    s = pd.read_parquet(panel_path, columns=probe_cols)
    alert = np.zeros(len(s), dtype=bool)
    for c in ["rx_power_dbm", "bias_current_ma", "tx_power_dbm"]:
        med = s.groupby("ont_id")[c].median()
        dev = (s[c] - s.ont_id.map(med)).abs()
        mad = dev.groupby(s.ont_id).median()
        thr = 6.0 * 1.4826 * s.ont_id.map(mad)
        alert |= (dev > thr).fillna(False).to_numpy()
    s["alert"] = alert
    fired = s.loc[alert, ["ont_id", "timestamp_utc"]].copy()
    # Compare in integer nanoseconds: the panel is tz-aware and the sidecars are parsed
    # per call site, so a Timestamp-to-Timestamp comparison is not safe here.
    # The panel column is timestamp[us]: `.astype("int64")` on it yields MICROSECONDS
    # while `pd.Timestamp.value` yields nanoseconds, so the two never compare equal and
    # every window silently matches nothing. The same trap is documented on E10 below.
    fired["ns"] = np.asarray(
        pd.to_datetime(fired.timestamp_utc, utc=True).dt.tz_localize(None),
        dtype="datetime64[ns]").astype("int64")
    fired_by_ent = {str(e): np.sort(g.ns.to_numpy())
                    for e, g in fired.groupby("ont_id", observed=True)}
    del s, fired

    if fei_p.exists():
        fei_d = pd.read_csv(fei_p, parse_dates=["active_start_ts", "active_end_ts", "impact_ts"])

        def _fires(entity, t0_ts, t1_ts):
            arr = fired_by_ent.get(str(entity))
            if arr is None:
                return False
            lo = np.searchsorted(arr, _as_utc_ns(t0_ts))
            hi = np.searchsorted(arr, _as_utc_ns(t1_ts))
            return bool(hi > lo)

        # The GATED statistic is detection anywhere inside the active interval. It is
        # scored over every episode rather than the minority with a non-degenerate
        # pre-impact window -- at 400 entities the pre-impact subset is small enough that
        # its sampling spread swamps the effect the band is meant to catch.
        det = float(np.mean([_fires(r.entity_id, r.active_start_ts, r.active_end_ts)
                             for r in fei_d.itertuples()])) if len(fei_d) else 0.0
        pre_set = fei_d.dropna(subset=["impact_ts"])
        pre_set = pre_set.loc[pre_set.impact_ts > pre_set.active_start_ts]
        pre = float(np.mean([_fires(r.entity_id, r.active_start_ts, r.impact_ts)
                             for r in pre_set.itertuples()])) if len(pre_set) else float("nan")
        faulty_e = set(fei_d.entity_id)
        clean_e = [e for e in topology.ont_id if e not in faulty_e]
        fa = (sum(1 for e in clean_e if str(e) in fired_by_ent) / max(len(clean_e), 1)) if clean_e else 0.0
        rec("D1 A union of trivial threshold rules neither solves nor fails the fixture",
            bool(0.30 <= det <= 0.85 and fa > 0.02),
            # 60 episodes puts the standard error on `det` near 0.06; below that the band
            # is wider than the thing it is trying to measure.
            f"6xMAD union (rx | bias | tx): in-episode detection={det:.2f} over {len(fei_d)} "
            f"episodes, pre-impact recall={pre:.2f} over {len(pre_set)} with a non-degenerate "
            f"window, clean-fleet alert rate={fa:.2f} "
            f"(v3.1 measured 0.94 pre-impact recall at 0.00 false alerts; v4.0 measured 0.74)",
            underpowered=bool(len(fei_d) < 60))
    else:
        rec("D1 A union of trivial threshold rules neither solves nor fails the fixture",
            False, "intervals missing")

    return pd.DataFrame(checks)


## 9. Versioning, manifests and reproducibility

Three independently versioned things — generator semantics, native format, and the individual dataset — plus content digests, an environment capture, an immutable publication step and a verification routine that regenerates a dataset and compares it table by table.

Digests hash the **data**, not the file bytes: Parquet embeds writer metadata and CSV float formatting is not stable across platforms, so byte comparison would answer the wrong question.

In [ ]:
# Versioned persistence, manifests and reproducibility verification.
#
# Scope discipline: this module makes generator runs **reproducible, versioned and
# discoverable**. It does not map anything to a canonical schema, and it must not. That
# mapping comes later, as a separate adapter layer, once several sources exist to
# generalise over. What it does instead is write down everything an adapter author will
# need — a native-format description, a config, a seed, an environment capture and content
# digests — so that mapping is a small job when it is time to do it.
#
# Three things are versioned independently and all three are recorded:
#
#   generator version   the simulator's semantics. Changing it changes the data.
#   native format       the on-disk shape. Adapters bind to THIS, not to a Python version.
#   dataset version     a specific run: generator version + config + seed + environment.
#
# Reproducibility is *verified*, not asserted. `verify_reproducibility` regenerates from
# the stored config and compares content digests row for row.


import hashlib
import json
import platform
import shutil
import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd

NATIVE_FORMAT_VERSION = "telemetry-synth native v5"

# Written by the validation or publication step, not by the generator. They travel with a
# published dataset but are not part of it, and must not be expected to regenerate.
DERIVED_FILES = {"fixture_validation_report.csv", "data_validation_report.csv",
                 "stage_01_state.json", "dataset_manifest.json"}

# The native format declaration. An adapter author reads THIS, not the generator source.
NATIVE_TABLES = {
    "reference_dataset.parquet": dict(
        role="telemetry_panel", grain="one row per entity per poll",
        note="wide panel of OBSERVABLES ONLY. Downstream receive, upstream receive as "
             "measured at the OLT, transmit, temperature, bias, voltage, the error "
             "cascade, uptime and throughput, plus the routing keys an OLT record "
             "carries. It contains no ground truth of any kind, and the generator "
             "raises if any column reaches it with the gt_ prefix."),
    "gt_panel.parquet": dict(
        role="ground_truth_panel", grain="one row per entity per poll",
        note="the latent state behind each row of the panel: the counterfactual healthy "
             "receive level, true degradation depth in each direction, laser progression, "
             "and the per-sample label block. Written from the same loop in the same row "
             "order as the panel, so it attaches positionally OR joins on "
             "(ont_id, timestamp_utc). EVALUATION ONLY."),
    "topology.csv": dict(
        role="entity_attributes", grain="one row per ONT",
        note="static device, plant, geography and commercial attributes, plus the "
             "OLT/PON/L1/L2 parentage. gt_frailty, gt_chronic and gt_noisy_plant are "
             "ground truth."),
    "entity_service_windows.csv": dict(
        role="entity_lifecycle", grain="one row per ONT",
        note="install_ts and decommission_ts. Entities are provisioned and decommissioned "
             "inside the observation window, so any grid reconstruction must intersect "
             "with these bounds."),
    "gt_fault_registry.csv": dict(
        role="ground_truth_faults", grain="one row per fault",
        note="onset, observability anchors, impact, counterfactual impact, repair, "
             "group membership and recurrence parentage."),
    "fault_entity_intervals.csv": dict(
        role="ground_truth_faults", grain="one row per fault per affected entity",
        note="a shared fault appears once per entity beneath the failed node."),
    "gt_fault_groups.csv": dict(
        role="ground_truth_faults", grain="one row per grouped cause",
        note="storm windows and their geographic footprint."),
    "alarms.csv": dict(
        role="operational_labels", grain="one row per alarm",
        note="an OPERATIONAL channel the OLT genuinely raises: loss of signal, dying "
             "gasp, signal-degrade and signal-fail threshold crossings. Legitimately "
             "available to a detector. Deliberately ambiguous -- a customer pulling the "
             "plug and a fibre break both raise LOS, and only the dying gasp separates "
             "them -- and every type fires on never-faulty lines."),
    "tickets.csv": dict(
        role="operational_labels", grain="one row per ticket",
        note="an OPERATIONAL channel, not ground truth: incomplete, delayed, duplicated, "
             "sometimes raised against the wrong line, and sometimes no-fault-found. "
             "Identifiers are opaque and carry no fault information."),
    "engineering_events.csv": dict(
        role="operational_context", grain="one row per planned change",
        note="firmware upgrades, plant rework, provisioning changes, planned maintenance. "
             "Known to the operator, so legitimately available to a detector."),
    "gt_benign_anomalies.csv": dict(
        role="ground_truth_negatives", grain="one row per benign excursion",
        note="labelled NON-fault excursions: sensor glitches, stuck values, transient "
             "bursts, re-ranging, CPE power cycles. Lets a false alert be attributed to a "
             "cause rather than merely counted."),
    "gt_collection_gaps.parquet": dict(
        role="ground_truth_missingness", grain="one row per missing poll",
        note="cause of every absent observation. The cause is ground truth; the absence "
             "itself is observable."),
    "parameter_provenance.csv": dict(
        role="provenance", grain="one row per material parameter",
        note="status is one of vendor_specification, engineering_estimate, "
             "uncalibrated_assumption, benchmark_tuning. Benchmark-tuned values are "
             "properties of this fixture and must never be quoted as operator facts."),
    "generator_config.json": dict(
        role="configuration", grain="single object",
        note="the complete settings object. Together with the generator version and the "
             "environment capture, this is sufficient to regenerate the dataset."),
}


# ======================================================================================
# Digests and environment
# ======================================================================================


def capture_environment() -> dict:
    """Library versions matter: the generator's output depends on NumPy's RNG stream."""
    import pyarrow
    import scipy
    return dict(
        python=sys.version.split()[0],
        platform=platform.platform(),
        numpy=np.__version__, pandas=pd.__version__,
        pyarrow=pyarrow.__version__, scipy=scipy.__version__,
    )


def _row_hashes(path: Path, batch_rows: int = 200_000):
    """Order-independent per-row hashes, streamed so a multi-million-row panel never
    has to be held in memory at once."""
    import pyarrow.parquet as pq
    if path.suffix == ".parquet":
        pf = pq.ParquetFile(path)
        cols = sorted(pf.schema_arrow.names)
        parts, n = [], 0
        for batch in pf.iter_batches(batch_size=batch_rows):
            df = batch.to_pandas().reindex(columns=cols)
            parts.append(pd.util.hash_pandas_object(df, index=False).to_numpy())
            n += len(df)
            del df
        h = np.concatenate(parts) if parts else np.zeros(0, dtype="uint64")
        return h, n, cols
    if path.suffix == ".csv":
        df = pd.read_csv(path)
        cols = sorted(df.columns)
        df = df.reindex(columns=cols)
        return pd.util.hash_pandas_object(df, index=False).to_numpy(), len(df), cols
    return None, None, None


def content_digest(path: Path) -> tuple:
    """(content_sha256, n_rows). Hashes the DATA, not the file bytes.

    File bytes are not a fair reproducibility test: Parquet embeds writer metadata, and
    CSV float formatting is not stable across platforms. Hashing the values means the
    check answers the question that actually matters -- is this the same dataset?
    """
    h, n, cols = _row_hashes(path)
    if h is None:
        return hashlib.sha256(path.read_bytes()).hexdigest(), None
    if not n:
        return hashlib.sha256(b"empty").hexdigest(), 0
    h = np.sort(h)
    digest = hashlib.sha256(np.ascontiguousarray(h).tobytes())
    digest.update(",".join(map(str, cols)).encode())
    del h
    return digest.hexdigest(), int(n)


def file_digest(path: Path) -> dict:
    file_hash = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            file_hash.update(chunk)
    c_hash, n_rows = content_digest(path)
    return dict(file=path.name, bytes=path.stat().st_size,
                file_sha256=file_hash.hexdigest()[:32],
                content_sha256=c_hash[:32], n_rows=n_rows)


# ======================================================================================
# Dataset identity, manifest and versioned write
# ======================================================================================


def config_fingerprint(cfg) -> str:
    payload = json.dumps(asdict(cfg), sort_keys=True, default=str)
    return hashlib.sha256(payload.encode()).hexdigest()[:12]


def dataset_id(cfg, generator_version: str) -> str:
    scenario = getattr(cfg, "scenario", "default")
    return (f"telco_gpon__{scenario}__{cfg.config_role}__gen{generator_version}"
            f"__seed{cfg.seed}__cfg{config_fingerprint(cfg)}")


def build_manifest(stage_dir, cfg, generator_version: str,
                   validation_report: pd.DataFrame | None = None,
                   notes: str = "") -> dict:
    stage_dir = Path(stage_dir)
    files = [file_digest(p) for p in sorted(stage_dir.iterdir())
             if p.is_file() and p.suffix in {".parquet", ".csv", ".json"}
             and p.name not in DERIVED_FILES]
    declared = set(NATIVE_TABLES)
    written = {f["file"] for f in files}
    manifest = dict(
        dataset_id=dataset_id(cfg, generator_version),
        sector="telco", domain="gpon_ftth", scenario=getattr(cfg, "scenario", "default"),
        generator_version=generator_version,
        native_format_version=NATIVE_FORMAT_VERSION,
        config_role=cfg.config_role,
        seed=int(cfg.seed),
        config_fingerprint=config_fingerprint(cfg),
        scale=dict(n_onts=int(cfg.n_onts), days=int(cfg.days),
                   sample_minutes=int(cfg.sample_minutes), start=str(cfg.start)),
        environment=capture_environment(),
        files=files,
        total_bytes=int(sum(f["bytes"] for f in files)),
        expected_files_missing=sorted(declared - written),
        undeclared_files_present=sorted(written - declared),
        notes=notes,
        created_ts=pd.Timestamp.now(tz="UTC").isoformat(),
    )
    if validation_report is not None and len(validation_report):
        manifest["fixture_validation"] = dict(
            n_checks=int(len(validation_report)),
            n_passed=int(validation_report.passed.sum()),
            all_passed=bool(validation_report.passed.all()),
            failed=validation_report.loc[~validation_report.passed, "check_name"].tolist())
    # identity of the dataset excludes anything that varies between identical runs
    stable = {k: manifest[k] for k in
              ("dataset_id", "generator_version", "native_format_version",
               "config_fingerprint", "seed", "scale", "scenario")}
    stable["content"] = {f["file"]: f["content_sha256"] for f in files}
    manifest["dataset_sha256"] = hashlib.sha256(
        json.dumps(stable, sort_keys=True).encode()).hexdigest()[:32]
    return manifest


def publish_dataset(stage_dir, datasets_root, cfg, generator_version: str,
                    validation_report: pd.DataFrame | None = None,
                    notes: str = "", overwrite: bool = False) -> dict:
    """Copy a completed run into the versioned dataset store and write its manifest.

    Published datasets are IMMUTABLE. Re-running with the same generator version, config
    and seed should produce the same bytes; if you want different data, change the seed or
    the config, which changes the dataset id. Overwriting in place is how a downstream
    result silently stops matching the data it was computed from.
    """
    stage_dir, datasets_root = Path(stage_dir), Path(datasets_root)
    manifest = build_manifest(stage_dir, cfg, generator_version, validation_report, notes)
    scenario = getattr(cfg, "scenario", "default")
    target = (datasets_root / "telco_gpon" / f"v{generator_version}"
              / f"{scenario}__{cfg.config_role}__seed{cfg.seed}"
                f"__{manifest['config_fingerprint']}")
    if target.exists():
        existing = target / "dataset_manifest.json"
        if existing.exists() and not overwrite:
            prev = json.loads(existing.read_text())
            same = prev.get("dataset_sha256") == manifest["dataset_sha256"]
            raise FileExistsError(
                f"{target} already exists and published datasets are immutable.\n"
                f"  content identical to the existing publication: {same}\n"
                f"  pass overwrite=True only if you are deliberately republishing.")
        if overwrite:
            shutil.rmtree(target)
    native = target / "native"
    native.mkdir(parents=True, exist_ok=True)
    for p in sorted(stage_dir.iterdir()):
        if p.is_file() and p.suffix in {".parquet", ".csv", ".json"} \
                and p.name not in DERIVED_FILES:
            shutil.copy2(p, native / p.name)
    (target / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2))
    (target / "NATIVE_FORMAT.md").write_text(native_format_doc(generator_version))
    if validation_report is not None:
        validation_report.to_csv(target / "fixture_validation_report.csv", index=False)
    manifest["published_to"] = str(target)
    register_dataset(datasets_root, manifest)
    return manifest


def native_format_doc(generator_version: str) -> str:
    lines = [f"# Native format — `{NATIVE_FORMAT_VERSION}` (generator {generator_version})",
             "",
             "The on-disk shape of a telemetry-synth run. **Adapters bind to this document, "
             "not to the generator source.** It is versioned independently of the "
             "generator: a change in simulator semantics does not necessarily change the "
             "format, and a change in format is a breaking change for every adapter.",
             "",
             "Three rules hold across every file.",
             "",
             "1. **Ground truth is a separate file, not a prefix.** `reference_dataset."
             "parquet` is observables only. Everything unobservable lives in "
             "`gt_panel.parquet` and the `gt_`-prefixed sidecars. v4 relied on a naming "
             "convention inside one table; v5 relies on the file boundary, and the "
             "generator raises if a `gt_` column reaches the panel.",
             "2. **Tickets and alarms are not ground truth.** `tickets.csv` and "
             "`alarms.csv` are operational channels with the defects real ones have — "
             "incomplete, delayed, duplicated, ambiguous. Both are legitimately available "
             "to a detector.",
             "3. **The two panels are row-aligned.** Same loop, same order, same length. "
             "Attach positionally for whole-panel work or join on "
             "`(ont_id, timestamp_utc)` for a subset.",
             "", "## Files", ""]
    for name, meta in NATIVE_TABLES.items():
        lines += [f"### `{name}`", "",
                  f"- **role** — {meta['role']}",
                  f"- **grain** — {meta['grain']}",
                  f"- {meta['note']}", ""]
    lines += ["## Regenerating a published dataset", "",
              "`dataset_manifest.json` carries the generator version, the full "
              "configuration, the seed and the library versions the run was made under. "
              "The generator's RNG streams are seeded per entity and per stage, so the "
              "same three inputs reproduce the dataset. `verify_reproducibility` performs "
              "that regeneration and compares content digests table by table.", ""]
    return "\n".join(lines)


# ======================================================================================
# Index and verification
# ======================================================================================

_INDEX_COLUMNS = ["dataset_id", "sector", "domain", "scenario", "generator_version",
                  "native_format_version", "config_role", "seed", "config_fingerprint",
                  "n_onts", "days", "sample_minutes", "total_bytes",
                  "fixture_checks_passed", "all_checks_passed", "dataset_sha256",
                  "created_ts", "path"]


def register_dataset(datasets_root, manifest: dict) -> pd.DataFrame:
    datasets_root = Path(datasets_root)
    datasets_root.mkdir(parents=True, exist_ok=True)
    index_path = datasets_root / "DATASET_INDEX.csv"
    fv = manifest.get("fixture_validation", {})
    row = {
        "dataset_id": manifest["dataset_id"], "sector": manifest["sector"],
        "domain": manifest["domain"], "scenario": manifest["scenario"],
        "generator_version": manifest["generator_version"],
        "native_format_version": manifest["native_format_version"],
        "config_role": manifest["config_role"], "seed": manifest["seed"],
        "config_fingerprint": manifest["config_fingerprint"],
        "n_onts": manifest["scale"]["n_onts"], "days": manifest["scale"]["days"],
        "sample_minutes": manifest["scale"]["sample_minutes"],
        "total_bytes": manifest["total_bytes"],
        "fixture_checks_passed": fv.get("n_passed"),
        "all_checks_passed": fv.get("all_passed"),
        "dataset_sha256": manifest["dataset_sha256"],
        "created_ts": manifest["created_ts"],
        "path": manifest.get("published_to", ""),
    }
    idx = pd.read_csv(index_path) if index_path.exists() else pd.DataFrame(columns=_INDEX_COLUMNS)
    idx = idx.loc[idx.dataset_id != row["dataset_id"]]
    idx = pd.concat([idx, pd.DataFrame([row])], ignore_index=True)
    idx = idx.reindex(columns=_INDEX_COLUMNS).sort_values("created_ts")
    idx.to_csv(index_path, index=False)
    return idx


def load_dataset_index(datasets_root) -> pd.DataFrame:
    p = Path(datasets_root) / "DATASET_INDEX.csv"
    return pd.read_csv(p) if p.exists() else pd.DataFrame(columns=_INDEX_COLUMNS)


def verify_manifest(dataset_dir) -> pd.DataFrame:
    """Recompute every digest and compare against the manifest. Integrity, not identity."""
    dataset_dir = Path(dataset_dir)
    manifest = json.loads((dataset_dir / "dataset_manifest.json").read_text())
    native = dataset_dir / "native"
    rows = []
    for rec in manifest["files"]:
        p = native / rec["file"]
        if not p.exists():
            rows.append(dict(file=rec["file"], status="MISSING", matches=False, detail=""))
            continue
        now = file_digest(p)
        ok = (now["file_sha256"] == rec["file_sha256"]
              and now["content_sha256"] == rec["content_sha256"])
        rows.append(dict(file=rec["file"], status="ok" if ok else "ALTERED", matches=ok,
                         detail="" if ok else
                         f"content {rec['content_sha256'][:8]} -> {now['content_sha256'][:8]}"))
    return pd.DataFrame(rows)


def verify_reproducibility(dataset_dir, generate_fn, settings_cls, work_dir=None,
                           ) -> pd.DataFrame:
    """Regenerate from the stored config and compare content digests table by table.

    Reproducibility is a claim about the generator, so it is tested by running the
    generator, not by trusting it. A mismatch here with a matching config fingerprint
    means either an undeclared source of randomness or a library-version dependency --
    both worth knowing before anything downstream is built on the data.
    """
    dataset_dir = Path(dataset_dir)
    manifest = json.loads((dataset_dir / "dataset_manifest.json").read_text())
    cfg_d = json.loads((dataset_dir / "native" / "generator_config.json").read_text())
    cfg = settings_cls(**{k: (tuple(v) if isinstance(v, list) else v)
                          for k, v in cfg_d.items()})
    work_dir = Path(work_dir or (dataset_dir.parent / "_repro_check"))
    if work_dir.exists():
        shutil.rmtree(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    generate_fn(cfg, out_dir=work_dir)

    env_now, env_then = capture_environment(), manifest["environment"]
    drifted = {k: (env_then.get(k), v) for k, v in env_now.items()
               if k != "platform" and env_then.get(k) != v}
    rows = []
    for rec in manifest["files"]:
        if rec["file"] == "generator_config.json" or rec["file"] in DERIVED_FILES:
            continue
        p = work_dir / rec["file"]
        if not p.exists():
            rows.append(dict(file=rec["file"], reproduced=False, detail="not regenerated"))
            continue
        c_hash, n_rows = content_digest(p)
        ok = c_hash[:32] == rec["content_sha256"]
        rows.append(dict(file=rec["file"], reproduced=ok,
                         detail="" if ok else f"rows {rec['n_rows']} -> {n_rows}"))
    out = pd.DataFrame(rows)
    out.attrs["environment_drift"] = drifted
    shutil.rmtree(work_dir, ignore_errors=True)
    return out


## 10. Run the generator

```python
RUN_DATA_STAGE = True
CONFIG_ROLE = 'benchmark_enriched'   # or 'field_prevalence'
```

`benchmark_enriched` carries an elevated fault rate so every family has enough events for per-family work. `field_prevalence` lowers the rates and leaves a much larger clean cohort; its per-family counts are thin **by construction**. Roughly 50 seconds at 400 ONTs × 180 days.

In [ ]:
RUN_DATA_STAGE = False
PUBLISH_TO_DRIVE = True
VERIFY_REPRODUCIBILITY = False   # regenerates the dataset; roughly doubles the run time
STRICT_VALIDATION = True

SIMULATION_SEED = 20250717
CONFIG_ROLE = 'benchmark_enriched'     # 'benchmark_enriched' | 'field_prevalence'
PUBLICATION_NOTES = 'v4.1.0 reference fixture'

STAGE_DIRECTORY = STAGE_ROOT / f'{CONFIG_ROLE}_seed{SIMULATION_SEED}'
STAGE_DIRECTORY.mkdir(parents=True, exist_ok=True)


In [ ]:
simulation_settings = None
if not RUN_DATA_STAGE:
    print('STOPPED SAFELY. Set RUN_DATA_STAGE=True in the cell above to generate.')
else:
    reference_configs = build_reference_configs(base_seed=SIMULATION_SEED)
    if CONFIG_ROLE not in reference_configs:
        raise ValueError(f'CONFIG_ROLE must be one of {sorted(reference_configs)}')
    simulation_settings = reference_configs[CONFIG_ROLE]
    generate_telecom_reference_data(simulation_settings, out_dir=STAGE_DIRECTORY)
    print('Generated into:', STAGE_DIRECTORY)
    print('Dataset id would be:', dataset_id(simulation_settings, GENERATOR_VERSION))


## 11. Fixture validation gate

In [ ]:
validation_report = None
if RUN_DATA_STAGE:
    validation_report = validate_reference_data(
        STAGE_DIRECTORY / simulation_settings.out_path,
        STAGE_DIRECTORY / 'gt_fault_registry.csv',
        STAGE_DIRECTORY / 'topology.csv',
        STAGE_DIRECTORY / 'tickets.csv',
        stage_dir=STAGE_DIRECTORY)
    validation_report.to_csv(STAGE_DIRECTORY / 'fixture_validation_report.csv', index=False)
    display(validation_report)
    print(f'{int(validation_report.passed.sum())} of {len(validation_report)} checks passed')
    if STRICT_VALIDATION and not validation_report.passed.all():
        failed = validation_report.loc[~validation_report.passed, 'check_name'].tolist()
        raise AssertionError(f'Fixture validation failed: {failed}')
else:
    print('Validation not run because RUN_DATA_STAGE=False.')


## 12. Publish a versioned dataset to Drive

In [ ]:
dataset_manifest = None
if RUN_DATA_STAGE and PUBLISH_TO_DRIVE:
    dataset_manifest = publish_dataset(
        STAGE_DIRECTORY, DATASETS_ROOT, simulation_settings, GENERATOR_VERSION,
        validation_report, notes=PUBLICATION_NOTES, overwrite=False)
    print('Published to:', dataset_manifest['published_to'])
    print('dataset_sha256:', dataset_manifest['dataset_sha256'])
    print('files:', len(dataset_manifest['files']),
          '| total:', f"{dataset_manifest['total_bytes'] / 1e6:.0f} MB")
    if dataset_manifest['expected_files_missing']:
        print('WARNING - declared but absent:', dataset_manifest['expected_files_missing'])
    if dataset_manifest['undeclared_files_present']:
        print('WARNING - present but undeclared in NATIVE_TABLES:',
              dataset_manifest['undeclared_files_present'])
    display(load_dataset_index(DATASETS_ROOT))
else:
    print('Publication skipped.')


## 13. Verify integrity, and optionally reproducibility

Integrity recomputes every digest against the manifest. Reproducibility regenerates the dataset from its stored config and compares content digests table by table — a claim about the generator, tested by running it. On the reference run: **11 of 11 files reproduced**, including the 6.5 M-row panel.

A mismatch with a matching config fingerprint means either an undeclared source of randomness or a library-version dependency, which is why the environment capture is compared too.

In [ ]:
if dataset_manifest is not None:
    published = Path(dataset_manifest['published_to'])

    integrity = verify_manifest(published)
    display(integrity)
    print(f'integrity: {int(integrity.matches.sum())} of {len(integrity)} files intact')

    if VERIFY_REPRODUCIBILITY:
        repro = verify_reproducibility(published, generate_telecom_reference_data,
                                       TelecomSimulationSettings)
        display(repro)
        print(f'reproduced: {int(repro.reproduced.sum())} of {len(repro)} files')
        drift = repro.attrs.get('environment_drift') or {}
        if drift:
            print('LIBRARY DRIFT since publication - a mismatch below may be explained '
                  'by this rather than by the generator:', drift)
    else:
        print('Reproducibility check skipped (VERIFY_REPRODUCIBILITY=False).')
else:
    print('Nothing published in this session, so nothing to verify.')


## 14. Parameter provenance

Several parameters were set by iterating against gate targets on this fixture. They are properties of the **benchmark**, not of any real network, and presenting them as engineering facts would be misleading in a client deliverable. This table travels with the data.

In [ ]:
provenance = parameter_provenance(
    simulation_settings if simulation_settings is not None else TelecomSimulationSettings())
display(provenance)
print(provenance.status.value_counts().to_string())

print('\nBenchmark-tuned parameters - properties of this fixture, not of any real network:')
display(provenance.loc[provenance.status.eq('benchmark_tuning'), ['parameter', 'value', 'evidence']])


## 15. Scenario grid

Seven configurations: `default_reference`, `field_prevalence`, `tight_margins`, `noisy_sensors`, `hourly_polling`, and two new in v4 — `poor_collection` (heterogeneous poll reliability, frequent collector outages) and `storm_season` (high grouped-cause density). Each is published as its own versioned dataset.

In [ ]:
def run_scenario_grid(scenarios: dict | None = None, publish: bool = True,
                      strict: bool = True) -> pd.DataFrame:
    """Generate, gate and publish every scenario as its own versioned dataset.

    Seeds vary the realisation of ONE environment; robustness claims need variation of the
    environment itself. Each scenario is published separately so a downstream result can
    always name the dataset it was computed from.
    """
    scenarios = scenarios or build_scenario_grid(base_seed=SIMULATION_SEED)
    rows = []
    for name, s_cfg in scenarios.items():
        s_dir = STAGE_ROOT / f'scenario_{name}'
        s_dir.mkdir(parents=True, exist_ok=True)
        generate_telecom_reference_data(s_cfg, out_dir=s_dir)
        rep = validate_reference_data(s_dir / s_cfg.out_path, s_dir / 'gt_fault_registry.csv',
                                      s_dir / 'topology.csv', s_dir / 'tickets.csv',
                                      stage_dir=s_dir)
        rep.to_csv(s_dir / 'fixture_validation_report.csv', index=False)
        ok = bool(rep.passed.all())
        if strict and not ok:
            raise AssertionError(f"scenario '{name}' failed: "
                                 f"{rep.loc[~rep.passed, 'check_name'].tolist()}")
        man = (publish_dataset(s_dir, DATASETS_ROOT, s_cfg, GENERATOR_VERSION, rep,
                               notes=f'scenario grid: {name}', overwrite=True)
               if publish else None)
        rows.append(dict(scenario=name, seed=s_cfg.seed, config_role=s_cfg.config_role,
                         checks_passed=int(rep.passed.sum()), all_passed=ok,
                         dataset_sha256=None if man is None else man['dataset_sha256']))
    summary = pd.DataFrame(rows)
    summary.to_csv(DATASETS_ROOT / 'scenario_grid_summary.csv', index=False)
    return summary


In [ ]:
RUN_SCENARIO_GRID = False

if RUN_SCENARIO_GRID:
    scenario_summary = run_scenario_grid()
    display(scenario_summary)
else:
    print('Scenario grid not run because RUN_SCENARIO_GRID=False.')


## Outputs, and what deliberately is not here

Each publication writes an immutable directory under `datasets/telco_gpon/v<generator_version>/<config_role>__seed<seed>__<config_fingerprint>/`:

- `native/` — the fourteen generator outputs: the **observable** panel, the **ground-truth** panel, topology, service windows, fault registry, fault-entity intervals, fault groups, tickets, alarms, engineering events, benign anomalies, collection gaps, provenance and the config
- `dataset_manifest.json` — generator version, native-format version, config fingerprint, seed, scale, **library versions**, per-file byte size, row count, file digest and content digest, and a single `dataset_sha256` identifying the dataset
- `NATIVE_FORMAT.md` — the on-disk shape, written for an adapter author who should never need to read generator source
- `fixture_validation_report.csv` — the 31 gate results for this run

`datasets/DATASET_INDEX.csv` lists every publication, so a downstream result can always name the dataset it came from.

**Publications are immutable.** Re-publishing the same generator version, config and seed raises rather than overwriting, because overwriting in place is how a downstream result silently stops matching the data it was computed from. Change the seed or the config — which changes the dataset id — or pass `overwrite=True` deliberately.

### Not in this notebook, by design

**No canonical schema.** v4.0.0 emitted canonical tables from inside the generator, which inverted the dependency and would have forced every future sector to inherit GPON's shape. The mapping belongs in a separate adapter layer, built once there are several sources to generalise over. `NATIVE_FORMAT.md` is what that layer will bind to.

**No features, no detectors, no models.** The one place this notebook comes close is check D1, which asks whether trivial threshold rules solve the fixture too easily. That is data quality assurance — it is how the v3.1 and v4.0 defects were found — and it builds nothing reusable.

**No canonical `margin` column.** v4.0 emitted `gt_margin_db` and called it ground truth; it was arithmetic on two observable columns. Margin is a *feature*, and features belong to the next notebook.

### Next

Generate the second regime (`CONFIG_ROLE = 'field_prevalence'`) and the seven-scenario grid, so the dataset store holds the full set before any downstream work starts.

Two things to settle before the adapter layer is written. **OD6**: the D1 band is set from two runs, and a band that has never been stress-tested is a guess with a gate around it. **OD7**: `us_ratio` per fault family is what makes upstream/downstream localisation learnable and it has no field basis — it is the single parameter in v4.1 most worth putting in front of a practitioner, ahead of anything in the storm or frailty models.
